# DiffMIC-v2: complete Google Colab runner

This notebook is generated from the current workspace and contains a `%%writefile` cell for **every Python file** in the project, including the APTOS research pipeline and tests. Running the source cells reconstructs the project under a persistent Google Drive directory, so the notebook can run the same code as the repository instead of maintaining a second hand-written implementation.

## What the notebook does

1. Mounts Google Drive and creates a persistent project/run directory.
2. Installs the Python dependencies without trying to install the Colab-managed CUDA/PyTorch stack from the Conda export.
3. Clones EfficientSAM, which is required by `model.py`.
4. Materializes every `.py`, YAML, and research requirements file from this notebook.
5. Runs lightweight integrity tests and preflight checks.
6. Validates the APTOS dataset and writes an immutable split manifest.
7. Trains or resumes DiffMIC-v2 with Lightning checkpoints on Drive.
8. Extracts resumable reverse-diffusion trajectories in chunks.
9. Runs classical experiments, the validation-only evidence gate, optional QRC, trust, and XAI outputs.

## Important Colab limitation

No notebook can guarantee that a free Colab runtime will never disconnect or be reclaimed. This notebook is designed so a disconnect is recoverable: logs, Lightning checkpoints, the split manifest, and the SQLite trajectory cache live on Drive; rerunning a stage resumes or skips completed work. Do not change the dataset manifest, checkpoint, seed, inference steps, or pooling mode after a stage has started.

Before enabling the expensive stages, edit the paths and flags in the configuration cell. You need a GPU, the APTOS images/CSV, a compatible pretrained DCG checkpoint, and—after training—a DiffMIC Lightning checkpoint.


## 1. Persistent Colab paths and run switches

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except Exception:
    IN_COLAB = False

if IN_COLAB:
    PERSIST_ROOT = Path('/content/drive/MyDrive/DiffMICv2_colab')
else:
    PERSIST_ROOT = Path.cwd() / 'DiffMICv2_colab'

REPO_ROOT = PERSIST_ROOT / 'workspace'
OUTPUT_ROOT = REPO_ROOT / 'outputs'
LOG_ROOT = REPO_ROOT / 'logs'
STATUS_ROOT = REPO_ROOT / '.colab_status'

# EDIT THESE THREE PATHS before running data/training/extraction stages.
APTOS_ROOT = Path('/content/drive/MyDrive/datasets/APTOS')
AUX_CHECKPOINT = Path('/content/drive/MyDrive/checkpoints/dcg.ckpt')
EXPLICIT_DIFFMIC_CHECKPOINT = ''  # Set to a trained .ckpt to skip auto-discovery.

SEED = 2026
RUN_NAME = 'aptos-v2-colab'
EPOCHS = 1000
BATCH_SIZE = 2  # Colab-safe default; use 32 to match configs/aptos.yml if memory permits.
NUM_WORKERS = 0  # Drive-backed data is more reliable with 0 workers in Colab.
INFERENCE_STEPS = 100
POOLING = 'flatten'
EXTRACTION_CHUNK = 32
EXTRACTION_CHUNKS_PER_CELL = 4

# Keep these False until paths and assets pass preflight. Set RUN_ALL=True to run every stage.
RUN_ALL = False
RUN_UNIT_TESTS = True
RUN_DATA_VALIDATION = False
RUN_TRAINING = False
RUN_EXTRACTION_SMOKE = False
RUN_FULL_EXTRACTION = False
RUN_CLASSICAL_EXPERIMENTS = False
RUN_GATE = False
RUN_QRC = False
RUN_TRUST = False
RUN_XAI = False

if RUN_ALL:
    RUN_DATA_VALIDATION = True
    RUN_TRAINING = True
    RUN_EXTRACTION_SMOKE = True
    RUN_FULL_EXTRACTION = True
    RUN_CLASSICAL_EXPERIMENTS = True
    RUN_GATE = True
    RUN_QRC = True
    RUN_TRUST = True
    RUN_XAI = True

for directory in (REPO_ROOT, OUTPUT_ROOT, LOG_ROOT, STATUS_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TORCH_HOME'] = str(PERSIST_ROOT / 'torch_cache')
os.environ['HF_HOME'] = str(PERSIST_ROOT / 'huggingface_cache')
os.environ['MPLCONFIGDIR'] = str(PERSIST_ROOT / 'matplotlib_cache')
for directory in (Path(os.environ['TORCH_HOME']), Path(os.environ['HF_HOME']), Path(os.environ['MPLCONFIGDIR'])):
    directory.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('REPO_ROOT:', REPO_ROOT)
print('APTOS_ROOT:', APTOS_ROOT)
print('AUX_CHECKPOINT:', AUX_CHECKPOINT)
print('GPU-dependent stages are disabled until you set their RUN_* flags.')


## 2. Install runtime dependencies

In [ ]:
# The repository's requirements.txt is a Conda environment export, not a pip requirements file.
# Colab already supplies a CUDA-enabled torch/torchvision pair, so do not replace it here.
packages = [
    'numpy>=1.23', 'Pillow>=9.3', 'scikit-learn>=1.3', 'scipy>=1.9',
    'matplotlib>=3.7', 'joblib>=1.3', 'pytorch-lightning>=2.0',
    'diffusers>=0.20', 'timm>=0.9', 'einops>=0.6', 'easydict>=1.10',
    'PyYAML>=6.0', 'opencv-python-headless', 'scikit-image', 'imageio',
    'tqdm', 'tensorboard', 'seaborn', 'ml-collections',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Installed project runtime dependencies; Colab-managed torch/torchvision were preserved.')


## 3. Install EfficientSAM

In [ ]:
efficient_sam_dir = REPO_ROOT / 'EfficientSAM'
efficient_sam_entry = efficient_sam_dir / 'efficient_sam' / 'build_efficient_sam.py'
if not efficient_sam_entry.exists():
    if efficient_sam_dir.exists():
        raise RuntimeError(
            f'{efficient_sam_dir} exists but EfficientSAM is incomplete. Remove it manually or point the notebook at a complete checkout.'
        )
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/yformer/EfficientSAM.git', str(efficient_sam_dir)
    ])
print('EfficientSAM entry point:', efficient_sam_entry)


## 4. Prepare directories for the embedded project files

In [ ]:
embedded_python_files = [
    'aptos_research/checkpoint.py', 'aptos_research/cli.py', 'aptos_research/data.py',
    'aptos_research/diffmic_adapter.py', 'aptos_research/experiments.py',
    'aptos_research/explain.py', 'aptos_research/extract.py', 'aptos_research/features.py',
    'aptos_research/gate.py', 'aptos_research/metrics.py', 'aptos_research/models.py',
    'aptos_research/registry.py', 'aptos_research/run_qrc.py', 'aptos_research/train_gru.py',
    'aptos_research/trust.py', 'aptos_research/trust_eval.py', 'aptos_research/xai_eval.py',
    'dataloader/functional.py', 'dataloader/loading.py', 'dataloader/transforms.py',
    'dataloader/transforms2.py', 'diffuser_trainer.py', 'model.py', 'optimizer.py',
    'option/config.py', 'pipeline.py', 'pretraining/dcg.py', 'pretraining/densenet.py',
    'pretraining/modules.py', 'pretraining/tools.py', 'tests/test_aptos_pipeline.py',
    'tests/test_research_components.py', 'utils.py',
]
for relative_path in embedded_python_files:
    (REPO_ROOT / relative_path).parent.mkdir(parents=True, exist_ok=True)
print(f'Prepared {len(embedded_python_files)} embedded Python source paths.')


### Embedded source: `aptos_research/checkpoint.py`

In [ ]:
%%writefile aptos_research/checkpoint.py
"""Checkpoint inventory and shape-safe partial load reporting."""
def inspect_state_dict(model_state, checkpoint_state):
    compatible={}; mismatched={}; unexpected=[]
    for key,value in checkpoint_state.items():
        if key not in model_state: unexpected.append(key)
        elif tuple(value.shape)!=tuple(model_state[key].shape): mismatched[key]={'checkpoint':tuple(value.shape),'model':tuple(model_state[key].shape)}
        else: compatible[key]=value
    missing=sorted(set(model_state)-set(compatible))
    return {'compatible':compatible,'report':{'loaded_keys':sorted(compatible),'missing_keys':missing,'mismatched_shapes':mismatched,'unexpected_keys':sorted(unexpected)}}

def load_compatible(model, checkpoint_state, strict_fraction=None):
    compatible=inspect_state_dict(model.state_dict(),checkpoint_state)
    report=compatible['report']; denom=max(1,len(model.state_dict())); frac=len(compatible['compatible'])/denom
    report['loaded_fraction']=frac
    if strict_fraction is not None and frac<strict_fraction:
        raise ValueError('Only %.1f%% checkpoint keys compatible; required %.1f%%' % (100*frac,100*strict_fraction))
    model.load_state_dict(compatible['compatible'],strict=False)
    return report


### Embedded source: `aptos_research/cli.py`

In [ ]:
%%writefile aptos_research/cli.py
"""Command-line entry points for indexing and trajectory experiments."""
import argparse, json, sys
from pathlib import Path
from aptos_research.data import resolve_aptos,make_splits,save_split_manifest,load_split_manifest
from aptos_research.features import FeatureStore
from aptos_research.gate import evaluate_gate
from aptos_research.registry import record

def validate(args):
    records,stats=resolve_aptos(args.root,args.csv,args.images)
    corrupt=[]
    try:
        from PIL import Image
        for r in records:
            try:
                with Image.open(r['path']) as im: im.verify()
            except Exception as exc: corrupt.append({'id':r['id'],'path':r['path'],'error':str(exc)})
    except ImportError: print('WARNING: Pillow unavailable; image corruption scan not run',file=sys.stderr)
    if corrupt: raise RuntimeError('%d corrupt/unreadable images; first=%s' % (len(corrupt, ),corrupt[0]))
    splits=make_splits(records,args.seed,args.validation_fraction,args.test_fraction)
    save_split_manifest(splits,args.output,args.seed,stats)
    record('outputs/experiments/registry.jsonl','dataset-validation','COMPLETED',{'root':stats['root'],'csv':stats['csv'],'seed':args.seed,'split_manifest':str(Path(args.output).resolve())},{'split_manifest':str(Path(args.output).resolve())},stats)
    print(json.dumps({'status':'validated','stats':stats,'split_counts':{k:len(v) for k,v in splits.items()},'manifest':str(Path(args.output).resolve())},indent=2))

def run_gate(args):
    results=json.loads(Path(args.results).read_text())
    result=evaluate_gate(results,args.output,args.minimum_delta,args.alpha,args.bootstrap_samples,args.seed)
    record('outputs/experiments/registry.jsonl','classical-gate',result['decision'],{'minimum_delta':args.minimum_delta,'alpha':args.alpha,'bootstrap_samples':args.bootstrap_samples,'seed':args.seed},{'gate':str(Path(args.output).resolve())},result)
    print(json.dumps(result,indent=2))
    if result['decision']!='PASS': return 2 if result['decision']=='FAIL' else 3
    return 0

def main():
    p=argparse.ArgumentParser(prog='aptos-research'); sub=p.add_subparsers(dest='command',required=True)
    v=sub.add_parser('validate-data',help='validate APTOS CSV/images and write stratified split manifest')
    v.add_argument('--root',required=True);v.add_argument('--csv');v.add_argument('--images');v.add_argument('--output',default='outputs/aptos_splits.json');v.add_argument('--seed',type=int,default=2026);v.add_argument('--validation-fraction',type=float,default=.15);v.add_argument('--test-fraction',type=float,default=.15);v.set_defaults(func=validate)
    i=sub.add_parser('inspect-cache');i.add_argument('cache');i.set_defaults(func=lambda a:print(json.dumps(FeatureStore(a.cache).integrity(),indent=2)))
    g=sub.add_parser('gate');g.add_argument('--results',required=True);g.add_argument('--output',default='outputs/experiments/gate.json');g.add_argument('--minimum-delta',type=float,default=.01);g.add_argument('--alpha',type=float,default=.05);g.add_argument('--bootstrap-samples',type=int,default=5000);g.add_argument('--seed',type=int,default=2026);g.set_defaults(func=run_gate)
    args=p.parse_args()
    try:
        result=args.func(args)
        return result if isinstance(result,int) else 0
    except Exception as exc:
        print('ERROR: %s' % exc,file=sys.stderr)
        return 2
if __name__=='__main__': raise SystemExit(main())


### Embedded source: `aptos_research/data.py`

In [ ]:
%%writefile aptos_research/data.py
"""Strict APTOS 2019 indexing, integrity checks, and reproducible split metadata."""
from __future__ import annotations
import csv, hashlib, json, random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

class DatasetError(ValueError):
    pass

def _identifier(value: str) -> str:
    value = Path(value.strip()).name
    return Path(value).stem.casefold()

def resolve_aptos(root: str, csv_path: Optional[str] = None, image_dir: Optional[str] = None):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise DatasetError('APTOS root does not exist or is not a directory: %s' % root)
    csv_candidates = [Path(csv_path).expanduser() if csv_path else root/'train.csv', root/'labels.csv', root/'trainLabels.csv']
    csv_file = next((p.resolve() for p in csv_candidates if p.is_file()), None)
    if csv_file is None:
        raise DatasetError('Could not find annotation CSV. Pass --csv or place train.csv under %s' % root)
    image_roots = [Path(image_dir).expanduser() if image_dir else root/'train_images', root/'images', root]
    image_roots = [p.resolve() for p in image_roots if p.is_dir()]
    if not image_roots:
        raise DatasetError('No image directory found; pass --images or provide train_images/')
    image_index: Dict[str, List[Path]] = defaultdict(list)
    for directory in image_roots:
        for path in directory.rglob('*'):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
                resolved=path.resolve()
                if resolved not in image_index[_identifier(path.name)]:
                    image_index[_identifier(path.name)].append(resolved)
    records, seen = [], set()
    duplicate_ids, missing, invalid, duplicate_files = [], [], [], []
    with csv_file.open(newline='', encoding='utf-8-sig') as stream:
        reader = csv.DictReader(stream)
        if not reader.fieldnames:
            raise DatasetError('Annotation CSV has no header: %s' % csv_file)
        fields = {x.strip().casefold(): x for x in reader.fieldnames}
        id_key = next((fields[k] for k in ('id_code','image','image_id','filename','id') if k in fields), None)
        label_key = next((fields[k] for k in ('diagnosis','label','grade','level') if k in fields), None)
        if id_key is None or label_key is None:
            raise DatasetError('Expected image id and diagnosis columns; found %s' % reader.fieldnames)
        for line, row in enumerate(reader, 2):
            ident = _identifier(row.get(id_key, ''))
            if not ident:
                invalid.append({'line': line, 'reason': 'empty identifier'}); continue
            if ident in seen:
                duplicate_ids.append(ident); continue
            seen.add(ident)
            try:
                label = int(row[label_key])
                if label not in range(5): raise ValueError()
            except (TypeError, ValueError):
                invalid.append({'line': line, 'id': ident, 'label': row.get(label_key)}); continue
            matches = image_index.get(ident, [])
            if not matches:
                missing.append(ident); continue
            if len(matches) > 1:
                duplicate_files.append({'id': ident, 'paths': [str(x) for x in matches]}); continue
            records.append({'id': ident, 'label': label, 'path': str(matches[0])})
    if duplicate_ids or missing or invalid or duplicate_files:
        raise DatasetError('APTOS integrity errors: %d duplicate annotation IDs, %d missing images, %d invalid rows, %d ambiguous image IDs. Use --report to inspect all details.' % (len(duplicate_ids),len(missing),len(invalid),len(duplicate_files)))
    if not records:
        raise DatasetError('APTOS index is empty; refusing to continue.')
    return records, {'root': str(root), 'csv': str(csv_file), 'image_roots': [str(x) for x in image_roots], 'samples': len(records), 'class_counts': {str(k):v for k,v in sorted(Counter(r['label'] for r in records).items())}, 'identifier_sha256': hashlib.sha256('\n'.join(sorted(r['id'] for r in records)).encode()).hexdigest()}

def make_splits(records: Sequence[dict], seed: int = 2026, validation_fraction: float = .15, test_fraction: float = .15):
    if validation_fraction <= 0 or test_fraction <= 0 or validation_fraction + test_fraction >= 1:
        raise ValueError('Validation/test fractions must be positive and sum to less than one')
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for rec in records: by_class[int(rec['label'])].append(rec)
    result = {'train': [], 'validation': [], 'test': []}
    for label, values in sorted(by_class.items()):
        values = list(values); rng.shuffle(values); n = len(values)
        nt = max(1, round(n*test_fraction)) if n >= 3 else 0
        nv = max(1, round(n*validation_fraction)) if n >= 2 else 0
        while nt + nv >= n and (nt or nv):
            if nt >= nv and nt: nt -= 1
            elif nv: nv -= 1
        result['test'].extend(values[:nt]); result['validation'].extend(values[nt:nt+nv]); result['train'].extend(values[nt+nv:])
    for split in result.values(): rng.shuffle(split)
    ids = [{x['id'] for x in result[k]} for k in ('train','validation','test')]
    if any(ids[i] & ids[j] for i in range(3) for j in range(i+1,3)):
        raise AssertionError('Internal split overlap')
    return result

def save_split_manifest(splits, path, seed, stats=None):
    payload = {'schema_version': 1, 'seed': int(seed), 'stats': stats or {}, 'splits': {k: [{'id':r['id'],'label':int(r['label']),'path':r['path']} for r in v] for k,v in splits.items()}}
    target=Path(path); target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(json.dumps(payload, indent=2, sort_keys=True)+'\n', encoding='utf-8')
    return payload

def load_split_manifest(path):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    splits=payload.get('splits')
    if not isinstance(splits,dict) or not {'train','validation','test'} <= set(splits): raise DatasetError('Invalid split manifest: require train/validation/test')
    sets=[]
    for name in ('train','validation','test'):
        ids=set()
        for r in splits[name]:
            if int(r['label']) not in range(5): raise DatasetError('Invalid grade for %s in %s' % (r.get('id'),name))
            if r['id'] in ids: raise DatasetError('Duplicate ID %s in %s' % (r['id'],name))
            ids.add(r['id'])
        sets.append(ids)
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)): raise DatasetError('Split manifest overlaps across train/validation/test')
    return payload

class AptosImageDataset:
    """Torch Dataset loaded lazily. Returns image, grade, identifier, split."""
    def __init__(self, records, split, size=224, training=False):
        if split not in ('train','validation','test'): raise ValueError(split)
        self.records=list(records); self.split=split; self.size=int(size); self.training=bool(training and split=='train')
    def __len__(self): return len(self.records)
    def __getitem__(self, index):
        try:
            import torch
            from PIL import Image, ImageOps
        except ImportError as exc: raise RuntimeError('Install torch, torchvision, and Pillow to load model batches') from exc
        rec=self.records[index]
        with Image.open(rec['path']) as image:
            image=ImageOps.exif_transpose(image).convert('RGB')
            w,h=image.size; side=min(w,h); left=(w-side)//2; top=(h-side)//2
            image=image.crop((left,top,left+side,top+side)).resize((self.size,self.size),Image.Resampling.BILINEAR)
            if self.training:
                import random
                if random.random()<.5: image=ImageOps.mirror(image)
                if random.random()<.5: image=ImageOps.flip(image)
                image=image.rotate(random.uniform(-30,30),resample=Image.Resampling.BILINEAR)
            import numpy as np
            arr=np.asarray(image,dtype='float32')/255.0
        tensor=torch.from_numpy(arr).permute(2,0,1)
        mean=torch.tensor([.485,.456,.406])[:,None,None]; std=torch.tensor([.229,.224,.225])[:,None,None]
        return (tensor-mean)/std, torch.tensor(int(rec['label']),dtype=torch.long), rec['id'], self.split


### Embedded source: `aptos_research/diffmic_adapter.py`

In [ ]:
%%writefile aptos_research/diffmic_adapter.py
"""Concrete adapter for the repository's ConditionalModel + DCG + SR3Sampler."""
import os, yaml
from pathlib import Path

def _state(payload, prefix=None):
 if isinstance(payload,(tuple,list)) and payload and isinstance(payload[0],dict): payload=payload[0]
 if not isinstance(payload,dict): raise ValueError('Checkpoint must contain a state dictionary')
 if 'state_dict' in payload and isinstance(payload['state_dict'],dict): payload=payload['state_dict']
 if prefix:
  stripped={k[len(prefix):]:v for k,v in payload.items() if k.startswith(prefix)}
  if stripped: return stripped
 return payload

def factory():
 import torch
 from easydict import EasyDict
 from torch.utils.data import DataLoader
 from aptos_research.data import load_split_manifest,AptosImageDataset
 from aptos_research.checkpoint import load_compatible
 from model import ConditionalModel
 from pretraining.dcg import DCG
 from pipeline import SR3Sampler,SR3scheduler
 root=os.environ.get('APTOS_ROOT'); manifest=os.environ.get('APTOS_SPLITS','outputs/aptos_splits.json')
 ckpt=os.environ.get('DIFFMIC_CHECKPOINT'); aux_ckpt=os.environ.get('DIFFMIC_AUX_CHECKPOINT')
 if not root: raise RuntimeError('Set APTOS_ROOT to the real APTOS directory')
 if not ckpt or not Path(ckpt).is_file(): raise RuntimeError('Set DIFFMIC_CHECKPOINT to an existing trained ConditionalModel checkpoint')
 if not aux_ckpt or not Path(aux_ckpt).is_file(): raise RuntimeError('Set DIFFMIC_AUX_CHECKPOINT to an existing trained DCG checkpoint')
 splits=load_split_manifest(manifest)
 config=yaml.safe_load(Path('configs/aptos.yml').read_text())
 config['data']['dataroot']=root; cfg=EasyDict(config)
 device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 denoiser=ConditionalModel(cfg,guidance=True).to(device)
 aux=DCG(cfg).to(device)
 try:
  den_payload=torch.load(ckpt,map_location='cpu',weights_only=False)
 except TypeError:
  den_payload=torch.load(ckpt,map_location='cpu')
 try:
  aux_payload=torch.load(aux_ckpt,map_location='cpu',weights_only=False)
 except TypeError:
  aux_payload=torch.load(aux_ckpt,map_location='cpu')
 dr=load_compatible(denoiser,_state(den_payload,'model.'),strict_fraction=.95)
 ar=load_compatible(aux,_state(aux_payload,'aux_model.'),strict_fraction=.95)
 print('ConditionalModel checkpoint compatibility:',dr['loaded_fraction'],len(dr['mismatched_shapes']))
 print('DCG checkpoint compatibility:',ar['loaded_fraction'],len(ar['mismatched_shapes']))
 denoiser.eval();aux.eval()
 scheduler=SR3scheduler(num_train_timesteps=1000,beta_start=.0001,beta_end=.02,beta_schedule='linear',diff_chns=5)
 scheduler.set_timesteps(100)
 sampler=SR3Sampler(denoiser,scheduler,eta=0.0)
 import hashlib
 def file_hash(path):
  h=hashlib.sha256()
  with open(path,'rb') as stream:
   for chunk in iter(lambda:stream.read(1024*1024),b''): h.update(chunk)
  return h.hexdigest()
 sampler.checkpoint_id='denoiser:'+file_hash(ckpt)+';dcg:'+file_hash(aux_ckpt)
 def batches():
  for split in ('train','validation','test'):
   records=splits['splits'][split]
   ds=AptosImageDataset(records,split,size=224,training=False)
   loader=DataLoader(ds,batch_size=8,shuffle=False,num_workers=0)
   for image,label,ids,split_names in loader:
    image=image.to(device);label=label.to(device)
    with torch.no_grad():
     y,y_global,y_local,patches,attns,attn_map=aux(image)
     b,nc=y.shape; n=attns.shape[1]
     distance=torch.abs(torch.arange(n,device=device)[:,None]-torch.arange(n,device=device)[None,:])
     wg=1-distance/(n-1);wl=distance/(n-1)
     cond=wl[None,None]*y_local[:,:,None,None]+wg[None,None]*y_global[:,:,None,None]
     diag=torch.arange(n,device=device)
     for j in range(b):
      for c in range(nc):
       cond[j,c,diag,diag]=y_global[j,c];cond[j,c,n-1,0]=y_local[j,c];cond[j,c,0,n-1]=y_local[j,c]
     attn=attns.unsqueeze(-1); attn=(attn*attn.transpose(1,2)).unsqueeze(1)
     noisy=torch.randn_like(cond)
    yield {'ids':list(ids),'labels':label.cpu().tolist(),'splits':list(split_names),'x_batch':image,'noisy_y':noisy,'conditions':[cond,patches,attn]}
 return sampler,batches()


### Embedded source: `aptos_research/experiments.py`

In [ ]:
%%writefile aptos_research/experiments.py
"""Static and classical trajectory controls on one shared feature cache."""
import argparse,json,random,time
from pathlib import Path
import numpy as np
from aptos_research.features import FeatureStore
from aptos_research.metrics import classification_metrics,ordinal_mae
from aptos_research.models import pool_trajectory,make_esn,esn_transform,fit_classifier
from aptos_research.registry import record

def selected(rows,k):
 out=[]
 for r in rows:
  x=r['states']; ix=np.array([len(x)-1]) if k==1 else np.linspace(0,len(x)-1,min(k,len(x)),dtype=int); out.append(x[ix])
 return np.stack(out)
def fit_eval(train,valid,test,seed,pca_dim=0,output=None,include_gru=False):
 if pca_dim:
  from sklearn.decomposition import PCA
  from joblib import dump
  pca=PCA(n_components=pca_dim,random_state=seed).fit(np.concatenate([x['states'] for x in train],axis=0))
  for rows in (train,valid,test):
   for row in rows: row['states']=pca.transform(row['states']).astype(np.float32)
  if output:
   Path(output).parent.mkdir(parents=True,exist_ok=True);dump(pca,str(output)+'.pca.joblib')
 ytr=np.array([x['label'] for x in train]); yv=np.array([x['label'] for x in valid]); yt=np.array([x['label'] for x in test]); result={'validation':{'labels':yv.tolist(),'ids':[x['id'] for x in valid]},'test':{'labels':yt.tolist(),'ids':[x['id'] for x in test]},'models':{},'predictions':{}}
 def evaluate(name,xtr,xv,xt):
  start=time.perf_counter(); model=fit_classifier(xtr,ytr,seed).fit(xtr,ytr); fit_s=time.perf_counter()-start
  vprob=model.predict_proba(xv); t0=time.perf_counter(); tprob=model.predict_proba(xt); latency=(time.perf_counter()-t0)/max(1,len(xt))
  vm=classification_metrics(yv,vprob.tolist()); tm=classification_metrics(yt,tprob.tolist());
  result['models'][name]={'parameters':int(model[-1].coef_.size+model[-1].intercept_.size),'fit_seconds':fit_s,'test_latency_seconds_per_sample':latency,'validation':{k:v for k,v in vm.items() if k!='predictions'},'test':{k:v for k,v in tm.items() if k!='predictions'}}
  if name=='ordered_esn_k16': result['validation']['ordered_predictions']=vm['predictions']
  result['validation'][name+'_predictions']=vm['predictions']; result['test'][name+'_predictions']=tm['predictions']
  result['predictions'][name]={'validation_probabilities':vprob.tolist(),'test_probabilities':tprob.tolist(),'validation_labels':yv.tolist(),'test_labels':yt.tolist()}
 for k in (1,4,8,16):
  if len(train[0]['states'])<k: continue
  tr=selected(train,k); va=selected(valid,k); te=selected(test,k)
  for name,fn in [('ordered_mean',lambda x:x.mean(axis=1)),('ordered_last',lambda x:x[:,-1])]: evaluate('%s_k%d'%(name,k),fn(tr),fn(va),fn(te))
  evaluate('unordered_mean_k%d'%k,pool_trajectory(tr),pool_trajectory(va),pool_trajectory(te))
  # Fixed cyclic permutation, same states exactly; record permutation seed.
  perm=np.roll(np.arange(k),1); evaluate('shuffled_k%d'%k,tr[:,perm].mean(axis=1),va[:,perm].mean(axis=1),te[:,perm].mean(axis=1))
  esn=make_esn(tr.shape[-1],64,seed+k); evaluate('ordered_esn_k%d'%k,esn_transform(tr,esn),esn_transform(va,esn),esn_transform(te,esn)); evaluate('shuffled_esn_k%d'%k,esn_transform(tr[:,perm],esn),esn_transform(va[:,perm],esn),esn_transform(te[:,perm],esn))
  reservoir_params=esn['input'].size+esn['reservoir'].size
  for key in ('ordered_esn_k%d'%k,'shuffled_esn_k%d'%k): result['models'][key]['parameters']+=int(reservoir_params)
 # Validation-only best individual state; the held-out test is evaluated only after selection.
 best_idx=None; best_score=-1.0
 for state_idx in range(len(train[0]['states'])):
  evaluate('single_timestep_%d'%state_idx,np.stack([x['states'][state_idx] for x in train]),np.stack([x['states'][state_idx] for x in valid]),np.stack([x['states'][state_idx] for x in test]))
  score=result['models']['single_timestep_%d'%state_idx]['validation']['macro_f1']
  if score>best_score: best_score=score;best_idx=state_idx
 result['best_single_timestep']={'state_index':best_idx,'selection_metric':'validation macro-F1','validation_macro_f1':best_score,'test_metrics':result['models']['single_timestep_%d'%best_idx]['test']}
 # Gate consumes a pre-named matched set of relevant controls only.
 result['validation']['control_predictions']=[result['validation']['ordered_last_k16_predictions'],result['validation']['single_timestep_%d_predictions'%best_idx],result['validation']['unordered_mean_k16_predictions'],result['validation']['shuffled_esn_k16_predictions']]
 if include_gru:
  from aptos_research.train_gru import train_gru
  train_seq=selected(train,16); val_seq=selected(valid,16); test_seq=selected(test,16)
  mean=train_seq.mean(axis=(0,1),keepdims=True);std=train_seq.std(axis=(0,1),keepdims=True)+1e-6
  train_seq=(train_seq-mean)/std;val_seq=(val_seq-mean)/std;test_seq=(test_seq-mean)/std
  for name,tr_seq,va_seq,te_seq in [('ordered_gru_k16',train_seq,val_seq,test_seq),('shuffled_gru_k16',np.roll(train_seq,1,axis=1),np.roll(val_seq,1,axis=1),np.roll(test_seq,1,axis=1))]:
   start=time.perf_counter();model,vprob,tprob,param_count=train_gru(tr_seq,ytr,va_seq,yv,te_seq,seed=seed,hidden=64,epochs=80,patience=10); elapsed=time.perf_counter()-start
   vm=classification_metrics(yv,vprob.tolist());tm=classification_metrics(yt,tprob.tolist())
   result['models'][name]={'parameters':param_count,'fit_seconds':elapsed,'validation':{k:v for k,v in vm.items() if k!='predictions'},'test':{k:v for k,v in tm.items() if k!='predictions'}}
   result['validation'][name+'_predictions']=vm['predictions'];result['test'][name+'_predictions']=tm['predictions']
   result['predictions'][name]={'validation_probabilities':vprob.tolist(),'test_probabilities':tprob.tolist(),'validation_labels':yv.tolist(),'test_labels':yt.tolist()}
   if name=='shuffled_gru_k16': result['validation']['control_predictions'].append(vm['predictions'])
 result['config']={'seed':seed,'classifier':'standardized multinomial logistic regression, max_iter=1000','pca_dim':pca_dim or None,'pca_fit_scope':'training states only','gru_included':include_gru,'esn':{'hidden_size':64,'connectivity':.1,'spectral_radius':.9,'leak':.5,'readout':'logistic regression'},'shuffle':'cyclic rotation by one; fixed, same states','best_single_timestep_selection':'validation macro-F1','test_use':'locked descriptive evaluation; validation controls selection'}
 return result

def main():
 p=argparse.ArgumentParser();p.add_argument('--cache',required=True);p.add_argument('--output',default='outputs/experiments/classical.json');p.add_argument('--seed',type=int,default=2026);p.add_argument('--pca-dim',type=int,default=0);p.add_argument('--gru',action='store_true',help='also run ordered and shuffled GRU; needs PyTorch');a=p.parse_args();s=FeatureStore(a.cache);tr=s.load_split('train');va=s.load_split('validation');te=s.load_split('test')
 if not tr or not va or not te: raise SystemExit('Feature cache must contain non-empty train, validation, and test splits')
 result=fit_eval(tr,va,te,a.seed,a.pca_dim,a.output,a.gru);Path(a.output).parent.mkdir(parents=True,exist_ok=True);Path(a.output).write_text(json.dumps(result,indent=2)+'\n');record('outputs/experiments/registry.jsonl','classical-experiments','COMPLETED',result['config'],{'results':str(Path(a.output).resolve())},result.get('best_single_timestep',{}));print('saved',a.output)
if __name__=='__main__':main()


### Embedded source: `aptos_research/explain.py`

In [ ]:
%%writefile aptos_research/explain.py
"""Architecture-valid trajectory plots and perturbation faithfulness helpers."""
def save_probability_trajectory(timesteps, probabilities, output, title='Class probability by reverse timestep'):
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(8,4))
    for cls in range(probabilities.shape[1]): ax.plot(timesteps,probabilities[:,cls],marker='o',label='grade %d'%cls)
    ax.set(xlabel='Original diffusion timestep',ylabel='Class probability',title=title); ax.legend(ncol=5); fig.tight_layout(); fig.savefig(output,dpi=160); plt.close(fig)

def class_scores_to_probabilities(states, channels, spatial_shape):
    import numpy as np
    x=np.asarray(states,dtype=float)
    h,w=spatial_shape
    if x.ndim!=2 or x.shape[1]!=channels*h*w: raise ValueError('cached state dimension does not match [classes,height,width] metadata')
    scores=x.reshape(len(x),channels,h,w).mean(axis=(2,3)); scores=scores-scores.max(axis=1,keepdims=True)
    exp=np.exp(scores);return exp/exp.sum(axis=1,keepdims=True)

def save_spatial_overlay(image, attention_map, output, alpha=.4):
    """Overlay a real DCG saliency/attention map on the matching RGB input image."""
    import numpy as np
    import matplotlib.pyplot as plt
    from PIL import Image
    img=np.asarray(image).astype(float)
    if img.ndim!=3 or img.shape[-1]!=3: raise ValueError('image must be RGB HxWx3')
    if img.max()>1: img=img/255.0
    heat=np.asarray(attention_map).squeeze()
    if heat.ndim!=2: raise ValueError('DCG attention map must reduce to HxW')
    heat=np.asarray(Image.fromarray(heat.astype('float32'),mode='F').resize((img.shape[1],img.shape[0]),Image.Resampling.BILINEAR))
    heat=(heat-heat.min())/(heat.max()-heat.min()+1e-12)
    fig,ax=plt.subplots(figsize=(6,6));ax.imshow(np.clip(img,0,1));ax.imshow(heat,cmap='jet',alpha=alpha,vmin=0,vmax=1);ax.axis('off');fig.tight_layout();fig.savefig(output,dpi=160,bbox_inches='tight',pad_inches=0);plt.close(fig)

def leave_one_state_out(states, score_fn, baseline_state=None):
    """Measure target-score change after replacing each observed state in turn."""
    import numpy as np
    sequence=np.asarray(states).copy();base=np.zeros_like(sequence[0]) if baseline_state is None else np.asarray(baseline_state)
    original=float(score_fn(sequence));contributions=[]
    for i in range(len(sequence)):
        altered=sequence.copy();altered[i]=base
        contributions.append({'state_index':i,'score_after_replacement':float(score_fn(altered)),'score_drop':original-float(score_fn(altered))})
    return {'original_score':original,'contributions':contributions}

def perturbation_counterfactual(image, score_fn, saliency, target_score, max_fraction=.5, step=.05, threshold=.5):
    """Greedily neutralize high-saliency regions until a target score is crossed."""
    import numpy as np
    x=np.asarray(image).copy(); sal=np.asarray(saliency)
    spatial=x.ndim==3 and sal.shape==x.shape[-2:]
    if not spatial and sal.shape!=x.shape: raise ValueError('saliency must match image or spatial HxW')
    order=np.argsort(-sal.reshape(-1)); result=x.copy(); total=sal.size
    for frac in np.arange(step,max_fraction+step/2,step):
        n=min(total,int(frac*total)); ix=order[:n]
        if spatial:
            coords=np.unravel_index(ix,sal.shape);result[:,coords[0],coords[1]]=0
        else: result.reshape(-1)[ix]=0
        score=float(score_fn(result))
        if score<=threshold:
            return {'found':True,'fraction_perturbed':float(frac),'score':score,'target_score':float(target_score),'counterfactual':result}
    return {'found':False,'fraction_perturbed':float(max_fraction),'score':float(score_fn(result)),'target_score':float(target_score),'counterfactual':result}

def perturbation_curve(image, score_fn, saliency, fractions=None, insertion=False):
    """Pixel-level deletion/insertion curve using a supplied valid saliency map and score function."""
    import numpy as np
    x=np.asarray(image).copy(); sal=np.asarray(saliency)
    spatial=(x.ndim==3 and sal.shape==x.shape[-2:])
    if spatial:
        order=np.argsort(-sal.reshape(-1)); base=np.zeros_like(x) if insertion else x.copy(); total=sal.size
    elif sal.shape==x.shape:
        order=np.argsort(-sal.reshape(-1));base=np.zeros_like(x).reshape(-1) if insertion else x.reshape(-1).copy();total=sal.size
    else: raise ValueError('saliency must match image dimensions or its spatial HxW map')
    fractions=fractions or [i/20 for i in range(21)]; scores=[]
    for frac in fractions:
        n=int(frac*total); cur=base.copy()
        if spatial:
            h,w=sal.shape; coords=np.unravel_index(order[:n],(h,w))
            if insertion: cur[:,coords[0],coords[1]]=x[:,coords[0],coords[1]]
            else: cur[:,coords[0],coords[1]]=0
        else:
            flat=x.reshape(-1)
            if insertion: cur[order[:n]]=flat[order[:n]]
            else: cur[order[:n]]=0
            cur=cur.reshape(x.shape)
        scores.append(float(score_fn(cur)))
    return fractions,scores


### Embedded source: `aptos_research/extract.py`

In [ ]:
%%writefile aptos_research/extract.py
"""Instrument real DiffMIC SR3Sampler runs through a project adapter.

Adapter contract: factory() -> (sampler, iterable[dict]) where each batch dict has
ids, labels, splits, x_batch, noisy_y, conditions. stochastic initial states must be
seeded in the adapter; the extractor records the seed/configuration in SQLite.
"""
import argparse, hashlib, importlib, json, random
from pathlib import Path
from aptos_research.features import FeatureStore
from aptos_research.registry import record

def _resolve(spec):
 module,name=spec.split(':',1); return getattr(importlib.import_module(module),name)
def extract(factory_spec, cache_path, checkpoint, seed=2026, max_samples=None, inference_steps=100, pooling='flatten'):
 import numpy as np, torch
 random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
 if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
 sampler,batches=_resolve(factory_spec)()
 scheduler=sampler.scheduler
 if inference_steps: scheduler.set_timesteps(inference_steps)
 ts=[int(t) for t in scheduler.timesteps]
 if not ts: raise ValueError('sampler scheduler has no timesteps; call set_timesteps in adapter')
 if pooling not in ('flatten','gap'): raise ValueError('pooling must be flatten or gap')
 config={'sampler':'SR3Sampler','state':'post-step class-score diffusion grid (not an image latent)','full_reverse_steps':len(ts),'state_selection':'uniform indices over reverse-loop order','projection':'flatten spatial class-score grid' if pooling=='flatten' else 'global average pool spatial class-score grid','input_state_shape':'recorded from callback','seed':seed}
 k=min(16,len(ts)); selected_indices=[len(ts)-1] if k==1 else np.linspace(0,len(ts)-1,k,dtype=int).tolist(); wanted={ts[i]:i for i in selected_indices}
 ckpt_hash=getattr(sampler,'checkpoint_id',None)
 if ckpt_hash is None and checkpoint and checkpoint!='unavailable':
  digest=hashlib.sha256()
  with open(checkpoint,'rb') as stream:
   for chunk in iter(lambda:stream.read(1024*1024),b''): digest.update(chunk)
  ckpt_hash=digest.hexdigest()
 ckpt_hash=ckpt_hash or 'unavailable'
 store=FeatureStore(cache_path); seen=0
 for batch in batches:
  all_ids=batch['ids'];all_labels=batch['labels'];all_splits=batch['splits']
  keep=[i for i,(sid,label,split) in enumerate(zip(all_ids,all_labels,all_splits)) if not store.contains(sid,split,int(label),ckpt_hash,config,seed)]
  if not keep: continue
  indices=torch.as_tensor(keep,dtype=torch.long,device=batch['x_batch'].device)
  def take(value):
   if torch.is_tensor(value) and value.ndim>0 and value.shape[0]==len(all_ids): return value.index_select(0,indices.to(value.device))
   if isinstance(value,list): return [take(v) for v in value]
   if isinstance(value,tuple): return tuple(take(v) for v in value)
   return value
  ids=[all_ids[i] for i in keep];labels=[all_labels[i] for i in keep];splits=[all_splits[i] for i in keep]
  x_batch=take(batch['x_batch']);noisy_y=take(batch['noisy_y']);conditions=take(batch['conditions']);captures={i:[] for i in range(len(ids))}
  def callback(timestep,state_index,state):
   if timestep not in wanted: return
   if state.ndim!=4 or state.shape[0]!=len(ids) or state.shape[1]!=5: raise RuntimeError('Expected five-class diffusion state [B,5,H,W], got %s' % (tuple(state.shape),))
   config['input_state_shape']=list(state.shape[1:])
   projected=state.detach().float().flatten(1) if pooling=='flatten' else state.detach().float().mean(dim=(-2,-1))
   pooled=projected.cpu().numpy()
   for j,v in enumerate(pooled): captures[j].append((wanted[timestep],timestep,v))
  sampler.sample_high_res(x_batch,noisy_y,conditions=conditions,trajectory_callback=callback,selected_timesteps=wanted.keys())
  for j,sid in enumerate(ids):
   values=sorted(captures[j],key=lambda z:z[0])
   if len(values)!=k: raise RuntimeError('Captured %d/%d selected states for %s' % (len(values),k,sid))
   store.add_sample(sid,splits[j],int(labels[j]),np.stack([x[2] for x in values]),[x[1] for x in values],ckpt_hash,config,seed); seen+=1
   if max_samples and seen>=max_samples: store.close(); return seen
 store.close(); return seen

def main():
 p=argparse.ArgumentParser();p.add_argument('--factory',required=True,help='Python module:function adapter for real DiffMIC model/checkpoint and APTOS batches');p.add_argument('--cache',default='outputs/trajectories.sqlite');p.add_argument('--checkpoint',required=True);p.add_argument('--seed',type=int,default=2026);p.add_argument('--max-samples',type=int);p.add_argument('--inference-steps',type=int,default=100);p.add_argument('--pooling',choices=['flatten','gap'],default='flatten');a=p.parse_args()
 count=extract(a.factory,a.cache,a.checkpoint,a.seed,a.max_samples,a.inference_steps,a.pooling)
 record('outputs/experiments/registry.jsonl','trajectory-extraction','PARTIAL_BOUNDED' if a.max_samples else 'COMPLETED',{'factory':a.factory,'seed':a.seed,'inference_steps':a.inference_steps,'pooling':a.pooling,'max_samples':a.max_samples},{'feature_cache':str(Path(a.cache).resolve())},{'new_samples':count})
 print(json.dumps({'new_samples':count}))
if __name__=='__main__':main()


### Embedded source: `aptos_research/features.py`

In [ ]:
%%writefile aptos_research/features.py
"""Resumable, split-aware SQLite trajectory cache (float32 vector BLOBs)."""
import hashlib, json, sqlite3, time
from pathlib import Path

class FeatureStore:
    def __init__(self, path):
        self.path=Path(path); self.path.parent.mkdir(parents=True,exist_ok=True)
        self.db=sqlite3.connect(str(self.path))
        self.db.execute('PRAGMA journal_mode=WAL')
        self.db.execute('''CREATE TABLE IF NOT EXISTS states (sample_id TEXT NOT NULL, split TEXT NOT NULL, label INTEGER NOT NULL, state_index INTEGER NOT NULL, diffusion_timestep INTEGER NOT NULL, dimension INTEGER NOT NULL, representation BLOB NOT NULL, checkpoint_id TEXT NOT NULL, config_json TEXT NOT NULL, seed INTEGER NOT NULL, created REAL NOT NULL, PRIMARY KEY(sample_id, split, state_index))''')
        self.db.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
        version=self.db.execute('SELECT value FROM metadata WHERE key="schema_version"').fetchone()
        if version and int(version[0])!=1: raise ValueError('Unsupported trajectory cache schema version: %s'%version[0])
        self.db.execute('INSERT OR REPLACE INTO metadata(key,value) VALUES("schema_version","1")')
        self.db.execute('CREATE INDEX IF NOT EXISTS states_split_idx ON states(split, sample_id)')
        self.db.commit()
    def add_sample(self, sample_id, split, label, states, timesteps, checkpoint_id, config, seed):
        import numpy as np
        if split not in ('train','validation','test'): raise ValueError('invalid split')
        states=np.asarray(states,dtype=np.float32)
        if states.ndim!=2 or not np.isfinite(states).all(): raise ValueError('states must be finite [K,D]')
        if states.shape[0]!=len(timesteps): raise ValueError('one original timestep required per state')
        if int(label) not in range(5): raise ValueError('APTOS grade must be 0..4')
        cfg=json.dumps(config,sort_keys=True)
        with self.db:
            for i,(state,t) in enumerate(zip(states,timesteps)):
                self.db.execute('INSERT OR REPLACE INTO states VALUES(?,?,?,?,?,?,?,?,?,?,?)',(str(sample_id),split,int(label),i,int(t),states.shape[1],sqlite3.Binary(state.tobytes()),str(checkpoint_id),cfg,int(seed),time.time()))
    def contains(self, sample_id, split, label=None, checkpoint_id=None, config=None, seed=None):
        row=self.db.execute('SELECT label,checkpoint_id,config_json,seed FROM states WHERE sample_id=? AND split=? LIMIT 1',(str(sample_id),split)).fetchone()
        if row is None: return False
        old_label,old_checkpoint,old_config,old_seed=row
        if label is not None and old_label!=int(label): raise ValueError('Cached label mismatch for %s'%sample_id)
        if checkpoint_id is not None and old_checkpoint!=str(checkpoint_id): raise ValueError('Cached checkpoint mismatch for %s; choose a new cache path'%sample_id)
        if seed is not None and old_seed!=int(seed): raise ValueError('Cached extraction seed mismatch for %s; choose a new cache path'%sample_id)
        if config is not None:
            old=json.loads(old_config); new=dict(config)
            old.pop('input_state_shape',None);new.pop('input_state_shape',None)
            if old!=new: raise ValueError('Cached extraction configuration mismatch for %s; choose a new cache path'%sample_id)
        return True
    def load_split(self, split, sample_ids=None, limit=None):
        import numpy as np
        query='SELECT sample_id,label,state_index,diffusion_timestep,dimension,representation,checkpoint_id,config_json,seed FROM states WHERE split=?'; params=[split]
        if sample_ids is not None:
            ids=list(sample_ids)
            if not ids: return []
            query+=' AND sample_id IN (%s)'%','.join('?' for _ in ids);params.extend(ids)
        if limit is not None:
            if int(limit)<1: return []
            query+=' AND sample_id IN (SELECT sample_id FROM states WHERE split=? ORDER BY sample_id LIMIT ?)';params.extend([split,int(limit)])
        query+=' ORDER BY sample_id,state_index'
        rows=self.db.execute(query,params).fetchall()
        grouped={}
        for sid,label,idx,t,d,blob,ckpt,cfg,seed in rows:
            entry=grouped.setdefault(sid,{'label':label,'states':[],'timesteps':[],'checkpoint_id':ckpt,'config':json.loads(cfg),'seed':seed})
            if idx!=len(entry['states']): raise ValueError('cache state indices are not contiguous for '+sid)
            entry['states'].append(np.frombuffer(blob,dtype=np.float32,count=d).copy()); entry['timesteps'].append(t)
        return [{'id':sid,**v,'states':np.stack(v['states'])} for sid,v in grouped.items()]
    def integrity(self):
        splits={}
        for split, in self.db.execute('SELECT DISTINCT split FROM states'):
            samples=self.db.execute('SELECT COUNT(DISTINCT sample_id) FROM states WHERE split=?',(split,)).fetchone()[0]
            dims=self.db.execute('SELECT COUNT(DISTINCT dimension) FROM states WHERE split=?',(split,)).fetchone()[0]
            splits[split]={'samples':samples,'dimensions':dims,'state_rows':self.db.execute('SELECT COUNT(*) FROM states WHERE split=?',(split,)).fetchone()[0]}
        issues=[]
        if not self.db.execute('SELECT 1 FROM states LIMIT 1').fetchone(): issues.append('feature cache is empty')
        if self.db.execute('SELECT COUNT(*) FROM states WHERE label<0 OR label>4 OR dimension<1 OR split NOT IN ("train","validation","test")').fetchone()[0]: issues.append('invalid label/dimension/split')
        for sid,split,n,min_i,max_i,dim_count,label_count in self.db.execute('SELECT sample_id,split,COUNT(*),MIN(state_index),MAX(state_index),COUNT(DISTINCT dimension),COUNT(DISTINCT label) FROM states GROUP BY sample_id,split'):
            if min_i!=0 or max_i!=n-1: issues.append('non-contiguous state indices for %s/%s'%(sid,split))
            if dim_count!=1 or label_count!=1: issues.append('inconsistent dimension or label for %s/%s'%(sid,split))
        if self.db.execute('SELECT COUNT(*) FROM (SELECT sample_id FROM states GROUP BY sample_id HAVING COUNT(DISTINCT split)>1)').fetchone()[0]: issues.append('sample IDs overlap across splits')
        state_count_variants=self.db.execute('SELECT COUNT(DISTINCT n) FROM (SELECT sample_id,split,COUNT(*) AS n FROM states GROUP BY sample_id,split)').fetchone()[0]
        if state_count_variants>1: issues.append('inconsistent trajectory lengths across samples')
        digest=None
        if self.path.exists():
            self.db.execute('PRAGMA wal_checkpoint(FULL)')
            h=hashlib.sha256()
            with self.path.open('rb') as stream:
                for chunk in iter(lambda:stream.read(1024*1024),b''): h.update(chunk)
            digest=h.hexdigest()
        return {'schema_version':1,'path':str(self.path.resolve()),'splits':splits,'valid':not issues,'issues':sorted(set(issues)),'sha256':digest}
    def close(self): self.db.close()


### Embedded source: `aptos_research/gate.py`

In [ ]:
%%writefile aptos_research/gate.py
"""Pre-specified evidence gate. Validation selects; locked test comparison is separate."""
import json, random
from pathlib import Path

def evaluate_gate(results, output, minimum_delta=.01, alpha=.05, bootstrap_samples=5000, seed=2026):
    """Paired percentile bootstrap on accuracy deltas vs strongest named validation control.

    PASS requires lower 95% bound and point improvement above the prespecified practical
    threshold. Test predictions are neither read nor accepted by this function.
    """
    val=results.get('validation',{})
    dynamic=val.get('ordered_predictions'); labels=val.get('labels')
    controls=val.get('control_predictions',[])
    if not bootstrap_samples or not 0 < alpha < 1 or minimum_delta < 0:
        raise ValueError('bootstrap_samples>0, alpha in (0,1), and nonnegative minimum_delta required')
    ids=val.get('ids')
    if dynamic is None or labels is None or ids is None or not controls or not len(labels):
        decision={'decision':'INCONCLUSIVE','reason':'validation IDs, labels, ordered predictions, and at least one matched control are required','minimum_accuracy_delta':minimum_delta,'alpha':alpha}
    elif len(dynamic)!=len(labels) or len(ids)!=len(labels) or len(set(ids))!=len(ids) or any(len(x)!=len(labels) for x in controls):
        decision={'decision':'INCONCLUSIVE','reason':'prediction arrays do not align with validation labels','minimum_accuracy_delta':minimum_delta,'alpha':alpha}
    elif any(int(y) not in range(5) for y in labels) or any(int(y) not in range(5) for y in dynamic) or any(int(p) not in range(5) for x in controls for p in x):
        decision={'decision':'INCONCLUSIVE','reason':'validation grades and predictions must be integers 0..4','minimum_accuracy_delta':minimum_delta,'alpha':alpha}
    else:
        correct=[[int(pred[i])==int(labels[i]) for pred in controls] for i in range(len(labels))]
        control_acc=[sum(row[j] for row in correct)/len(labels) for j in range(len(controls))]
        best=max(range(len(controls)),key=lambda j:control_acc[j])
        delta=[(1.0 if int(dynamic[i])==int(labels[i]) else 0.0)-(1.0 if correct[i][best] else 0.0) for i in range(len(labels))]
        rng=random.Random(seed); boots=[]
        for _ in range(bootstrap_samples):
            boots.append(sum(delta[rng.randrange(len(delta))] for _ in delta)/len(delta))
        boots.sort(); lo=boots[max(0,int(alpha/2*bootstrap_samples))]; hi=boots[min(bootstrap_samples-1,int((1-alpha/2)*bootstrap_samples))]; point=sum(delta)/len(delta)
        state='PASS' if point>minimum_delta and lo>minimum_delta else ('FAIL' if hi<=minimum_delta else 'INCONCLUSIVE')
        decision={'decision':state,'reason':'paired percentile bootstrap versus highest-accuracy named validation control','minimum_accuracy_delta':minimum_delta,'alpha':alpha,'point_delta_accuracy':point,'bootstrap_ci':[lo,hi],'samples':len(labels),'controls':len(controls),'strongest_control_index':best,'control_accuracies':control_acc,'seed':seed,'bootstrap_samples':bootstrap_samples}
    target=Path(output); target.parent.mkdir(parents=True,exist_ok=True); target.write_text(json.dumps(decision,indent=2)+'\n')
    return decision

def qrc_is_eligible(gate_path): return json.loads(Path(gate_path).read_text()).get('decision')=='PASS'


### Embedded source: `aptos_research/metrics.py`

In [ ]:
%%writefile aptos_research/metrics.py
"""Metric definitions and calibration helpers for five-class ordinal APTOS."""
import math

def _matrix(labels, predictions, classes=5):
    cm=[[0]*classes for _ in range(classes)]
    for y,p in zip(labels,predictions):
        y,p=int(y),int(p)
        if not 0<=y<classes or not 0<=p<classes: raise ValueError('labels/predictions must be grades 0..4')
        cm[y][p]+=1
    return cm

def classification_metrics(labels, probabilities):
    if len(labels)!=len(probabilities) or not labels: raise ValueError('non-empty aligned labels/probabilities required')
    k=len(probabilities[0]); pred=[max(range(k),key=lambda j:row[j]) for row in probabilities]
    cm=_matrix(labels,pred,k); n=len(labels)
    f1=[]
    for c in range(k):
        tp=cm[c][c]; fp=sum(cm[r][c] for r in range(k) if r!=c); fn=sum(cm[c][j] for j in range(k) if j!=c)
        f1.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0)
    total=sum(cm[i][j] for i in range(k) for j in range(k)); observed=sum(cm[i][j]*(i-j)**2 for i in range(k) for j in range(k))/max(1,total)
    rows=[sum(row) for row in cm]; cols=[sum(cm[i][j] for i in range(k)) for j in range(k)]
    expected=sum(rows[i]*cols[j]*(i-j)**2 for i in range(k) for j in range(k))/max(1,total*total)
    brier=sum(sum((float(row[j])-(j==int(y)))**2 for j in range(k)) for y,row in zip(labels,probabilities))/n
    confidence=[max(row) for row in probabilities]; correct=[p==int(y) for p,y in zip(pred,labels)]
    ece=0.0
    for b in range(10):
        ix=[i for i,c in enumerate(confidence) if (b/10<=c<(b+1)/10) or (b==9 and c==1)]
        if ix: ece+=len(ix)/n*abs(sum(correct[i] for i in ix)/len(ix)-sum(confidence[i] for i in ix)/len(ix))
    aucs=[]
    for c in range(k):
        positive=[float(row[c]) for y,row in zip(labels,probabilities) if int(y)==c]
        negative=[float(row[c]) for y,row in zip(labels,probabilities) if int(y)!=c]
        if positive and negative:
            aucs.append(sum((1.0 if p>q else .5 if p==q else 0.0) for p in positive for q in negative)/(len(positive)*len(negative)))
    return {'accuracy':sum(pred[i]==int(labels[i]) for i in range(n))/n,'macro_f1':sum(f1)/k,'per_class_f1':f1,'quadratic_weighted_kappa':1-observed/expected if expected else 0.0,'ordinal_mae':ordinal_mae(labels,pred),'macro_auroc_ovr':sum(aucs)/len(aucs) if aucs else None,'brier_score':brier,'ece_10_bins':ece,'confusion_matrix':cm,'predictions':pred}

def ordinal_mae(labels, predictions): return sum(abs(int(a)-int(b)) for a,b in zip(labels,predictions))/len(labels)


### Embedded source: `aptos_research/models.py`

In [ ]:
%%writefile aptos_research/models.py
"""Static, permutation-invariant, classical recurrent and gated QRC models."""
def pool_trajectory(states, method='mean'):
    import numpy as np
    x=np.asarray(states,dtype=float)
    if x.ndim!=3: raise ValueError('expected [N,K,D]')
    if method=='mean': return x.mean(axis=1)
    if method=='mean_std': return np.concatenate([x.mean(axis=1),x.std(axis=1)],axis=1)
    if method=='flatten_unordered': return np.sort(x,axis=1).reshape(x.shape[0],-1)
    raise ValueError(method)

def make_esn(input_dim, hidden_dim=64, seed=0, spectral_radius=.9, connectivity=.1, leak=.5):
    import numpy as np
    rng=np.random.default_rng(seed); w=rng.normal(0,1,(hidden_dim,hidden_dim)); w[rng.random(w.shape)>connectivity]=0
    radius=max(abs(np.linalg.eigvals(w)))
    return {'input':rng.normal(0,.5,(input_dim,hidden_dim)), 'reservoir':w*(spectral_radius/(radius or 1)), 'leak':float(leak), 'spectral_radius':spectral_radius,'connectivity':connectivity}

def esn_transform(states, reservoir):
    import numpy as np
    x=np.asarray(states,dtype=float); outs=[]
    for sample in x:
        h=np.zeros(reservoir['reservoir'].shape[0])
        for u in sample: h=(1-reservoir['leak'])*h+reservoir['leak']*np.tanh(u@reservoir['input']+h@reservoir['reservoir'])
        outs.append(np.concatenate([h, sample[-1]]))
    return np.asarray(outs)

def qrc_features(states, n_qubits=4, seed=0, shots=0, noise=0.0):
    """Small state-vector reservoir: angle encode each compact state and apply fixed entangling layers.

    Returned Z expectation features include encoding/simulator settings in metadata. This is a research
    implementation, not evidence of quantum advantage; noisy finite-shot readout is optional.
    """
    import numpy as np
    if n_qubits<1 or n_qubits>10: raise ValueError('state-vector QRC supports 1..10 qubits')
    if not 0.0 <= noise <= 1.0: raise ValueError('depolarizing probability must be in [0,1]')
    rng=np.random.default_rng(seed); dim=2**n_qubits; out=[]
    for sequence in np.asarray(states,dtype=float):
        state=np.zeros(dim,dtype=complex); state[0]=1
        for vector in sequence:
            # Deterministically pool compact classical coordinates to each qubit angle.
            angles=np.asarray([vector[q::n_qubits].mean() for q in range(n_qubits)])
            angles=np.pi*np.tanh(angles)
            # Product RY encoding
            for q,theta in enumerate(angles):
                bit=1<<q; new=state.copy(); c=np.cos(theta/2); s=np.sin(theta/2)
                for i in range(dim):
                    if i&bit: continue
                    a,b=state[i],state[i|bit]; new[i]=c*a-s*b; new[i|bit]=s*a+c*b
                state=new
            # Fixed ring entanglement (CNOTs) is unitary and introduces cross-qubit mixing.
            for q in range(n_qubits):
                control=1<<q; target=1<<((q+1)%n_qubits)
                for i in range(dim):
                    if i&control and not i&target:
                        j=i|target; state[i],state[j]=state[j],state[i]
        probs=np.abs(state)**2; z=[]
        for q in range(n_qubits):
            z.append(sum((1 if not i&(1<<q) else -1)*probs[i] for i in range(dim)))
        # Simple readout depolarization model shrinks Pauli-Z expectations.
        z=np.asarray(z)*(1.0-noise)
        if shots:
            z=np.asarray([2*rng.binomial(int(shots),(1+float(v))/2)/int(shots)-1 for v in z])
        out.append(z)
    return np.asarray(out), {'n_qubits':n_qubits,'encoding':'per-state RY angle encoding; mean-pool coordinates per qubit; angle=pi*tanh(pooled value)','reservoir':'fixed ring CNOT layer after each state','readout':'single-qubit Pauli-Z expectations','shots':int(shots),'readout_depolarizing_probability':float(noise),'simulator':'numpy state-vector'}

def fit_classifier(features, labels, seed=0, max_iter=1000):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    return make_pipeline(StandardScaler(),LogisticRegression(max_iter=max_iter,class_weight=None,random_state=seed,multi_class='auto'))

def make_gru(input_dim, hidden_dim=64, classes=5):
    import torch
    from torch import nn
    return nn.GRU(input_dim,hidden_dim,batch_first=True), nn.Linear(hidden_dim,classes)


### Embedded source: `aptos_research/registry.py`

In [ ]:
%%writefile aptos_research/registry.py
"""Append-only JSONL experiment event log."""
import json,time
from pathlib import Path

def record(path, stage, status, config, artifacts=None, metrics=None, note=''):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    event={'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'stage':stage,'status':status,'config':config,'artifacts':artifacts or [],'metrics':metrics or {},'note':note}
    with p.open('a',encoding='utf-8') as f:f.write(json.dumps(event,sort_keys=True)+'\n')
    return event


### Embedded source: `aptos_research/run_qrc.py`

In [ ]:
%%writefile aptos_research/run_qrc.py
"""Gate-controlled QRC evaluation. Requires a PASS gate artifact."""
import argparse, json
from pathlib import Path
from aptos_research.features import FeatureStore
from aptos_research.gate import qrc_is_eligible
from aptos_research.models import qrc_features,fit_classifier,make_esn,esn_transform
from aptos_research.metrics import classification_metrics
from aptos_research.registry import record

def main():
 p=argparse.ArgumentParser();p.add_argument('--gate',required=True);p.add_argument('--cache',required=True);p.add_argument('--output',default='outputs/experiments/qrc.json');p.add_argument('--registry',default='outputs/experiments/registry.jsonl');p.add_argument('--qubits',type=int,default=4);p.add_argument('--shots',type=int,default=0);p.add_argument('--seed',type=int,default=2026);a=p.parse_args()
 if not qrc_is_eligible(a.gate):
  result={'status':'SKIPPED_GATE_NOT_PASS','gate':json.loads(Path(a.gate).read_text()).get('decision')}
  Path(a.output).parent.mkdir(parents=True,exist_ok=True);Path(a.output).write_text(json.dumps(result,indent=2)+'\n');record(a.registry,'qrc','SKIPPED_GATE_NOT_PASS',{'gate':result['gate']},{'result':str(Path(a.output).resolve())},result);print(json.dumps(result));return 2
 import numpy as np
 store=FeatureStore(a.cache);tr=store.load_split('train');va=store.load_split('validation');te=store.load_split('test')
 def seq(rows): return np.stack([x['states'] for x in rows])
 xt,meta=qrc_features(seq(tr),a.qubits,a.seed,a.shots); xv,_=qrc_features(seq(va),a.qubits,a.seed+1,a.shots); xe,_=qrc_features(seq(te),a.qubits,a.seed+2,a.shots)
 model=fit_classifier(xt,[x['label'] for x in tr],a.seed).fit(xt,[x['label'] for x in tr]); valp=model.predict_proba(xv); testp=model.predict_proba(xe)
 ytr=[x['label'] for x in tr]; yv=[x['label'] for x in va]; yt=[x['label'] for x in te]
 # Same compact trajectory inputs for a classical reservoir reference.
 xtr=seq(tr);xv0=seq(va);xte0=seq(te); reservoir=make_esn(xtr.shape[-1],64,a.seed)
 er=fit_classifier(esn_transform(xtr,reservoir),ytr,a.seed).fit(esn_transform(xtr,reservoir),ytr)
 evp=er.predict_proba(esn_transform(xv0,reservoir));etp=er.predict_proba(esn_transform(xte0,reservoir))
 result={'status':'COMPLETED_CONDITIONAL','gate_decision':'PASS','config':meta,'validation':{'qrc':classification_metrics(yv,valp.tolist()),'esn':classification_metrics(yv,evp.tolist())},'test':{'qrc':classification_metrics(yt,testp.tolist()),'esn':classification_metrics(yt,etp.tolist())},'resource_accounting':{'qrc_readout_parameters':int(xt.shape[1]*5+5),'esn_reservoir_parameters':int(reservoir['input'].size+reservoir['reservoir'].size),'esn_readout_parameters':int(er[-1].coef_.size+er[-1].intercept_.size),'state_dimension':int(xtr.shape[-1]),'trajectory_length':int(xtr.shape[1]),'quantum_hardware_execution':False},'note':'Resource and predictive comparisons are simulator-only; test results are descriptive and do not change the precomputed gate.'}
 Path(a.output).parent.mkdir(parents=True,exist_ok=True);Path(a.output).write_text(json.dumps(result,indent=2)+'\n');record(a.registry,'qrc','COMPLETED_CONDITIONAL',meta,{'result':str(Path(a.output).resolve())},result['test']);print(json.dumps(result,indent=2))
if __name__=='__main__':raise SystemExit(main())


### Embedded source: `aptos_research/train_gru.py`

In [ ]:
%%writefile aptos_research/train_gru.py
"""Small deterministic GRU sequence classifier for [N,K,D] feature arrays."""
def train_gru(train_x,train_y,val_x,val_y,test_x=None,seed=2026,hidden=64,epochs=100,patience=12,device='cpu'):
 import copy, numpy as np, torch
 from torch import nn
 torch.manual_seed(seed); np.random.seed(seed)
 class Net(nn.Module):
  def __init__(self,d,h):
   super().__init__();self.gru=nn.GRU(d,h,batch_first=True);self.head=nn.Linear(h,5)
  def forward(self,x):return self.head(self.gru(x)[0][:,-1])
 model=Net(train_x.shape[-1],hidden).to(device); opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4); loss_fn=nn.CrossEntropyLoss()
 tx=torch.tensor(train_x,dtype=torch.float32,device=device);ty=torch.tensor(train_y,dtype=torch.long,device=device);vx=torch.tensor(val_x,dtype=torch.float32,device=device);vy=torch.tensor(val_y,dtype=torch.long,device=device)
 best=None;best_loss=float('inf');stale=0
 for _ in range(epochs):
  model.train()
  for indices in torch.randperm(len(tx),device=device).split(128):
   opt.zero_grad();loss_fn(model(tx[indices]),ty[indices]).backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
  model.eval()
  with torch.no_grad(): score=loss_fn(model(vx),vy).item()
  if score<best_loss:best_loss=score;best=copy.deepcopy(model.state_dict());stale=0
  else: stale+=1
  if stale>=patience:break
 model.load_state_dict(best);model.eval()
 with torch.no_grad():
  vp=model(vx).softmax(1).cpu().numpy()
  tx_test=torch.tensor(test_x if test_x is not None else val_x,dtype=torch.float32,device=device)
  tp=model(tx_test).softmax(1).cpu().numpy()
 return model,vp,tp,sum(p.numel() for p in model.parameters())


### Embedded source: `aptos_research/trust.py`

In [ ]:
%%writefile aptos_research/trust.py
"""Training-only Mahalanobis OOD scores and selective risk/coverage."""
def fit_mahalanobis(train_features, ridge=1e-4):
 import numpy as np
 x=np.asarray(train_features,dtype=float)
 if x.ndim!=2 or len(x)<2: raise ValueError('training features must be [N,D], N>=2')
 mean=x.mean(axis=0); cov=np.cov(x,rowvar=False)
 cov=np.atleast_2d(cov)+ridge*np.eye(x.shape[1])
 return {'mean':mean,'precision':np.linalg.pinv(cov),'ridge':ridge}
def mahalanobis(model, features):
 import numpy as np
 d=np.asarray(features,dtype=float)-model['mean'];return np.sqrt(np.maximum(0,np.einsum('bi,ij,bj->b',d,model['precision'],d)))
def risk_coverage(labels,probabilities):
 import numpy as np
 y=np.asarray(labels);p=np.asarray(probabilities);conf=p.max(axis=1);pred=p.argmax(axis=1);order=np.argsort(-conf);err=(pred!=y)[order]
 return [{'coverage':float(i/len(y)),'risk':float(err[:i].mean()) if i else 0.0,'threshold':float(conf[order[i-1]]) if i else 1.0} for i in range(1,len(y)+1)]
def corrupt_brightness(images,offset):
 import numpy as np
 return np.clip(np.asarray(images,dtype=float)+float(offset),0,1)


### Embedded source: `aptos_research/trust_eval.py`

In [ ]:
%%writefile aptos_research/trust_eval.py
"""Evaluate calibration, selective risk and train-fitted Mahalanobis distances from saved outputs."""
import argparse,json
from pathlib import Path
import numpy as np
from aptos_research.features import FeatureStore
from aptos_research.metrics import classification_metrics
from aptos_research.trust import fit_mahalanobis,mahalanobis,risk_coverage
from aptos_research.registry import record

def reliability_plot(labels, probabilities, path, bins=10):
 import matplotlib.pyplot as plt
 confidence=[max(row) for row in probabilities]; correct=[max(range(len(row)),key=lambda j:row[j])==int(y) for y,row in zip(labels,probabilities)]
 xs=[];ys=[]
 for b in range(bins):
  indices=[i for i,c in enumerate(confidence) if b/bins<=c<(b+1)/bins or (b==bins-1 and c==1)]
  if indices: xs.append(sum(confidence[i] for i in indices)/len(indices));ys.append(sum(correct[i] for i in indices)/len(indices))
 fig,ax=plt.subplots(figsize=(5,5));ax.plot([0,1],[0,1],'--',color='gray');ax.plot(xs,ys,'o-');ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean confidence',ylabel='Accuracy',title='Reliability diagram');fig.tight_layout();fig.savefig(path,dpi=160);plt.close(fig)

def main():
 p=argparse.ArgumentParser();p.add_argument('--cache',required=True);p.add_argument('--results',required=True);p.add_argument('--model',default='ordered_esn_k16');p.add_argument('--output',default='outputs/experiments/trust.json');a=p.parse_args()
 experiment=json.loads(Path(a.results).read_text());store=FeatureStore(a.cache);tr=store.load_split('train');te=store.load_split('test')
 train_features=np.stack([x['states'][-1] for x in tr]);test_features=np.stack([x['states'][-1] for x in te]);stats=fit_mahalanobis(train_features)
 prediction=experiment['predictions'][a.model];labels=experiment['test']['labels'];probs=prediction['test_probabilities']
 result={'model':a.model,'classification':classification_metrics(labels,probs),'test_mahalanobis_distance':mahalanobis(stats,test_features).tolist(),'selective_risk_coverage':risk_coverage(labels,probs),'protocol':{'mahalanobis_fit':'training endpoint representations only','ece_bins':10,'risk_coverage':'descending maximum class confidence'}}
 Path(a.output).parent.mkdir(parents=True,exist_ok=True);Path(a.output).write_text(json.dumps(result,indent=2)+'\n')
 plot_path=str(Path(a.output).with_suffix('.reliability.png'));reliability_plot(labels,probs,plot_path)
 record('outputs/experiments/registry.jsonl','trust-evaluation','COMPLETED',{'model':a.model},{'result':str(Path(a.output).resolve()),'reliability_diagram':plot_path},result['classification']);print('saved',a.output)
if __name__=='__main__':main()


### Embedded source: `aptos_research/xai_eval.py`

In [ ]:
%%writefile aptos_research/xai_eval.py
"""Plot class-score-derived probability trajectories for IDs in the feature cache."""
import argparse
from pathlib import Path
from aptos_research.features import FeatureStore
from aptos_research.explain import class_scores_to_probabilities,save_probability_trajectory
from aptos_research.registry import record

def main():
 p=argparse.ArgumentParser();p.add_argument('--cache',required=True);p.add_argument('--sample-id',required=True);p.add_argument('--split',choices=['train','validation','test'],default='test');p.add_argument('--output',default='outputs/experiments/xai');a=p.parse_args()
 rows=FeatureStore(a.cache).load_split(a.split);row=next((x for x in rows if x['id']==a.sample_id),None)
 if row is None: raise SystemExit('sample id not found in cache split: '+a.sample_id)
 shape=row['config'].get('input_state_shape')
 if not shape or len(shape)!=3: raise SystemExit('state cache lacks verified [classes,height,width] metadata')
 probs=class_scores_to_probabilities(row['states'],int(shape[0]),(int(shape[1]),int(shape[2])))
 target=Path(a.output);target.mkdir(parents=True,exist_ok=True);path=target/(a.sample_id+'_trajectory.png')
 save_probability_trajectory(row['timesteps'],probs,str(path),'DiffMIC class-score state softmax: '+a.sample_id)
 record('outputs/experiments/registry.jsonl','trajectory-explanation','COMPLETED',{'sample_id':a.sample_id,'split':a.split,'derived':'spatially averaged class-score grid followed by softmax'},{'plot':str(path.resolve())},{'grade':row['label']});print(path)
if __name__=='__main__':main()


### Embedded source: `dataloader/functional.py`

In [ ]:
%%writefile dataloader/functional.py
import math
import random

from PIL import Image, ImageEnhance, ImageOps

try:
    import accimage
except ImportError:
    accimage = None
import collections
import numbers
import types
import warnings

import cv2
import numpy as np
from PIL import Image

_cv2_pad_to_str = {
    'constant': cv2.BORDER_CONSTANT,
    'edge': cv2.BORDER_REPLICATE,
    'reflect': cv2.BORDER_REFLECT_101,
    'symmetric': cv2.BORDER_REFLECT
}
_cv2_interpolation_to_str = {
    'nearest': cv2.INTER_NEAREST,
    'bilinear': cv2.INTER_LINEAR,
    'area': cv2.INTER_AREA,
    'bicubic': cv2.INTER_CUBIC,
    'lanczos': cv2.INTER_LANCZOS4
}
_cv2_interpolation_from_str = {v: k for k, v in _cv2_interpolation_to_str.items()}


def _is_pil_image(img):
    if accimage is not None:
        return isinstance(img, (Image.Image, accimage.Image))
    else:
        return isinstance(img, Image.Image)


def _is_numpy_image(img):
    return isinstance(img, np.ndarray) and (img.ndim in {2, 3, 4})

def resize(img, size, interpolation=cv2.INTER_LINEAR):
    r"""Resize the input numpy ndarray to the given size.
    Args:
        img (numpy ndarray): Image to be resized.
        size (sequence or int): Desired output size. If size is a sequence like
            (h, w), the output size will be matched to this. If size is an int,
            the smaller edge of the image will be matched to this number maintaing
            the aspect ratio. i.e, if height > width, then image will be rescaled to
            :math:`\left(\text{size} \times \frac{\text{height}}{\text{width}}, \text{size}\right)`
        interpolation (int, optional): Desired interpolation. Default is
            ``cv2.INTER_LINEAR``
    Returns:
        PIL Image: Resized image.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy image. Got {}'.format(type(img)))
    if not (isinstance(size, int) or (isinstance(size, collections.abc.Iterable) and len(size) == 2)):
        raise TypeError('Got inappropriate size arg: {}'.format(size))
    h, w = img.shape[0], img.shape[1]

    if isinstance(size, int):
        if (w <= h and w == size) or (h <= w and h == size):
            return img
        if w < h:
            ow = size
            oh = int(size * h / w)
        else:
            oh = size
            ow = int(size * w / h)
    else:
        ow, oh = size[1], size[0]
    output = cv2.resize(img, dsize=(ow, oh), interpolation=interpolation)
    if img.shape[2] == 1:
        return output[:, :, np.newaxis]
    else:
        return output


def scale(*args, **kwargs):
    warnings.warn("The use of the transforms.Scale transform is deprecated, " + "please use transforms.Resize instead.")
    return resize(*args, **kwargs)


def pad(img, padding, fill=0, padding_mode='constant'):
    r"""Pad the given numpy ndarray on all sides with specified padding mode and fill value.
    Args:
        img (numpy ndarray): image to be padded.
        padding (int or tuple): Padding on each border. If a single int is provided this
            is used to pad all borders. If tuple of length 2 is provided this is the padding
            on left/right and top/bottom respectively. If a tuple of length 4 is provided
            this is the padding for the left, top, right and bottom borders
            respectively.
        fill: Pixel fill value for constant fill. Default is 0. If a tuple of
            length 3, it is used to fill R, G, B channels respectively.
            This value is only used when the padding_mode is constant
        padding_mode: Type of padding. Should be: constant, edge, reflect or symmetric. Default is constant.
            - constant: pads with a constant value, this value is specified with fill
            - edge: pads with the last value on the edge of the image
            - reflect: pads with reflection of image (without repeating the last value on the edge)
                       padding [1, 2, 3, 4] with 2 elements on both sides in reflect mode
                       will result in [3, 2, 1, 2, 3, 4, 3, 2]
            - symmetric: pads with reflection of image (repeating the last value on the edge)
                         padding [1, 2, 3, 4] with 2 elements on both sides in symmetric mode
                         will result in [2, 1, 1, 2, 3, 4, 4, 3]
    Returns:
        Numpy image: padded image.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy ndarray. Got {}'.format(type(img)))
    if not isinstance(padding, (numbers.Number, tuple, list)):
        raise TypeError('Got inappropriate padding arg')
    if not isinstance(fill, (numbers.Number, str, tuple)):
        raise TypeError('Got inappropriate fill arg')
    if not isinstance(padding_mode, str):
        raise TypeError('Got inappropriate padding_mode arg')
    if isinstance(padding, collections.Sequence) and len(padding) not in [2, 4]:
        raise ValueError("Padding must be an int or a 2, or 4 element tuple, not a " +
                         "{} element tuple".format(len(padding)))

    assert padding_mode in ['constant', 'edge', 'reflect', 'symmetric'], \
        'Padding mode should be either constant, edge, reflect or symmetric'

    if isinstance(padding, int):
        pad_left = pad_right = pad_top = pad_bottom = padding
    if isinstance(padding, collections.Sequence) and len(padding) == 2:
        pad_left = pad_right = padding[0]
        pad_top = pad_bottom = padding[1]
    if isinstance(padding, collections.Sequence) and len(padding) == 4:
        pad_left = padding[0]
        pad_top = padding[1]
        pad_right = padding[2]
        pad_bottom = padding[3]
    if img.shape[2] == 1:
        return cv2.copyMakeBorder(img,
                                  top=pad_top,
                                  bottom=pad_bottom,
                                  left=pad_left,
                                  right=pad_right,
                                  borderType=_cv2_pad_to_str[padding_mode],
                                  value=fill)[:, :, np.newaxis]
    else:
        return cv2.copyMakeBorder(img,
                                  top=pad_top,
                                  bottom=pad_bottom,
                                  left=pad_left,
                                  right=pad_right,
                                  borderType=_cv2_pad_to_str[padding_mode],
                                  value=fill)


def crop(img, i, j, h, w):
    """Crop the given PIL Image.
    Args:
        img (numpy ndarray): Image to be cropped.
        i: Upper pixel coordinate.
        j: Left pixel coordinate.
        h: Height of the cropped image.
        w: Width of the cropped image.
    Returns:
        numpy ndarray: Cropped image.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy image. Got {}'.format(type(img)))

    return img[i:i + h, j:j + w, :]

def crop3d(img, i, j, k, d, h, w):
    """Crop the given PIL Image.
    Args:
        img (numpy ndarray): Image to be cropped.
    Returns:
        numpy ndarray: Cropped image.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy image. Got {}'.format(type(img)))

    return img[i:i + d, j:j + h, k:k+w, :]


def center_crop(img, output_size):
    if len(img.shape) == 3:
        if isinstance(output_size, numbers.Number):
            output_size = (int(output_size), int(output_size))
        h, w = img.shape[0:2]
        th, tw = output_size
        i = int(round((h - th) / 2.))
        j = int(round((w - tw) / 2.))
        return crop(img, i, j, th, tw)
    elif len(img.shape) == 4:
        d, h, w = img.shape[:3]
        if isinstance(output_size, numbers.Number):
            output_size = (int(output_size), int(output_size), int(output_size))
        td, th, tw = output_size
        i = int(round((d - td) / 2.))
        j = int(round((h - th) / 2.))
        k = int(round((w - tw) / 2.))
        return crop3d(img, i, j, k, td, th, tw)


def resized_crop(img, i, j, h, w, size, interpolation=cv2.INTER_LINEAR):
    """Crop the given numpy ndarray and resize it to desired size.
    Notably used in :class:`~torchvision.transforms.RandomResizedCrop`.
    Args:
        img (numpy ndarray): Image to be cropped.
        i: Upper pixel coordinate.
        j: Left pixel coordinate.
        h: Height of the cropped image.
        w: Width of the cropped image.
        size (sequence or int): Desired output size. Same semantics as ``scale``.
        interpolation (int, optional): Desired interpolation. Default is
            ``cv2.INTER_CUBIC``.
    Returns:
        PIL Image: Cropped image.
    """
    assert _is_numpy_image(img), 'img should be numpy image'
    img = crop(img, i, j, h, w)
    img = resize(img, size, interpolation=interpolation)
    return img

def flip(im, axis):
    """
    axis: str
          z - flip along Depth  (z-axis)
          y - flip along Height (y-axis)
          x - flip along Width  (x-axis)
    """
    if len(im.shape) == 4: # D, H, W, C
        axis = {"z": 0, "y": 1, "x": 2}[axis]
    elif len(im.shape) == 3: # H, W, C
        axis = {"x": 0, "y": 1}[axis]
    im = np.flip(im, axis)
    return im

def five_crop(img, size):
    """Crop the given numpy ndarray into four corners and the central crop.
    .. Note::
        This transform returns a tuple of images and there may be a
        mismatch in the number of inputs and targets your ``Dataset`` returns.
    Args:
       size (sequence or int): Desired output size of the crop. If size is an
           int instead of sequence like (h, w), a square crop (size, size) is
           made.
    Returns:
       tuple: tuple (tl, tr, bl, br, center)
                Corresponding top left, top right, bottom left, bottom right and center crop.
    """
    if isinstance(size, numbers.Number):
        size = (int(size), int(size))
    else:
        assert len(size) == 2, "Please provide only two dimensions (h, w) for size."

    h, w = img.shape[0:2]
    crop_h, crop_w = size
    if crop_w > w or crop_h > h:
        raise ValueError("Requested crop size {} is bigger than input size {}".format(size, (h, w)))
    tl = crop(img, 0, 0, crop_h, crop_w)
    tr = crop(img, 0, w - crop_w, crop_h, crop_w)
    bl = crop(img, h - crop_h, 0, crop_h, crop_w)
    br = crop(img, h - crop_h, w - crop_w, crop_h, crop_w)
    center = center_crop(img, (crop_h, crop_w))
    return tl, tr, bl, br, center


def ten_crop(img, size, vertical_flip=False):
    r"""Crop the given numpy ndarray into four corners and the central crop plus the
        flipped version of these (horizontal flipping is used by default).
    .. Note::
        This transform returns a tuple of images and there may be a
        mismatch in the number of inputs and targets your ``Dataset`` returns.
    Args:
       size (sequence or int): Desired output size of the crop. If size is an
            int instead of sequence like (h, w), a square crop (size, size) is
            made.
       vertical_flip (bool): Use vertical flipping instead of horizontal
    Returns:
       tuple: tuple (tl, tr, bl, br, center, tl_flip, tr_flip, bl_flip, br_flip, center_flip)
                Corresponding top left, top right, bottom left, bottom right and center crop
                and same for the flipped image.
    """
    if isinstance(size, numbers.Number):
        size = (int(size), int(size))
    else:
        assert len(size) == 2, "Please provide only two dimensions (h, w) for size."

    first_five = five_crop(img, size)

    if vertical_flip:
        img = vflip(img)
    else:
        img = hflip(img)

    second_five = five_crop(img, size)
    return first_five + second_five


def adjust_brightness(img, brightness_factor):
    """Adjust brightness of an Image.
    Args:
        img (numpy ndarray): numpy ndarray to be adjusted.
        brightness_factor (float):  How much to adjust the brightness. Can be
            any non negative number. 0 gives a black image, 1 gives the
            original image while 2 increases the brightness by a factor of 2.
    Returns:
        numpy ndarray: Brightness adjusted image.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))
    table = np.array([i * brightness_factor for i in range(0, 256)]).clip(0, 255).astype('uint8')
    # same thing but a bit slower
    # cv2.convertScaleAbs(img, alpha=brightness_factor, beta=0)
    if img.shape[2] == 1:
        return cv2.LUT(img, table)[:, :, np.newaxis]
    else:
        return cv2.LUT(img, table)


def adjust_contrast(img, contrast_factor):
    """Adjust contrast of an mage.
    Args:
        img (numpy ndarray): numpy ndarray to be adjusted.
        contrast_factor (float): How much to adjust the contrast. Can be any
            non negative number. 0 gives a solid gray image, 1 gives the
            original image while 2 increases the contrast by a factor of 2.
    Returns:
        numpy ndarray: Contrast adjusted image.
    """
    # much faster to use the LUT construction than anything else I've tried
    # it's because you have to change dtypes multiple times
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))

    # input is RGB
    if img.ndim > 2 and img.shape[2] == 3:
        mean_value = round(cv2.mean(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY))[0])
    elif img.ndim == 2:
        # grayscale input
        mean_value = round(cv2.mean(img)[0])
    else:
        # multichannel input
        mean_value = round(np.mean(img))

    table = np.array([(i - mean_value) * contrast_factor + mean_value for i in range(0, 256)]).clip(0,
                                                                                                    255).astype('uint8')
    # enhancer = ImageEnhance.Contrast(img)
    # img = enhancer.enhance(contrast_factor)
    if img.ndim == 2 or img.shape[2] == 1:
        return cv2.LUT(img, table)[:, :, np.newaxis]
    else:
        return cv2.LUT(img, table)


def adjust_saturation(img, saturation_factor):
    """Adjust color saturation of an image.
    Args:
        img (numpy ndarray): numpy ndarray to be adjusted.
        saturation_factor (float):  How much to adjust the saturation. 0 will
            give a black and white image, 1 will give the original image while
            2 will enhance the saturation by a factor of 2.
    Returns:
        numpy ndarray: Saturation adjusted image.
    """
    # ~10ms slower than PIL!
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))
    img = Image.fromarray(img)
    enhancer = ImageEnhance.Color(img)
    img = enhancer.enhance(saturation_factor)
    return np.array(img)


def adjust_hue(img, hue_factor):
    """Adjust hue of an image.
    The image hue is adjusted by converting the image to HSV and
    cyclically shifting the intensities in the hue channel (H).
    The image is then converted back to original image mode.
    `hue_factor` is the amount of shift in H channel and must be in the
    interval `[-0.5, 0.5]`.
    See `Hue`_ for more details.
    .. _Hue: https://en.wikipedia.org/wiki/Hue
    Args:
        img (numpy ndarray): numpy ndarray to be adjusted.
        hue_factor (float):  How much to shift the hue channel. Should be in
            [-0.5, 0.5]. 0.5 and -0.5 give complete reversal of hue channel in
            HSV space in positive and negative direction respectively.
            0 means no shift. Therefore, both -0.5 and 0.5 will give an image
            with complementary colors while 0 gives the original image.
    Returns:
        numpy ndarray: Hue adjusted image.
    """
    # After testing, found that OpenCV calculates the Hue in a call to
    # cv2.cvtColor(..., cv2.COLOR_BGR2HSV) differently from PIL

    # This function takes 160ms! should be avoided
    if not (-0.5 <= hue_factor <= 0.5):
        raise ValueError('hue_factor is not in [-0.5, 0.5].'.format(hue_factor))
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))
    img = Image.fromarray(img)
    input_mode = img.mode
    if input_mode in {'L', '1', 'I', 'F'}:
        return np.array(img)

    h, s, v = img.convert('HSV').split()

    np_h = np.array(h, dtype=np.uint8)
    # uint8 addition take cares of rotation across boundaries
    with np.errstate(over='ignore'):
        np_h += np.uint8(hue_factor * 255)
    h = Image.fromarray(np_h, 'L')

    img = Image.merge('HSV', (h, s, v)).convert(input_mode)
    return np.array(img)


def adjust_gamma(img, gamma, gain=1):
    r"""Perform gamma correction on an image.
    Also known as Power Law Transform. Intensities in RGB mode are adjusted
    based on the following equation:
    .. math::
        I_{\text{out}} = 255 \times \text{gain} \times \left(\frac{I_{\text{in}}}{255}\right)^{\gamma}
    See `Gamma Correction`_ for more details.
    .. _Gamma Correction: https://en.wikipedia.org/wiki/Gamma_correction
    Args:
        img (numpy ndarray): numpy ndarray to be adjusted.
        gamma (float): Non negative real number, same as :math:`\gamma` in the equation.
            gamma larger than 1 make the shadows darker,
            while gamma smaller than 1 make dark regions lighter.
        gain (float): The constant multiplier.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))

    if gamma < 0:
        raise ValueError('Gamma should be a non-negative real number')
    # from here
    # https://stackoverflow.com/questions/33322488/how-to-change-image-illumination-in-opencv-python/41061351
    table = np.array([((i / 255.0)**gamma) * 255 * gain for i in np.arange(0, 256)]).astype('uint8')
    if img.shape[2] == 1:
        return cv2.LUT(img, table)[:, :, np.newaxis]
    else:
        return cv2.LUT(img, table)


def rotate(img, angle, resample=False, expand=False, center=None):
    """Rotate the image by angle.
    Args:
        img (numpy ndarray): numpy ndarray to be rotated.
        angle (float or int): In degrees degrees counter clockwise order.
        resample (``PIL.Image.NEAREST`` or ``PIL.Image.BILINEAR`` or ``PIL.Image.BICUBIC``, optional):
            An optional resampling filter. See `filters`_ for more information.
            If omitted, or if the image has mode "1" or "P", it is set to ``PIL.Image.NEAREST``.
        expand (bool, optional): Optional expansion flag.
            If true, expands the output image to make it large enough to hold the entire rotated image.
            If false or omitted, make the output image the same size as the input image.
            Note that the expand flag assumes rotation around the center and no translation.
        center (2-tuple, optional): Optional center of rotation.
            Origin is the upper left corner.
            Default is the center of the image.
    .. _filters: https://pillow.readthedocs.io/en/latest/handbook/concepts.html#filters
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))
    rows, cols = img.shape[0:2]
    if center is None:
        center = (cols / 2, rows / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1)
    if img.shape[2] == 1:
        return cv2.warpAffine(img, M, (cols, rows))[:, :, np.newaxis]
    else:
        return cv2.warpAffine(img, M, (cols, rows))


def _get_affine_matrix(center, angle, translate, scale, shear):
    # Helper method to compute matrix for affine transformation
    # We need compute affine transformation matrix: M = T * C * RSS * C^-1
    # where T is translation matrix: [1, 0, tx | 0, 1, ty | 0, 0, 1]
    #       C is translation matrix to keep center: [1, 0, cx | 0, 1, cy | 0, 0, 1]
    #       RSS is rotation with scale and shear matrix
    #       RSS(a, scale, shear) = [ cos(a)*scale    -sin(a + shear)*scale     0]
    #                              [ sin(a)*scale    cos(a + shear)*scale     0]
    #                              [     0                  0          1]

    angle = math.radians(angle)
    shear = math.radians(shear)
    # scale = 1.0 / scale

    T = np.array([[1, 0, translate[0]], [0, 1, translate[1]], [0, 0, 1]])
    C = np.array([[1, 0, center[0]], [0, 1, center[1]], [0, 0, 1]])
    RSS = np.array([[math.cos(angle) * scale, -math.sin(angle + shear) * scale, 0],
                    [math.sin(angle) * scale, math.cos(angle + shear) * scale, 0], [0, 0, 1]])
    matrix = T @ C @ RSS @ np.linalg.inv(C)

    return matrix[:2, :]


def affine(img, angle, translate, scale, shear, interpolation=cv2.INTER_LINEAR, mode=cv2.BORDER_CONSTANT, fillcolor=0):
    """Apply affine transformation on the image keeping image center invariant
    Args:
        img (numpy ndarray): numpy ndarray to be transformed.
        angle (float or int): rotation angle in degrees between -180 and 180, clockwise direction.
        translate (list or tuple of integers): horizontal and vertical translations (post-rotation translation)
        scale (float): overall scale
        shear (float): shear angle value in degrees between -180 to 180, clockwise direction.
        interpolation (``cv2.INTER_NEAREST` or ``cv2.INTER_LINEAR`` or ``cv2.INTER_AREA``, ``cv2.INTER_CUBIC``):
            An optional resampling filter.
            See `filters`_ for more information.
            If omitted, it is set to ``cv2.INTER_CUBIC``, for bicubic interpolation.
        mode (``cv2.BORDER_CONSTANT`` or ``cv2.BORDER_REPLICATE`` or ``cv2.BORDER_REFLECT`` or ``cv2.BORDER_REFLECT_101``)
            Method for filling in border regions.
            Defaults to cv2.BORDER_CONSTANT, meaning areas outside the image are filled with a value (val, default 0)
        val (int): Optional fill color for the area outside the transform in the output image. Default: 0
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy Image. Got {}'.format(type(img)))

    assert isinstance(translate, (tuple, list)) and len(translate) == 2, \
        "Argument translate should be a list or tuple of length 2"

    assert scale > 0.0, "Argument scale should be positive"

    output_size = img.shape[0:2]
    center = (img.shape[1] * 0.5 + 0.5, img.shape[0] * 0.5 + 0.5)
    matrix = _get_affine_matrix(center, angle, translate, scale, shear)

    if img.shape[2] == 1:
        return cv2.warpAffine(img, matrix, output_size[::-1], interpolation, borderMode=mode,
                              borderValue=fillcolor)[:, :, np.newaxis]
    else:
        return cv2.warpAffine(img, matrix, output_size[::-1], interpolation, borderMode=mode, borderValue=fillcolor)


def to_grayscale(img, num_output_channels: int = 1):
    """Convert image to grayscale version of image.
    Args:
        img (numpy ndarray): Image to be converted to grayscale.
        num_output_channels: int
            if 1 : returned image is single channel
            if 3 : returned image is 3 channel with r = g = b
    Returns:
        numpy ndarray: Grayscale version of the image.
    """
    if not _is_numpy_image(img):
        raise TypeError('img should be numpy ndarray. Got {}'.format(type(img)))

    if num_output_channels == 1:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[:, :, np.newaxis]
    elif num_output_channels == 3:
        # much faster than doing cvtColor to go back to gray
        img = np.broadcast_to(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[:, :, np.newaxis], img.shape)
    return img

### Embedded source: `dataloader/loading.py`

In [ ]:
%%writefile dataloader/loading.py
import os, torch, cv2, random
import numpy as np
from torch.utils.data import Dataset, Sampler
import torchvision.transforms as transforms
from scipy.ndimage.morphology import binary_erosion
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from skimage import filters
import numpy as np
import imageio
import dataloader.transforms as trans
import json, numbers
from glob import glob
import pickle

class BUDataset(Dataset):
    def __init__(self, data_list, train=True):
        self.trainsize = (224,224)
        self.train = train
        with open(data_list, "rb") as f:
            tr_dl = pickle.load(f)
        self.data_list = tr_dl

        self.size = len(self.data_list)
        #print(self.size)
        if train:
            self.transform_center = transforms.Compose([
                trans.CropCenterSquare(),
                transforms.Resize(self.trainsize),
                #trans.CenterCrop(self.trainsize),
                trans.RandomHorizontalFlip(),
                # trans.RandomVerticalFlip(),
                trans.RandomRotation(30),
                # trans.adjust_light(),
                transforms.ToTensor(),
                # lambda x: x*255
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
        else:
            self.transform_center = transforms.Compose([
                trans.CropCenterSquare(),
                transforms.Resize(self.trainsize),
                #trans.CenterCrop(self.trainsize),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                # lambda x: x*255
                ])


    def __getitem__(self, index):
        data_pac = self.data_list[index]
        img_path = data_pac['img_root']
        #cl_img, cr_img, ml_img, mr_img = None
        img = Image.open(img_path).convert('RGB')

        img_torch = self.transform_center(img)

        label = int(data_pac['label'])

        
        return img_torch, label


    def __len__(self):
        return self.size


class APTOSDataset(Dataset):
    def __init__(self, data_list, train=True):
        self.trainsize = (224,224)
        self.train = train
        with open(data_list, "rb") as f:
            tr_dl = pickle.load(f)
        self.data_list = tr_dl

        self.size = len(self.data_list)
        #print(self.size)
        if train:
            self.transform_center = transforms.Compose([
                trans.CropCenterSquare(),
                transforms.Resize(self.trainsize),
                #trans.CenterCrop(self.trainsize),
                trans.RandomHorizontalFlip(),
                trans.RandomVerticalFlip(),
                trans.RandomRotation(30),
                # transforms.GaussianBlur(3),
                #trans.adjust_light(),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
        else:
            self.transform_center = transforms.Compose([
                trans.CropCenterSquare(),
                transforms.Resize(self.trainsize),
                #trans.CenterCrop(self.trainsize),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])

        #self.depths_transform = transforms.Compose([transforms.Resize((self.trainsize, self.trainsize)),transforms.ToTensor()])

    def __getitem__(self, index):
        data_pac = self.data_list[index]
        img_path = data_pac['img_root']
        #cl_img, cr_img, ml_img, mr_img = None
        img = Image.open(img_path).convert('RGB')

        img_torch = self.transform_center(img)

        label = int(data_pac['label'])

        
        return img_torch, label

    def __len__(self):
        return self.size



class ISICDataset(Dataset):
    def __init__(self, data_list, train=True):
        self.trainsize = (224,224)
        self.train = train
        with open(data_list, "rb") as f:
            tr_dl = pickle.load(f)
        self.data_list = tr_dl

        self.size = len(self.data_list)

        if train:
            self.transform_center = transforms.Compose([
                trans.CropCenterSquare(),
                transforms.Resize(self.trainsize),
                #trans.CenterCrop(self.trainsize),
                trans.RandomHorizontalFlip(),
                #trans.RandomVerticalFlip(),
                trans.RandomRotation(30),
                #trans.adjust_light(),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
        else:
            self.transform_center = transforms.Compose([
                trans.CropCenterSquare(),
                transforms.Resize(self.trainsize),
                #trans.CenterCrop(self.trainsize),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
        #self.depths_transform = transforms.Compose([transforms.Resize((self.trainsize, self.trainsize)),transforms.ToTensor()])

    def __getitem__(self, index):
        data_pac = self.data_list[index]
        img_path = data_pac['img_root']
        #cl_img, cr_img, ml_img, mr_img = None
        img = Image.open(img_path).convert('RGB')

        img_torch = self.transform_center(img)

        label = int(data_pac['label'])

        
        return img_torch, label


    def __len__(self):
        return self.size


class ChestXrayDataSet(Dataset):
    def __init__(self, image_list_file, train=True):
        """
        self:
            data_dir: path to image directory.
            image_list_file: path to the file containing images
                with corresponding labels.
            transform: optional transform to be applied on a sample.
        """
        data_dir = "dataset/chest/all/images/images"
        # data_dir = "/home/yijun/project/DiffMIC/dataset/chest/all/images_enhanced"
        self.trainsize = (256, 256)
        image_names = []
        labels = []
        with open(image_list_file, "r") as f:
            for line in f:
                items = line.split()
                image_name= items[0]
                label = items[1:]
                label = [int(i) for i in label]
                label.append(1) if (np.array(label)==0).all() else label.append(0)
                image_name = os.path.join(data_dir, image_name)
                image_names.append(image_name)
                labels.append(label)

        self.image_names = image_names
        self.labels = labels
        #print(len(self.image_names))
        normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])
        if train:
            self.transform_center = transforms.Compose([
                                        transforms.RandomResizedCrop(224),
                                        
                                        # transforms.RandomCrop(224),
                                        # transforms.GaussianBlur(3),
                                        trans.RandomHorizontalFlip(),
                                        trans.RandomRotation(20),
                                        transforms.ToTensor(),
                                        normalize
                                        #transforms.TenCrop(224),
                                        #transforms.Lambda
                                        #(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops])),
                                        #transforms.Lambda
                                        #(lambda crops: torch.stack([normalize(crop) for crop in crops]))
                                    ])
        else:
            # self.image_names = image_names[:1000]
            # self.labels = labels[:1000]
            self.transform_center = transforms.Compose([
                                        transforms.Resize(224),
                                        # transforms.TenCrop(224),
                                        # transforms.CenterCrop(224),
                                        transforms.ToTensor(),
                                        normalize
                                        # transforms.Lambda
                                        # (lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops])),
                                        # transforms.Lambda
                                        # (lambda crops: torch.stack([transforms.RandomHorizontalFlip()(crop) for crop in crops])),
                                        # # transforms.Lambda
                                        # # (lambda crops: torch.stack([transforms.RandomRotation(20)(crop) for crop in crops])),
                                        # transforms.Lambda
                                        # (lambda crops: torch.stack([normalize(crop) for crop in crops]))
                                    ])

    def __getitem__(self, index):
        """
        self:
            index: the index of item
        Returns:
            image and its labels
        """
        image_name = self.image_names[index]
        image = Image.open(image_name).convert('RGB')
        label = self.labels[index]
        image = self.transform_center(image)
        return image, torch.FloatTensor(label)

    def __len__(self):
        return len(self.image_names)


### Embedded source: `dataloader/transforms.py`

In [ ]:
%%writefile dataloader/transforms.py
from __future__ import division

import collections
import math
import numbers
import random
import types
import warnings

import cv2
import numpy as np
from PIL import Image, ImageOps



class CropCenterSquare(object):
    def __init__(self):
        pass

    def __call__(self, img):

        # assert img.width == mask.width
        # assert img.height == mask.height
        img_w, img_h = img.size
        h = min(img_h, img_w)
        crop = CenterCrop(h)

        return crop(img)


class CenterCrop(object):
    def __init__(self, size):
        if isinstance(size, numbers.Number):
            self.size = (int(size), int(size))
        else:
            self.size = size

    def __call__(self, img):

        # assert img.width == mask.width
        # assert img.height == mask.height
        w, h = img.size
        th, tw = self.size
        x1 = int(round((w - tw) / 2.))
        # y1 = int(round((h - th) / 2.))
        y1 = int(round((h - th) / 2.))
        img = img.crop((x1, y1, x1 + tw, y1 + th))

        return img


class RandomRotation(object):
    def __init__(self, degree):
        self.degree = degree

    def __call__(self, img):
        rotate_degree = random.random() * 2 * self.degree - self.degree
        return img.rotate(rotate_degree, Image.BILINEAR)

class RandomHorizontalFlip(object):
    def __call__(self, img):
        if random.random() < 0.5:
            return img.transpose(Image.FLIP_LEFT_RIGHT)
        return img


class RandomVerticalFlip(object):
    """Vertically flip the given PIL Image randomly with a given probability.
    Args:
        p (float): probability of the image being flipped. Default value is 0.5
    """
    def __call__(self, img):
        if random.random() < 0.5:
            return img.transpose(Image.FLIP_TOP_BOTTOM)
        return img


class adjust_light():
    def __call__(self, image):
        seed = random.random()
        if seed > 0.5:
            gamma = random.random() * 3 + 0.5
            invGamma = 1.0 / gamma
            table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype(np.uint8)
            image = cv2.LUT(np.array(image).astype(np.uint8), table).astype(np.uint8)

        return image


class RandomCrop(object):
    def __init__(self, size, padding=0):
        if isinstance(size, numbers.Number):
            self.size = (int(size), int(size))
        else:
            self.size = size
        self.padding = padding

    def __call__(self, img):
        if self.padding > 0:
            img = ImageOps.expand(img, border=self.padding, fill=0)

        w, h = img.size
        ch, cw = self.size
        if w == cw and h == ch:
            return img
        if w < cw or h < ch:
            pw = cw - w if cw > w else 0
            ph = ch - h if ch > h else 0
            padding = (pw,ph,pw,ph)
            img  = ImageOps.expand(img,  padding, fill=0)
            w, h = img.size
            
        x1 = random.randint(0, w - cw)
        y1 = random.randint(0, h - ch)
        return img.crop((x1, y1, x1 + cw, y1 + ch))

class RandomScaleCrop(object):
    def __init__(self, size):
        self.size = size
        self.crop = RandomCrop(self.size)
            
    def __call__(self, img):

        #r = random.uniform(0.5, 2.0)
        w, h = img.size
        new_size = (int(w*random.uniform(1, 1.5)),int(h*random.uniform(1, 1.5)))
        return self.crop(img.resize(new_size, Image.BILINEAR))



### Embedded source: `dataloader/transforms2.py`

In [ ]:
%%writefile dataloader/transforms2.py
import random
import torch
import numpy as np
import SimpleITK as sitk
import torch.nn.functional as F
from scipy import ndimage
from timm.models.layers import to_3tuple

def load_nii_file(nii_image):
    image = sitk.ReadImage(nii_image)
    image_array = sitk.GetArrayFromImage(image)
    return image_array

def resize3D(image, size):
    size = to_3tuple(size)
    image = image.astype(np.float32)
    image = torch.from_numpy(image).unsqueeze(0).unsqueeze(0)
    x = F.interpolate(image, size=size, mode='trilinear', align_corners=True).squeeze(0).squeeze(0)
    return x.cpu().numpy()

def image_normalization(image, win=None, adaptive=True):
    if win is not None:
        image = 1. * (image - win[0]) / (win[1] - win[0])
        image[image < 0] = 0.
        image[image > 1] = 1.
        return image
    elif adaptive:
        min, max = np.min(image), np.max(image)
        image = (image - min) / (max - min)
        return image
    else:
        return image

def random_crop(image, crop_shape):
    crop_shape = to_3tuple(crop_shape)
    _, z_shape, y_shape, x_shape = image.shape
    z_min = np.random.randint(0, z_shape - crop_shape[0])
    y_min = np.random.randint(0, y_shape - crop_shape[1])
    x_min = np.random.randint(0, x_shape - crop_shape[2])
    image = image[..., z_min:z_min+crop_shape[0], y_min:y_min+crop_shape[1], x_min:x_min+crop_shape[2]]
    return image

def center_crop(image, target_shape=(10, 80, 80)):
    target_shape = to_3tuple(target_shape)
    b, z_shape, y_shape, x_shape = image.shape
    z_min = z_shape // 2 - target_shape[0] // 2
    y_min = y_shape // 2 - target_shape[1] // 2
    x_min = x_shape // 2 - target_shape[2] // 2
    image = image[:, z_min:z_min+target_shape[0], y_min:y_min+target_shape[1], x_min:x_min+target_shape[2]]
    return image

def randomflip_z(image, p=0.5):
    if random.random() > p:
        return image
    else:
        return image[:, ::-1, ...]

def randomflip_x(image, p=0.5):
    if random.random() > p:
        return image
    else:
        return image[..., ::-1]

def randomflip_y(image, p=0.5):
    if random.random() > p:
        return image
    else:
        return image[:, :, ::-1, ...]

def random_flip(image, mode='x', p=0.5):
    if mode == 'x':
        image = randomflip_x(image, p=p)
    elif mode == 'y':
        image = randomflip_y(image, p=p)
    elif mode == 'z':
        image = randomflip_z(image, p=p)
    else:
        raise NotImplementedError(f'Unknown flip mode ({mode})')
    return image

def rotate(image, angle=10):
    angle = random.randint(-10, 10)
    r_image = ndimage.rotate(image, angle=angle, axes=(-2, -1), reshape=True)
    if r_image.shape != image.shape:
        r_image = center_crop(r_image, target_shape=image.shape[1:])
    return r_image

### Embedded source: `diffuser_trainer.py`

In [ ]:
%%writefile diffuser_trainer.py
from typing import Optional
import os
import numpy as np
import copy
from pytorch_lightning.utilities.types import EVAL_DATALOADERS, STEP_OUTPUT
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from tqdm import tqdm

import pytorch_lightning as pl
import yaml
from easydict import EasyDict
import random
from pytorch_lightning import callbacks
from pytorch_lightning.accelerators import accelerator
from pytorch_lightning.core.hooks import CheckpointHooks
from pytorch_lightning.callbacks import ModelCheckpoint,DeviceStatsMonitor,EarlyStopping,LearningRateMonitor
from pytorch_lightning.strategies import DDPStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from argparse import Namespace

from torch.utils.data import DataLoader
import pipeline

from torchvision.utils import save_image
from torchvision.models import vgg16
output_dir = 'logs'
version_name='Baseline'
logger = TensorBoardLogger(name='placental',save_dir = output_dir )
import matplotlib.pyplot as plt
# import tent
import math
from pretraining.dcg import DCG as AuxCls
from model import *
from utils import *


class CoolSystem(pl.LightningModule):
    
    def __init__(self, hparams):
        super(CoolSystem, self).__init__()

        self.params = hparams
        self.epochs = self.params.training.n_epochs
        self.initlr = self.params.optim.lr

        
        config_path = r'option/diff_DDIM.yaml'
        with open(config_path, 'r') as f:
            params = yaml.safe_load(f)
        config = EasyDict(params)
        self.diff_opt = config

        self.model = ConditionalModel(self.params, guidance=self.params.diffusion.include_guidance)
        self.aux_model = AuxCls(self.params)
        aux_checkpoint = getattr(self.params.data, 'aux_checkpoint', None)
        if aux_checkpoint:
            self.init_weight(ckpt_path=aux_checkpoint)
        self.aux_model.eval()

        self.save_hyperparameters()
        
        self.gts = []
        self.preds = []

        self.DiffSampler = pipeline.SR3Sampler(
            model=self.model,
            scheduler = pipeline.create_SR3scheduler(self.diff_opt['scheduler'], 'train'),
        )
        self.DiffSampler.scheduler.set_timesteps(self.diff_opt['scheduler']['num_test_timesteps'])
        self.DiffSampler.scheduler.diff_chns = self.params.data.num_classes

    def configure_optimizers(self):
        # REQUIRED
        optimizer = get_optimizer(self.params.optim, filter(lambda p: p.requires_grad, self.model.parameters()))
        # optimizer = Lion(filter(lambda p: p.requires_grad, self.model.parameters()), lr=self.initlr,betas=[0.9,0.99],weight_decay=0)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs, eta_min=self.initlr * 0.01)

        return [optimizer], [scheduler]


    def init_weight(self,ckpt_path=None):
        if ckpt_path:
            try:
                checkpoint = torch.load(ckpt_path,map_location='cpu',weights_only=False)
            except TypeError:  # PyTorch 1.13 compatibility
                checkpoint = torch.load(ckpt_path,map_location='cpu')
            if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
                checkpoint = checkpoint['state_dict']
            if isinstance(checkpoint, (tuple, list)):
                checkpoint = checkpoint[0]
            if not isinstance(checkpoint, dict):
                raise ValueError('Auxiliary checkpoint must contain a state dictionary: %s' % ckpt_path)
            if any(k.startswith('aux_model.') for k in checkpoint):
                checkpoint = {k[len('aux_model.'):]:v for k,v in checkpoint.items() if k.startswith('aux_model.')}
            target = self.aux_model.state_dict(); compatible={}; mismatched=[]
            for key,value in checkpoint.items():
                if key in target and tuple(value.shape)==tuple(target[key].shape): compatible[key]=value
                elif key in target: mismatched.append((key,tuple(value.shape),tuple(target[key].shape)))
            if not compatible: raise ValueError('No compatible auxiliary checkpoint weights found in %s' % ckpt_path)
            fraction=len(compatible)/max(1,len(target))
            if fraction<.95: raise ValueError('Auxiliary checkpoint only %.1f%% compatible (<95%%); mismatched=%s' % (100*fraction,mismatched))
            print('Aux checkpoint: loaded %d/%d tensors (%.1f%%); mismatches=%s; missing=%d' % (len(compatible),len(target),100*fraction,mismatched,len(set(target)-set(compatible))))
            target.update(compatible); self.aux_model.load_state_dict(target)

    def diffusion_focal_loss(self, prior, targets, noise, noise_gt, gamma=1, alpha=10):
        probs = F.softmax(prior, dim=1)
        probs = (probs * targets).sum(dim=1)
        weights = 1+alpha*(1 - probs) ** gamma
        weights = weights.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        loss = weights*(noise-noise_gt).square()
        return loss.mean()



    def guided_prob_map(self, y0_g, y0_l, bz, nc, np):
    
        distance_to_diag = torch.tensor([[abs(i-j)  for j in range(np)] for i in range(np)]).to(self.device)

        weight_g = 1 - distance_to_diag / (np-1)
        weight_l = distance_to_diag / (np-1)
        interpolated_value = weight_l.unsqueeze(0).unsqueeze(0) * y0_l.unsqueeze(-1).unsqueeze(-1) + weight_g.unsqueeze(0).unsqueeze(0) * y0_g.unsqueeze(-1).unsqueeze(-1)
        diag_indices = torch.arange(np)
        map = interpolated_value.clone()
        for i in range(bz):
            for j in range(nc):
                map[i,j,diag_indices,diag_indices] = y0_g[i,j]
                map[i,j, np-1, 0] = y0_l[i,j]
                map[i,j, 0, np-1] = y0_l[i,j]
        return map

    def training_step(self, batch, batch_idx):
        self.model.train()
        self.aux_model.eval()
        
        x_batch, y_batch = batch[:2]
        y_batch, _ = cast_label_to_one_hot_and_prototype(y_batch, self.params)
        y_batch = y_batch.to(self.device)
        #bicubic = bicubic.cuda()
        x_batch = x_batch.to(self.device)
        with torch.no_grad():
            y0_aux, y0_aux_global, y0_aux_local, patches, attns, attn_map = self.aux_model(x_batch)
            # y0_aux_global,y0_aux_local = y0_aux_global.softmax(1),y0_aux_local.softmax(1)
        # loss_aux = self.aux_cost_function(y0_aux,y_batch)
        # loss_aux.backward()
        
        
        bz, nc, H, W = attn_map.size()
        bz, np = attns.size()
        
        y_map = y_batch.unsqueeze(1).expand(-1,np*np,-1).reshape(bz*np*np,nc)
        noise = torch.randn_like(y_map).to(self.device)
        timesteps = torch.randint(0, self.DiffSampler.scheduler.config.num_train_timesteps, (bz*np*np,), device=self.device).long()

        noisy_y = self.DiffSampler.scheduler.add_noise(y_map, timesteps=timesteps, noise=noise)
        noisy_y = noisy_y.view(bz,np*np,-1).permute(0,2,1).reshape(bz,nc,np,np)
        
        y0_cond = self.guided_prob_map(y0_aux_global,y0_aux_local,bz,nc,np)
        y_fusion = torch.cat([y0_cond, noisy_y],dim=1)

        attns = attns.unsqueeze(-1)
        attns = (attns*attns.transpose(1,2)).unsqueeze(1)
        noise_pred = self.model(x_batch, y_fusion, timesteps, patches, attns)

        noise = noise.view(bz,np*np,-1).permute(0,2,1).reshape(bz,nc,np,np)
        loss = self.diffusion_focal_loss(y0_aux,y_batch,noise_pred,noise)

        self.log("train_loss",loss,prog_bar=True)
        return {"loss":loss}

    # def validation_step_end(self,step_output):
    #     model_state_dict = self.model.state_dict()
    #     torch.save(model_state_dict, os.path.join(self.save_path,'ckp.pth'))
    #     print('checkpoint save!')
    #     ema_model_state_dict = self.ema_model.state_dict()
    #     for key in model_state_dict:
    #         ema_model_state_dict[key] = 0.999*ema_model_state_dict[key] + 0.001*model_state_dict[key]
    #     self.ema_model.load_state_dict(ema_model_state_dict)
    def on_validation_epoch_end(self):
        gt = torch.cat(self.gts)
        pred = torch.cat(self.preds)
        ACC, BACC, Prec, Rec, F1, AUC_ovo, kappa = compute_isic_metrics(gt, pred)

        self.log('accuracy',ACC)
        self.log('f1',F1)
        self.log('Precision',Prec)        
        self.log('Recall',Rec)
        self.log('AUC',AUC_ovo)
        self.log('kappa',kappa)   
        
        self.gts = []
        self.preds = []
        print("Val: Accuracy {0}, F1 score {1}, Precision {2}, Recall {3}, AUROC {4}, Cohen Kappa {5}".format(ACC,F1,Prec,Rec,AUC_ovo,kappa))


    def validation_step(self,batch,batch_idx):
        self.model.eval()
        self.aux_model.eval()

        
        x_batch, y_batch = batch[:2]
        y_batch, _ = cast_label_to_one_hot_and_prototype(y_batch, self.params)
        y_batch = y_batch.to(self.device)
        x_batch = x_batch.to(self.device)
        y0_aux, y0_aux_global, y0_aux_local, patches, attns, attn_map = self.aux_model(x_batch)

        bz, nc, H, W = attn_map.size()
        bz, np = attns.size()

        
        y0_cond = self.guided_prob_map(y0_aux_global,y0_aux_local,bz,nc,np)
        # yT = torch.rand_like(y0_cond)
        yT = self.guided_prob_map(torch.rand_like(y0_aux_global),torch.rand_like(y0_aux_local),bz,nc,np)
        attns = attns.unsqueeze(-1)
        attns = (attns*attns.transpose(1,2)).unsqueeze(1)
        y_pred = self.DiffSampler.sample_high_res(x_batch,yT,conditions=[y0_cond, patches, attns])
        y_pred = y_pred.reshape(bz, nc, np*np)
        y_pred = y_pred.mean(2)
        self.preds.append(y_pred)
        self.gts.append(y_batch)

        
        # self.log('accuracy',ACC)
        # self.log('f1',F1)
        
        # return {"gt":y_batch,"pred":y_pred}
    
    def train_dataloader(self):
        data_object, train_dataset, test_dataset = get_dataset(self.params)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.params.training.batch_size,
            shuffle=True,
            num_workers=self.params.data.num_workers,
            #sampler=sampler
        )
        return train_loader
    
    def val_dataloader(self):
        data_object, train_dataset, test_dataset = get_dataset(self.params)

        test_loader = DataLoader(
            test_dataset,
            batch_size=self.params.testing.batch_size,
            shuffle=False,
            num_workers=self.params.data.num_workers,
        )
        return test_loader  

    def test_step(self, batch, batch_idx):
        # Reuse the exact inference path; test labels are only consumed for final reporting.
        self.validation_step(batch, batch_idx)

    def on_test_epoch_end(self):
        if not self.gts:
            return
        gt=torch.cat(self.gts); pred=torch.cat(self.preds)
        ACC,BACC,Prec,Rec,F1,AUC_ovo,kappa=compute_isic_metrics(gt,pred)
        for name,value in [('test_accuracy',ACC),('test_balanced_accuracy',BACC),('test_precision_macro',Prec),('test_recall_macro',Rec),('test_macro_f1',F1),('test_macro_auroc_ovo',AUC_ovo),('test_quadratic_weighted_kappa',kappa)]:
            self.log(name,value)
        print('Held-out test: Accuracy %.4f Macro-F1 %.4f AUROC-OVO %.4f QWK %.4f' % (ACC,F1,AUC_ovo,kappa))
        self.gts=[];self.preds=[]

    def test_dataloader(self):
        from aptos_research.data import load_split_manifest,AptosImageDataset
        manifest=load_split_manifest(self.params.data.splits_manifest)
        dataset=AptosImageDataset(manifest['splits']['test'],'test',size=224,training=False)
        return DataLoader(dataset,batch_size=self.params.testing.batch_size,shuffle=False,num_workers=self.params.data.num_workers)


def main():
    import argparse
    parser=argparse.ArgumentParser(description='Train the original DiffMIC-v2 diffusion model.')
    parser.add_argument('--config',default='configs/placental.yml')
    parser.add_argument('--resume',default=None)
    parser.add_argument('--output-dir',default='logs')
    parser.add_argument('--run-name',default=None)
    parser.add_argument('--evaluate-only',action='store_true',help='evaluate --resume on the held-out APTOS test split')
    args=parser.parse_args()
    with open(args.config, 'r') as f:
        params = yaml.safe_load(f)
    seed = int(params.get('data',{}).get('seed',10))
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    if params.get('data',{}).get('dataset')=='APTOS':
        params['data']['splits_manifest']=os.environ.get('APTOS_SPLITS',params['data'].get('splits_manifest','outputs/aptos_splits.json'))
        params['data']['aux_checkpoint']=os.environ.get('DIFFMIC_AUX_CHECKPOINT',params['data'].get('aux_checkpoint'))
        if isinstance(params['data']['aux_checkpoint'],str) and params['data']['aux_checkpoint'].startswith('${'):
            params['data']['aux_checkpoint']=None
        if not params['data']['aux_checkpoint']:
            raise RuntimeError('APTOS training requires DIFFMIC_AUX_CHECKPOINT pointing to a compatible pretrained DCG checkpoint')
    config = EasyDict(params)


    # hparams = Namespace(**args)

    model = CoolSystem(config)

    logger = TensorBoardLogger(name=args.run_name or config.data.dataset.lower(),save_dir=args.output_dir)
    checkpoint_callback = ModelCheckpoint(
        monitor='f1',
        filename=config.data.dataset.lower()+'-epoch{epoch:02d}-accuracy-{accuracy:.4f}-f1-{f1:.4f}',
        auto_insert_metric_name=False,   
        every_n_epochs=1,
        save_top_k=1,
        mode = "max",
        save_last=True
    )
    lr_monitor_callback = LearningRateMonitor(logging_interval='step')
    trainer = pl.Trainer(
        check_val_every_n_epoch=5,
        max_epochs=config.training.n_epochs,
        accelerator='auto',
        devices=1 if torch.cuda.is_available() else 'auto',
        precision=32,
        logger=logger,
        strategy="auto",
        enable_progress_bar=True,
        log_every_n_steps=5,
        callbacks = [checkpoint_callback,lr_monitor_callback]
    ) 

    #train
    if args.evaluate_only:
        if config.data.dataset!='APTOS': raise RuntimeError('--evaluate-only is currently defined only for the manifest-based APTOS test split')
        if not args.resume: raise RuntimeError('--evaluate-only requires --resume with a checkpoint')
        trainer.test(model,ckpt_path=args.resume)
    else:
        trainer.fit(model,ckpt_path=args.resume)
        if config.data.dataset=='APTOS': trainer.test(model,ckpt_path='best')
    
    #validate
    # val_path=r'DiffMIC/logs/placental/version_4/checkpoints/placental-epoch924-accuracy-0.9350-f1-0.9327.ckpt'
    # trainer.validate(model,ckpt_path=val_path)
    
if __name__ == '__main__':
	#your code
    main()


### Embedded source: `model.py`

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.resnet import resnet18, resnet50, resnet101
from torchvision.models.densenet import densenet169

from timm.models import create_model
from pretraining.dcg import DCG
import numpy as np
from einops import rearrange, reduce

class ConditionalLinear(nn.Module):
    def __init__(self, num_in, num_out, n_steps):
        super(ConditionalLinear, self).__init__()
        self.num_out = num_out
        self.lin = nn.Linear(num_in, num_out)
        self.embed = nn.Embedding(n_steps, num_out)
        self.embed.weight.data.uniform_()

    def forward(self, x, t):
        out = self.lin(x)
        gamma = self.embed(t)
        out = gamma.view(-1, self.num_out) * out
        return out

class ConditionalConv2d(nn.Module):
    def __init__(self, num_in, num_out, n_steps):
        super(ConditionalConv2d, self).__init__()
        self.num_out = num_out
        self.lin = nn.Conv2d(num_in, num_out, kernel_size=1, stride=1)
        self.embed = nn.Embedding(n_steps, num_out)
        self.embed.weight.data.uniform_()

    def forward(self, x, t):
        out = self.lin(x)
        bz, dim, h, w = out.size()
        gamma = self.embed(t)
        gamma = gamma.reshape(bz, h*w, dim).permute(0,2,1).reshape(bz,dim,h,w)
        out = gamma*out
        return out


from EfficientSAM.efficient_sam.build_efficient_sam import build_efficient_sam_vits, build_efficient_sam_vitt

class DenoiseUNet(nn.Module):
    def __init__(self, y_dim, feature_dim, n_steps, guidance=True):
        super(DenoiseUNet, self).__init__()

        if guidance:
            self.lin1 = ConditionalConv2d(y_dim * 2, feature_dim, n_steps)
        else:
            self.lin1 = ConditionalConv2d(y_dim, feature_dim, n_steps)

        self.unetnorm1 = nn.BatchNorm2d(feature_dim)
        self.lin2 = ConditionalConv2d(feature_dim, feature_dim, n_steps)
        self.unetnorm2 = nn.BatchNorm2d(feature_dim)
        self.lin3 = ConditionalConv2d(feature_dim, feature_dim, n_steps)
        self.unetnorm3 = nn.BatchNorm2d(feature_dim)
        self.lin4 = nn.Conv2d(feature_dim, y_dim, kernel_size=1, stride=1)
        self.cond_weight = nn.Parameter(torch.randn((1, feature_dim, 7)), requires_grad=True)

        
    def forward(self, x, y, t):

        y = self.lin1(y, t)
        y = self.unetnorm1(y)
        y = F.softplus(y)
        skip_y1 = y

        w = torch.softmax(self.cond_weight,dim=2)
        x_weight = torch.sum(x*w,dim=-1)
        # print(x_weight.shape)
        y = x_weight.unsqueeze(-1).unsqueeze(-1) * y
        # y = x*y
        y = self.lin2(y, t)
        y = self.unetnorm2(y)
        y = F.softplus(y)
        
        # y = x * y
        y = self.lin3(y, t)
        y = self.unetnorm3(y)
        y = F.softplus(y)
        y = y + skip_y1

        y = self.lin4(y)
            
        return y


class ConditionalModel(nn.Module):
    def __init__(self, config, guidance=True):
        super(ConditionalModel, self).__init__()
        n_steps = config.diffusion.timesteps + 1
        data_dim = config.model.data_dim
        y_dim = config.data.num_classes
        arch = config.model.arch
        feature_dim = config.model.feature_dim
        hidden_dim = config.model.hidden_dim
        self.guidance = guidance
        
        # encoder for x
        self.encoder_x = SamEncoder(arch=arch, feature_dim=feature_dim, config=config)
        self.norm = nn.BatchNorm1d(feature_dim)

        self.encoder_x_l = ResNetEncoder(arch=arch, feature_dim=feature_dim, config=config, local=True)
        self.norm_l = nn.BatchNorm1d(feature_dim)

        # gated attention
        # self.gate = nn.Sequential(
        #     nn.Linear(feature_dim * 2, feature_dim),
        #     nn.Sigmoid()
        # )
        
        if self.guidance:
            self.lin1 = ConditionalConv2d(y_dim * 2, feature_dim, n_steps)
        else:
            self.lin1 = ConditionalConv2d(y_dim, feature_dim, n_steps)
        self.unetnorm1 = nn.BatchNorm2d(feature_dim)
        self.lin2 = ConditionalConv2d(feature_dim, feature_dim, n_steps)
        self.unetnorm2 = nn.BatchNorm2d(feature_dim)
        self.lin3 = ConditionalConv2d(feature_dim, feature_dim, n_steps)
        self.unetnorm3 = nn.BatchNorm2d(feature_dim)
        self.lin4 = nn.Conv2d(feature_dim, y_dim, kernel_size=1, stride=1)
        self.cond_weight = nn.Parameter(torch.randn((1, feature_dim, 7)), requires_grad=True)

        
    def forward(self, x, y, t, x_l, attn):
        bz, np, I, J = x_l.shape

        x_l = x_l.view(bz * np, I, J).unsqueeze(1).expand(-1, 3, -1 , -1)
        x_l = self.encoder_x_l(x_l)
        x_l = self.norm_l(x_l)
        
        x = self.encoder_x(x)
        x = self.norm(x)


        y = self.lin1(y, t)
        y = self.unetnorm1(y)
        y = F.softplus(y)
        
        x_l = x_l.reshape(bz , np, x_l.shape[1]).permute(0,2,1)
        x = torch.cat([x.unsqueeze(-1),x_l],dim=-1)
        w = torch.softmax(self.cond_weight,dim=2)
        # print(w.shape)
        # print(x.shape)
        x_weight = torch.sum(x*w,dim=-1)
        # print(x_weight.shape)
        y = x_weight.unsqueeze(-1).unsqueeze(-1) * y
        
        y = self.lin2(y, t)
        y = self.unetnorm2(y)
        y = F.softplus(y)
        y = self.lin3(y, t)
        y = self.unetnorm3(y)
        y = F.softplus(y)
        y = self.lin4(y)
            
        return y




# ResNet 18 or 50 as image encoder
class ResNetEncoder(nn.Module):
    def __init__(self, arch='resnet18', feature_dim=128, config=None, local=False):
        super(ResNetEncoder, self).__init__()

        self.f = []
        #print(arch)
        if arch == 'resnet50':
            backbone = resnet50()
            self.featdim = backbone.fc.weight.shape[1]
        elif arch == 'resnet18':
            backbone = resnet18()
            self.featdim = backbone.fc.weight.shape[1]
        elif arch == 'densenet121':
            backbone = densenet121(pretrained=True)
            self.featdim = backbone.classifier.weight.shape[1]
        elif arch == 'vit':
            backbone = create_model('pvt_v2_b2',
                pretrained=True,
                num_classes=4,
                drop_rate=0,
                drop_path_rate=0.1,
                drop_block_rate=None,
            )
            backbone.head = nn.Sequential()
            self.featdim = 512

            

        for name, module in backbone.named_children():
            if name != 'fc':
                self.f.append(module)
        # encoder
        self.f = nn.Sequential(*self.f)
        
        self.g = nn.Linear(self.featdim, feature_dim)


    def forward_feature(self, x):
        feature = self.f(x)
        feature = torch.flatten(feature, start_dim=1)
        feature = self.g(feature)

        return feature

    def forward(self, x):
        feature = self.forward_feature(x)
        return feature

class SamEncoder(nn.Module):
    def __init__(self, arch='resnet18', feature_dim=128, config=None, image_size=224):
        super(SamEncoder, self).__init__()

        self.f = []
        self.f = build_efficient_sam_vits(image_size=image_size)
        self.featdim = 384
        
            
        self.g = nn.Conv2d(self.featdim, feature_dim, kernel_size=1, stride=1)

    def forward_feature(self, x):
        feature = self.f(x)
        feature = self.g(feature)
        feature = F.adaptive_avg_pool2d(feature,(1,1))
        return torch.flatten(feature,start_dim=1)

    def forward(self, x):
        feature = self.forward_feature(x)
        return feature



### Embedded source: `optimizer.py`

In [ ]:
%%writefile optimizer.py
# Copyright 2023 Google Research. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================
"""PyTorch implementation of the Lion optimizer."""

import torch
from torch.optim.optimizer import Optimizer


class Lion(Optimizer):
  r"""Implements Lion algorithm."""

  def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
    """Initialize the hyperparameters.

    Args:
      params (iterable): iterable of parameters to optimize or dicts defining
        parameter groups
      lr (float, optional): learning rate (default: 1e-4)
      betas (Tuple[float, float], optional): coefficients used for computing
        running averages of gradient and its square (default: (0.9, 0.99))
      weight_decay (float, optional): weight decay coefficient (default: 0)
    """

    if not 0.0 <= lr:
      raise ValueError('Invalid learning rate: {}'.format(lr))
    if not 0.0 <= betas[0] < 1.0:
      raise ValueError('Invalid beta parameter at index 0: {}'.format(betas[0]))
    if not 0.0 <= betas[1] < 1.0:
      raise ValueError('Invalid beta parameter at index 1: {}'.format(betas[1]))
    defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
    super().__init__(params, defaults)

  @torch.no_grad()
  def step(self, closure=None):
    """Performs a single optimization step.

    Args:
      closure (callable, optional): A closure that reevaluates the model
        and returns the loss.

    Returns:
      the loss.
    """
    loss = None
    if closure is not None:
      with torch.enable_grad():
        loss = closure()

    for group in self.param_groups:
      for p in group['params']:
        if p.grad is None:
          continue

        # Perform stepweight decay
        p.data.mul_(1 - group['lr'] * group['weight_decay'])

        grad = p.grad
        state = self.state[p]
        # State initialization
        if len(state) == 0:
          # Exponential moving average of gradient values
          state['exp_avg'] = torch.zeros_like(p)

        exp_avg = state['exp_avg']
        beta1, beta2 = group['betas']

        # Weight update
        update = exp_avg * beta1 + grad * (1 - beta1)
        p.add_(torch.sign(update), alpha=-group['lr'])
        # Decay the momentum running average coefficient
        exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)

    return loss


class SAM(Optimizer):
    """
    SAM is an optimizer proposed to seek parameters that lie in neighborhoods having uniformly low loss.

    Sharpness-Aware Minimization for Efficiently Improving Generalization
    https://arxiv.org/abs/2010.01412
    """

    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)

        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None:
                    continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (
                    (torch.pow(p, 2) if group["adaptive"] else 1.0)
                    * p.grad
                    * scale.to(p)
                )
                p.add_(e_w)  # climb to the local maximum "w + e(w)"

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.data = self.state[p]["old_p"]  # get back to "w" from "w + e(w)"

        self.base_optimizer.step()  # do the actual "sharpness-aware" update

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert (
            closure is not None
        ), "Sharpness Aware Minimization requires closure, but it was not provided"
        closure = torch.enable_grad()(
            closure
        )  # the closure should do a full forward-backward pass

        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][
            0
        ].device  # put everything on the same device, in case of model parallelism
        norm = torch.norm(
            torch.stack(
                [
                    ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad)
                    .norm(p=2)
                    .to(shared_device)
                    for group in self.param_groups
                    for p in group["params"]
                    if p.grad is not None
                ]
            ),
            p=2,
        )
        return norm

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups


### Embedded source: `option/config.py`

In [ ]:
%%writefile option/config.py
# coding=utf-8
# Copyright 2022 The Google Research Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Configuration."""

import ml_collections


def config_dict(**kwargs):
    return ml_collections.ConfigDict(initial_dictionary=kwargs)


def get_config():
    return config_dict(
        seed=0,
        dataset=config_dict(
            name='cifar10',
            path="datasets/",
            resolution=32,
            args=config_dict(
                class_conditional=False,
                randflip=True,
            ),
        ),
        model=config_dict(
            # architecture, see main.py and model.py
            name='unet0',
            args=config_dict(
                channel=128,
                in_channel=3,
                out_channel=3,
                channel_multiplier=[1, 2, 2, 2],
                n_res_blocks=2,
                attn_resolutions=[16],
                num_heads=1,
                fold=1,
                dropout=0.1,
                model_output='logistic_pars',  # logits  or logistic_pars
                num_pixel_vals=256
            ),
            # diffusion betas, see diffusion_categorical.get_diffusion_betas
            diffusion_betas=config_dict(
                type='linear',
                # start, stop only relevant for linear, power, jsdtrunc schedules.
                start=1e-4,  # 1e-4 gauss, 0.02 uniform
                stop=0.02,  # 0.02, gauss, 1. uniform
                num_timesteps=1000,
            ),
            # Settings used in diffusion_categorical.py
            model_prediction='x_start',  # 'x_start','xprev'
            # 'gaussian','uniform','absorbing'
            transition_mat_type='gaussian',
            transition_bands=None,
            loss_type='hybrid',  # kl,cross_entropy_x_start, hybrid
            hybrid_coeff=0.001,  # only used for hybrid loss type.
        ),
        train=config_dict(
            # optimizer
            batch_size=128,
            optimizer='adam',
            learning_rate=2e-4,
            learning_rate_warmup_steps=0,
            weight_decay=0.0,
            ema_decay=0.9999,
            grad_clip=1.0,
            substeps=10,
            num_train_steps=1500000,  # multiple of substeps
            # logging
            log_loss_every_steps=1000,
            checkpoint_every_secs=900,  # 15 minutes
            retain_checkpoint_every_steps=100000,
            eval_every_steps=50000,
            log_img_dir="exp/cifar10/images",
            eval_every_epoch=50
        ))

### Embedded source: `pipeline.py`

In [ ]:
%%writefile pipeline.py
import torch
from tqdm import tqdm
from functools import partial

from diffusers import DDIMScheduler,DPMSolverMultistepScheduler
import math



class SR3scheduler(DDIMScheduler):
    def __init__(self, num_train_timesteps: int = 1000, beta_start: float = 0.0001,beta_end: float = 0.02, beta_schedule: str = 'linear',diff_chns=3):
        super().__init__(num_train_timesteps, beta_start ,beta_end,beta_schedule)
        # Initialize other attributes specific to SR3scheduler class
        # ...
        self.diff_chns = diff_chns
        # self.config.prediction_type = "sample"

    def add_noise(
        self,
        original_samples: torch.FloatTensor,
        noise: torch.FloatTensor,
        # temporal_noise: torch.FloatTensor,
        timesteps: torch.IntTensor,
    ) -> torch.FloatTensor:
        # Make sure alphas_cumprod and timestep have same device and dtype as original_samples
        alphas_cumprod = self.alphas_cumprod.to(device=original_samples.device, dtype=original_samples.dtype)
        timesteps = timesteps.to(original_samples.device)

        sqrt_alpha_prod = alphas_cumprod[timesteps] ** 0.5
        sqrt_alpha_prod = sqrt_alpha_prod.flatten()
        while len(sqrt_alpha_prod.shape) < len(original_samples.shape):
            sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)

        sqrt_one_minus_alpha_prod = (1 - alphas_cumprod[timesteps]) ** 0.5
        sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.flatten()
        while len(sqrt_one_minus_alpha_prod.shape) < len(original_samples.shape):
            sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.unsqueeze(-1)

        # Only modify the last three channels of the tensor (assuming channels are in the second dimension)
        num_channels = original_samples.shape[1]
        if num_channels > self.diff_chns:
            # print(num_channels)
            # print(self.diff_chns)
            original_samples_select = original_samples[:, -self.diff_chns:].contiguous()
            noise_select = noise[:, -self.diff_chns:].contiguous()
            # temporal_noise_select = temporal_noise[:, -self.diff_chns:].contiguous()

            noisy_samples_select = sqrt_alpha_prod * original_samples_select + sqrt_one_minus_alpha_prod * noise_select

            noisy_samples = original_samples.clone()
            noisy_samples[:, -self.diff_chns:] = noisy_samples_select
        else:
            noisy_samples = sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise

        return noisy_samples

def create_SR3scheduler(opt,phase):
    
    steps= opt['num_train_timesteps'] if phase=="train" else opt['num_test_timesteps']
    scheduler=SR3scheduler(
        num_train_timesteps = steps,
        beta_start = opt['beta_start'],
        beta_end = opt['beta_end'],
        beta_schedule = opt['beta_schedule']
    )
    return scheduler
    
class SR3Sampler():
    
    def __init__(self,model: torch.nn.Module, scheduler:SR3scheduler,eta: float =.0):
        self.model = model
        # self.refine = refine
        self.scheduler = scheduler
        self.eta = eta
        


    def sample_high_res(self,x_batch: torch.Tensor, noisy_y, conditions=None,
                        trajectory_callback=None, selected_timesteps=None):
        "Using Diffusers built-in samplers"
        device = next(self.model.parameters()).device
        eta = torch.Tensor([self.eta]).to(device)
        x_batch=x_batch.to(device)
        y0, patches, attn = conditions[0].to(device),conditions[1].to(device),conditions[2].to(device)
        bz,nc,h,w = y0.shape
        # The evolving tensor is a class-score diffusion grid (B, C, H, W), not
        # an image latent. Optional capture is observational and happens after
        # each scheduler transition; without a callback this path is unchanged.
        selected = None if selected_timesteps is None else {int(v) for v in selected_timesteps}
        for state_index, t in enumerate(self.scheduler.timesteps):
            self.model.eval()
            timesteps = t * torch.ones(bz*h*w, dtype=t.dtype, device=x_batch.device)
            with torch.no_grad():
                noise = self.model(x_batch, torch.cat([y0,noisy_y],dim=1), timesteps, patches, attn)
            noisy_y = self.scheduler.step(model_output = noise,timestep = t,  sample = noisy_y).prev_sample #eta = eta
            if trajectory_callback is not None and (selected is None or int(t) in selected):
                trajectory_callback(int(t), state_index, noisy_y)
            del noise
            torch.cuda.empty_cache()
            

        return noisy_y



def create_SR3Sampler(model,opt):
    
    scheduler = create_SR3scheduler(opt,"test")
    scheduler.set_timesteps(opt['num_test_timesteps'])
    sampler = SR3Sampler(
        model = model,
        scheduler = scheduler,
        eta = opt['eta']
    )
    return sampler

def KL(logit1,logit2,reverse=False):
    if reverse:
        logit1, logit2 = logit2, logit1
    p1 = logit1.softmax(1)
    logp1 = logit1.log_softmax(1)
    logp2 = logit2.log_softmax(1) 
    return (p1*(logp1-logp2)).sum(1)


### Embedded source: `pretraining/dcg.py`

In [ ]:
%%writefile pretraining/dcg.py
import torch
import torch.nn as nn
import numpy as np
import pretraining.tools as tools
import pretraining.modules as m


class DCG(nn.Module):
    def __init__(self, parameters):
        super(DCG, self).__init__()

        # save parameters
        self.experiment_parameters = {
        "device_type": 'gpu',
        "gpu_number": 6,
        # model related hyper-parameters
        "cam_size": (7, 7),
        "K": parameters.model.num_k,
        "crop_shape": (32, 32),
        "post_processing_dim":512,
        "num_classes":parameters.data.num_classes,
        "use_v1_global":True,
        "percent_t": 1.0,
        }
        self.cam_size = self.experiment_parameters["cam_size"]

        # construct networks
        # global network
        self.global_network = m.GlobalNetwork(self.experiment_parameters, self)
        self.global_network.add_layers()

        # aggregation function
        self.aggregation_function = m.TopTPercentAggregationFunction(self.experiment_parameters, self)

        # detection module
        self.retrieve_roi_crops = m.RetrieveROIModule(self.experiment_parameters, self)

        # detection network
        self.local_network = m.LocalNetwork(self.experiment_parameters, self)
        self.local_network.add_layers()

        # MIL module
        self.attention_module = m.AttentionModule(self.experiment_parameters, self)
        self.attention_module.add_layers()
        # fusion branch
        # self.fusion_dnn = nn.Linear(1664*2, self.experiment_parameters["num_classes"], bias=False)
        self.fusion_dnn = nn.Linear(self.experiment_parameters["post_processing_dim"]+512, self.experiment_parameters["num_classes"], bias=False)



    def _convert_crop_position(self, crops_x_small, cam_size, x_original):
        """
        Function that converts the crop locations from cam_size to x_original
        :param crops_x_small: N, k*c, 2 numpy matrix
        :param cam_size: (h,w)
        :param x_original: N, C, H, W pytorch variable
        :return: N, k*c, 2 numpy matrix
        """
        # retrieve the dimension of both the original image and the small version
        h, w = cam_size
        _, _, H, W = x_original.size()

        # interpolate the 2d index in h_small to index in x_original
        top_k_prop_x = crops_x_small[:, :, 0] / h
        top_k_prop_y = crops_x_small[:, :, 1] / w
        # sanity check
        assert np.max(top_k_prop_x) <= 1.0, "top_k_prop_x >= 1.0"
        assert np.min(top_k_prop_x) >= 0.0, "top_k_prop_x <= 0.0"
        assert np.max(top_k_prop_y) <= 1.0, "top_k_prop_y >= 1.0"
        assert np.min(top_k_prop_y) >= 0.0, "top_k_prop_y <= 0.0"
        # interpolate the crop position from cam_size to x_original
        top_k_interpolate_x = np.expand_dims(np.around(top_k_prop_x * H), -1)
        top_k_interpolate_y = np.expand_dims(np.around(top_k_prop_y * W), -1)
        top_k_interpolate_2d = np.concatenate([top_k_interpolate_x, top_k_interpolate_y], axis=-1)
        return top_k_interpolate_2d

    def _retrieve_crop(self, x_original_pytorch, crop_positions, crop_method):
        """
        Function that takes in the original image and cropping position and returns the crops
        :param x_original_pytorch: PyTorch Tensor array (N,C,H,W)
        :param crop_positions:
        :return:
        """
        batch_size, num_crops, _ = crop_positions.shape
        crop_h, crop_w = self.experiment_parameters["crop_shape"]

        output = torch.ones((batch_size, num_crops, crop_h, crop_w))
        if self.experiment_parameters["device_type"] == "gpu":
            # device = torch.device("cuda:{}".format(self.experiment_parameters["gpu_number"]))
            output = output.to(x_original_pytorch.device)
        for i in range(batch_size):
            for j in range(num_crops):
                tools.crop_pytorch(x_original_pytorch[i, 0, :, :],
                                                    self.experiment_parameters["crop_shape"],
                                                    crop_positions[i,j,:],
                                                    output[i,j,:,:],
                                                    method=crop_method)
        return output



    def forward(self, x_original):
        """
        :param x_original: N,H,W,C numpy matrix
        """
        # global network: x_small -> class activation map
        h_g, self.saliency_map = self.global_network.forward(x_original)
        # calculate y_global
        # note that y_global is not directly used in inference
        self.y_global = self.aggregation_function.forward(self.saliency_map)

        # region proposal network
        small_x_locations = self.retrieve_roi_crops.forward(x_original, self.cam_size, self.saliency_map)

        # convert crop locations that is on self.cam_size to x_original
        self.patch_locations = self._convert_crop_position(small_x_locations, self.cam_size, x_original)

        # patch retriever
        crops_variable = self._retrieve_crop(x_original, self.patch_locations, self.retrieve_roi_crops.crop_method)
        self.patches = crops_variable.data.cpu().numpy()
        patches = crops_variable.clone()
        # print(crops_variable.shape)
        
        # detection network
        batch_size, num_crops, I, J = crops_variable.size()
        
        crops_variable = crops_variable.view(batch_size * num_crops, I, J).unsqueeze(1)
        h_crops = self.local_network.forward(crops_variable).view(batch_size, num_crops, -1)
        # print(h_crops.shape)
        # MIL module
        # y_local is not directly used during inference
        z, self.patch_attns, self.y_local = self.attention_module.forward(h_crops)
        # self.y_global = self.y_global.softmax(1)
        self.y_fusion = 0.5* (self.y_global+self.y_local)

        # g1, _ = torch.max(h_g, dim=2)
        # global_vec, _ = torch.max(g1, dim=2)
        # concat_vec = torch.cat([global_vec, z], dim=1)
        # # print(concat_vec.shape)
        # self.y_fusion = self.fusion_dnn(concat_vec)

        # self.y_fusion = self.saliency_map
        return self.y_fusion, self.y_global, self.y_local, patches, self.patch_attns, self.saliency_map
    

### Embedded source: `pretraining/densenet.py`

In [ ]:
%%writefile pretraining/densenet.py
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from collections import OrderedDict
from torchvision._internally_replaced_utils import load_state_dict_from_url
from torch import Tensor
from typing import Any, List, Tuple


__all__ = ['DenseNet', 'densenet121', 'densenet169', 'densenet201', 'densenet161']

model_urls = {
    'densenet121': 'https://download.pytorch.org/models/densenet121-a639ec97.pth',
    'densenet169': 'https://download.pytorch.org/models/densenet169-b2777c0a.pth',
    'densenet201': 'https://download.pytorch.org/models/densenet201-c1103571.pth',
    'densenet161': 'https://download.pytorch.org/models/densenet161-8d451a50.pth',
}


class _DenseLayer(nn.Module):
    def __init__(
        self,
        num_input_features: int,
        growth_rate: int,
        bn_size: int,
        drop_rate: float,
        memory_efficient: bool = False
    ) -> None:
        super(_DenseLayer, self).__init__()
        self.norm1: nn.BatchNorm2d
        self.add_module('norm1', nn.BatchNorm2d(num_input_features))
        self.relu1: nn.ReLU
        self.add_module('relu1', nn.ReLU(inplace=True))
        self.conv1: nn.Conv2d
        self.add_module('conv1', nn.Conv2d(num_input_features, bn_size *
                                           growth_rate, kernel_size=1, stride=1,
                                           bias=False))
        self.norm2: nn.BatchNorm2d
        self.add_module('norm2', nn.BatchNorm2d(bn_size * growth_rate))
        self.relu2: nn.ReLU
        self.add_module('relu2', nn.ReLU(inplace=True))
        self.conv2: nn.Conv2d
        self.add_module('conv2', nn.Conv2d(bn_size * growth_rate, growth_rate,
                                           kernel_size=3, stride=1, padding=1,
                                           bias=False))
        self.drop_rate = float(drop_rate)
        self.memory_efficient = memory_efficient

    def bn_function(self, inputs: List[Tensor]) -> Tensor:
        concated_features = torch.cat(inputs, 1)
        bottleneck_output = self.conv1(self.relu1(self.norm1(concated_features)))  # noqa: T484
        return bottleneck_output

    # todo: rewrite when torchscript supports any
    def any_requires_grad(self, input: List[Tensor]) -> bool:
        for tensor in input:
            if tensor.requires_grad:
                return True
        return False

    @torch.jit.unused  # noqa: T484
    def call_checkpoint_bottleneck(self, input: List[Tensor]) -> Tensor:
        def closure(*inputs):
            return self.bn_function(inputs)

        return cp.checkpoint(closure, *input)

    @torch.jit._overload_method  # noqa: F811
    def forward(self, input: List[Tensor]) -> Tensor:
        pass

    @torch.jit._overload_method  # noqa: F811
    def forward(self, input: Tensor) -> Tensor:
        pass

    # torchscript does not yet support *args, so we overload method
    # allowing it to take either a List[Tensor] or single Tensor
    def forward(self, input: Tensor) -> Tensor:  # noqa: F811
        if isinstance(input, Tensor):
            prev_features = [input]
        else:
            prev_features = input

        if self.memory_efficient and self.any_requires_grad(prev_features):
            if torch.jit.is_scripting():
                raise Exception("Memory Efficient not supported in JIT")

            bottleneck_output = self.call_checkpoint_bottleneck(prev_features)
        else:
            bottleneck_output = self.bn_function(prev_features)

        new_features = self.conv2(self.relu2(self.norm2(bottleneck_output)))
        if self.drop_rate > 0:
            new_features = F.dropout(new_features, p=self.drop_rate,
                                     training=self.training)
        return new_features


class _DenseBlock(nn.ModuleDict):
    _version = 2

    def __init__(
        self,
        num_layers: int,
        num_input_features: int,
        bn_size: int,
        growth_rate: int,
        drop_rate: float,
        memory_efficient: bool = False
    ) -> None:
        super(_DenseBlock, self).__init__()
        for i in range(num_layers):
            layer = _DenseLayer(
                num_input_features + i * growth_rate,
                growth_rate=growth_rate,
                bn_size=bn_size,
                drop_rate=drop_rate,
                memory_efficient=memory_efficient,
            )
            self.add_module('denselayer%d' % (i + 1), layer)

    def forward(self, init_features: Tensor) -> Tensor:
        features = [init_features]
        for name, layer in self.items():
            new_features = layer(features)
            features.append(new_features)
        return torch.cat(features, 1)


class _Transition(nn.Sequential):
    def __init__(self, num_input_features: int, num_output_features: int) -> None:
        super(_Transition, self).__init__()
        self.add_module('norm', nn.BatchNorm2d(num_input_features))
        self.add_module('relu', nn.ReLU(inplace=True))
        self.add_module('conv', nn.Conv2d(num_input_features, num_output_features,
                                          kernel_size=1, stride=1, bias=False))
        self.add_module('pool', nn.AvgPool2d(kernel_size=2, stride=2))


class DenseNet(nn.Module):
    r"""Densenet-BC model class, based on
    `"Densely Connected Convolutional Networks" <https://arxiv.org/pdf/1608.06993.pdf>`_.

    Args:
        growth_rate (int) - how many filters to add each layer (`k` in paper)
        block_config (list of 4 ints) - how many layers in each pooling block
        num_init_features (int) - the number of filters to learn in the first convolution layer
        bn_size (int) - multiplicative factor for number of bottle neck layers
          (i.e. bn_size * k features in the bottleneck layer)
        drop_rate (float) - dropout rate after each dense layer
        num_classes (int) - number of classification classes
        memory_efficient (bool) - If True, uses checkpointing. Much more memory efficient,
          but slower. Default: *False*. See `"paper" <https://arxiv.org/pdf/1707.06990.pdf>`_.
    """

    def __init__(
        self,
        growth_rate: int = 32,
        block_config: Tuple[int, int, int, int] = (6, 12, 24, 16),
        num_init_features: int = 64,
        bn_size: int = 4,
        drop_rate: float = 0,
        num_classes: int = 1000,
        memory_efficient: bool = False
    ) -> None:

        super(DenseNet, self).__init__()

        # First convolution
        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, num_init_features, kernel_size=7, stride=2,
                                padding=3, bias=False)),
            ('norm0', nn.BatchNorm2d(num_init_features)),
            ('relu0', nn.ReLU(inplace=True)),
            ('pool0', nn.MaxPool2d(kernel_size=3, stride=2, padding=1)),
        ]))

        # Each denseblock
        num_features = num_init_features
        for i, num_layers in enumerate(block_config):
            block = _DenseBlock(
                num_layers=num_layers,
                num_input_features=num_features,
                bn_size=bn_size,
                growth_rate=growth_rate,
                drop_rate=drop_rate,
                memory_efficient=memory_efficient
            )
            self.features.add_module('denseblock%d' % (i + 1), block)
            num_features = num_features + num_layers * growth_rate
            if i != len(block_config) - 1:
                trans = _Transition(num_input_features=num_features,
                                    num_output_features=num_features // 2)
                self.features.add_module('transition%d' % (i + 1), trans)
                num_features = num_features // 2

        # Final batch norm
        self.features.add_module('norm5', nn.BatchNorm2d(num_features))

        # Linear layer
        self.classifier = nn.Linear(num_features, num_classes)

        # Official init from torch repo.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x: Tensor) -> Tensor:
        features = self.features(x)
        out = F.relu(features, inplace=True)
        #out = F.adaptive_avg_pool2d(out, (1, 1))
        #out = torch.flatten(out, 1)
        #out = self.classifier(out)
        return out


def _load_state_dict(model: nn.Module, model_url: str, progress: bool) -> None:
    # '.'s are no longer allowed in module names, but previous _DenseLayer
    # has keys 'norm.1', 'relu.1', 'conv.1', 'norm.2', 'relu.2', 'conv.2'.
    # They are also in the checkpoints in model_urls. This pattern is used
    # to find such keys.
    pattern = re.compile(
        r'^(.*denselayer\d+\.(?:norm|relu|conv))\.((?:[12])\.(?:weight|bias|running_mean|running_var))$')

    state_dict = load_state_dict_from_url(model_url, progress=progress)
    for key in list(state_dict.keys()):
        res = pattern.match(key)
        if res:
            new_key = res.group(1) + res.group(2)
            state_dict[new_key] = state_dict[key]
            del state_dict[key]
    model.load_state_dict(state_dict)


def _densenet(
    arch: str,
    growth_rate: int,
    block_config: Tuple[int, int, int, int],
    num_init_features: int,
    pretrained: bool,
    progress: bool,
    **kwargs: Any
) -> DenseNet:
    model = DenseNet(growth_rate, block_config, num_init_features, **kwargs)
    if pretrained:
        _load_state_dict(model, model_urls[arch], progress)
    return model


def densenet121(pretrained: bool = False, progress: bool = True, **kwargs: Any) -> DenseNet:
    r"""Densenet-121 model from
    `"Densely Connected Convolutional Networks" <https://arxiv.org/pdf/1608.06993.pdf>`_.
    The required minimum input size of the model is 29x29.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
        memory_efficient (bool) - If True, uses checkpointing. Much more memory efficient,
          but slower. Default: *False*. See `"paper" <https://arxiv.org/pdf/1707.06990.pdf>`_.
    """
    return _densenet('densenet121', 32, (6, 12, 24, 16), 64, pretrained, progress,
                     **kwargs)


def densenet161(pretrained: bool = False, progress: bool = True, **kwargs: Any) -> DenseNet:
    r"""Densenet-161 model from
    `"Densely Connected Convolutional Networks" <https://arxiv.org/pdf/1608.06993.pdf>`_.
    The required minimum input size of the model is 29x29.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
        memory_efficient (bool) - If True, uses checkpointing. Much more memory efficient,
          but slower. Default: *False*. See `"paper" <https://arxiv.org/pdf/1707.06990.pdf>`_.
    """
    return _densenet('densenet161', 48, (6, 12, 36, 24), 96, pretrained, progress,
                     **kwargs)


def densenet169(pretrained: bool = False, progress: bool = True, **kwargs: Any) -> DenseNet:
    r"""Densenet-169 model from
    `"Densely Connected Convolutional Networks" <https://arxiv.org/pdf/1608.06993.pdf>`_.
    The required minimum input size of the model is 29x29.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
        memory_efficient (bool) - If True, uses checkpointing. Much more memory efficient,
          but slower. Default: *False*. See `"paper" <https://arxiv.org/pdf/1707.06990.pdf>`_.
    """
    return _densenet('densenet169', 32, (6, 12, 32, 32), 64, pretrained, progress,
                     **kwargs)


def densenet201(pretrained: bool = False, progress: bool = True, **kwargs: Any) -> DenseNet:
    r"""Densenet-201 model from
    `"Densely Connected Convolutional Networks" <https://arxiv.org/pdf/1608.06993.pdf>`_.
    The required minimum input size of the model is 29x29.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
        memory_efficient (bool) - If True, uses checkpointing. Much more memory efficient,
          but slower. Default: *False*. See `"paper" <https://arxiv.org/pdf/1707.06990.pdf>`_.
    """
    return _densenet('densenet201', 32, (6, 12, 48, 32), 64, pretrained, progress,
                     **kwargs)


### Embedded source: `pretraining/modules.py`

In [ ]:
%%writefile pretraining/modules.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

import pretraining.tools as tools
from torchvision.models.resnet import conv3x3, resnet18, resnet50


class BasicBlockV2(nn.Module):
    """
    Basic Residual Block of ResNet V2
    """
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlockV2, self).__init__()
        self.relu = nn.ReLU(inplace=True)

        self.bn1 = nn.BatchNorm2d(inplanes)
        self.conv1 = conv3x3(inplanes, planes, stride=stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride=1)

        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        # Phase 1
        out = self.bn1(x)
        out = self.relu(out)
        if self.downsample is not None:
            residual = self.downsample(out)
        out = self.conv1(out)

        # Phase 2
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv2(out)

        out += residual

        return out


class BasicBlockV1(nn.Module):
    """
    Basic Residual Block of ResNet V1
    """
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlockV1, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class ResNetV2(nn.Module):
    """
    Adapted fom torchvision ResNet, converted to v2
    """
    def __init__(self,
                 input_channels, num_filters,
                 first_layer_kernel_size, first_layer_conv_stride,
                 blocks_per_layer_list, block_strides_list, block_fn,
                 first_layer_padding=0,
                 first_pool_size=None, first_pool_stride=None, first_pool_padding=0,
                 growth_factor=2):
        super(ResNetV2, self).__init__()
        self.first_conv = nn.Conv2d(
            in_channels=input_channels, out_channels=num_filters,
            kernel_size=first_layer_kernel_size,
            stride=first_layer_conv_stride,
            padding=first_layer_padding,
            bias=False,
        )
        self.first_pool = nn.MaxPool2d(
            kernel_size=first_pool_size,
            stride=first_pool_stride,
            padding=first_pool_padding,
        )

        self.layer_list = nn.ModuleList()
        current_num_filters = num_filters
        self.inplanes = num_filters
        for i, (num_blocks, stride) in enumerate(zip(
                blocks_per_layer_list, block_strides_list)):
            self.layer_list.append(self._make_layer(
                block=block_fn,
                planes=current_num_filters,
                blocks=num_blocks,
                stride=stride,
            ))
            current_num_filters *= growth_factor
        self.final_bn = nn.BatchNorm2d(
            current_num_filters // growth_factor * block_fn.expansion
        )
        self.relu = nn.ReLU()

        # Expose attributes for downstream dimension computation
        self.num_filters = num_filters
        self.growth_factor = growth_factor

    def forward(self, x):
        h = self.first_conv(x)
        h = self.first_pool(h)
        for i, layer in enumerate(self.layer_list):
            h = layer(h)
        h = self.final_bn(h)
        h = self.relu(h)
        return h

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = nn.Sequential(
            nn.Conv2d(self.inplanes, planes * block.expansion,
                      kernel_size=1, stride=stride, bias=False),
        )

        layers_ = [
            block(self.inplanes, planes, stride, downsample)
        ]
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers_.append(block(self.inplanes, planes))

        return nn.Sequential(*layers_)


class ResNetV1(nn.Module):
    """
    Class that represents a ResNet with classifier sequence removed
    """
    def __init__(self, initial_filters, block, layers, input_channels=1):

        self.inplanes = initial_filters
        self.num_layers = len(layers)
        super(ResNetV1, self).__init__()

        # initial sequence
        # the first sequence only has 1 input channel which is different from original ResNet
        self.conv1 = nn.Conv2d(input_channels, initial_filters, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(initial_filters)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # residual sequence
        for i in range(self.num_layers):
            num_filters = initial_filters * pow(2,i)
            num_stride = (1 if i == 0 else 2)
            setattr(self, 'layer{0}'.format(i+1), self._make_layer(block, num_filters, layers[i], stride=num_stride))
        self.num_filter_last_seq = initial_filters * pow(2, self.num_layers-1)

        # initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        # first sequence
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # residual sequences
        for i in range(self.num_layers):
            x = getattr(self, 'layer{0}'.format(i+1))(x)
        return x


class DownsampleNetworkResNet18V1(nn.Module):
    """
    Downsampling using ResNet V1
    First conv is 7*7, stride 2, padding 3, cut 1/2 resolution
    """
    def __init__(self):
        super(DownsampleNetworkResNet18V1, self).__init__()
        self.f = []
        # backbone = resnet50(pretrained=True)
        backbone = resnet18(pretrained=True)

        
        for name, module in backbone.named_children():
            if name != 'fc' and name != 'avgpool':
                self.f.append(module)
        #print(self.f)
        # encoder
        self.f = nn.Sequential(*self.f)

    def forward(self, x):
        last_feature_map = self.f(x)
        #print(last_feature_map.shape)
        return last_feature_map


class AbstractMILUnit:
    """
    An abstract class that represents an MIL unit module
    """
    def __init__(self, parameters, parent_module):
        self.parameters = parameters
        self.parent_module = parent_module


class PostProcessingStandard(nn.Module):
    """
    Unit in Global Network that takes in x_out and produce saliency maps
    """
    def __init__(self, parameters):
        super(PostProcessingStandard, self).__init__()
        # map all filters to output classes
        self.gn_conv_last = nn.Conv2d(parameters["post_processing_dim"],
                                      parameters["num_classes"],
                                      (1, 1), bias=False)

    def forward(self, x_out):
        out = self.gn_conv_last(x_out)
        return out


class GlobalNetwork(AbstractMILUnit):
    """
    Implementation of Global Network using ResNet-22
    """
    def __init__(self, parameters, parent_module):
        super(GlobalNetwork, self).__init__(parameters, parent_module)
        # downsampling-branch
        if "use_v1_global" in parameters and parameters["use_v1_global"]:
            self.downsampling_branch = DownsampleNetworkResNet18V1()
        else:
            self.downsampling_branch = ResNetV2(input_channels=1, num_filters=16,
                     # first conv layer
                     first_layer_kernel_size=(7,7), first_layer_conv_stride=2,
                     first_layer_padding=3,
                     # first pooling layer
                     first_pool_size=3, first_pool_stride=2, first_pool_padding=0,
                     # res blocks architecture
                     blocks_per_layer_list=[2, 2, 2, 2, 2],
                     block_strides_list=[1, 2, 2, 2, 2],
                     block_fn=BasicBlockV2,
                     growth_factor=2)
        # post-processing
        self.postprocess_module = PostProcessingStandard(parameters)

    def add_layers(self):
        self.parent_module.ds_net = self.downsampling_branch
        self.parent_module.left_postprocess_net = self.postprocess_module

    def forward(self, x):
        # retrieve results from downsampling network at all 4 levels
        last_feature_map = self.downsampling_branch.forward(x)
        # feed into postprocessing network
        cam = self.postprocess_module.forward(last_feature_map)
        return last_feature_map, cam





class TopTPercentAggregationFunction(AbstractMILUnit):
    """
    An aggregator that uses the SM to compute the y_global.
    Use the sum of topK value
    """
    def __init__(self, parameters, parent_module):
        super(TopTPercentAggregationFunction, self).__init__(parameters, parent_module)
        self.percent_t = parameters["percent_t"]
        self.parent_module = parent_module

    def forward(self, cam):
        batch_size, num_class, H, W = cam.size()
        cam_flatten = cam.view(batch_size, num_class, -1)
        top_t = int(round(W*H*self.percent_t))
        selected_area = cam_flatten.topk(top_t, dim=2)[0]
        return selected_area.mean(dim=2)


class RetrieveROIModule(AbstractMILUnit):
    """
    A Regional Proposal Network instance that computes the locations of the crops
    Greedy select crops with largest sums
    """
    def __init__(self, parameters, parent_module):
        super(RetrieveROIModule, self).__init__(parameters, parent_module)
        self.crop_method = "upper_left"
        self.num_crops_per_class = parameters["K"]
        self.crop_shape = parameters["crop_shape"]
        # self.gpu_number = None

    def forward(self, x_original, cam_size, h_small):
        """
        Function that use the low-res image to determine the position of the high-res crops
        :param x_original: N, C, H, W pytorch tensor
        :param cam_size: (h, w)
        :param h_small: N, C, h_h, w_h pytorch tensor
        :return: N, num_classes*k, 2 numpy matrix; returned coordinates are corresponding to x_small
        """
        # retrieve parameters
        _, _, H, W = x_original.size()
        (h, w) = cam_size
        N, C, h_h, w_h = h_small.size()
        #print(h_small.size())
        # make sure that the size of h_small == size of cam_size
        # print(h_h)
        # print(cam_size)
        assert h_h == h, "h_h!=h"
        assert w_h == w, "w_h!=w"

        # adjust crop_shape since crop shape is based on the original image
        crop_x_adjusted = int(np.round(self.crop_shape[0] * h / H))
        crop_y_adjusted = int(np.round(self.crop_shape[1] * w / W))
        crop_shape_adjusted = (crop_x_adjusted, crop_y_adjusted)

        # greedily find the box with max sum of weights
        current_images = h_small
        all_max_position = []
        # combine channels
        max_vals = current_images.view(N, C, -1).max(dim=2, keepdim=True)[0].unsqueeze(-1)
        min_vals = current_images.view(N, C, -1).min(dim=2, keepdim=True)[0].unsqueeze(-1)
        range_vals = max_vals - min_vals
        normalize_images = current_images - min_vals
        normalize_images = normalize_images / range_vals
        current_images = normalize_images.sum(dim=1, keepdim=True)

        for _ in range(self.num_crops_per_class):
            max_pos = tools.get_max_window(current_images, crop_shape_adjusted, "avg")
            all_max_position.append(max_pos)
            mask = tools.generate_mask_uplft(current_images, crop_shape_adjusted, max_pos, current_images.device)
            current_images = current_images * mask
        return torch.cat(all_max_position, dim=1).data.cpu().numpy()
    
    
class LocalNetwork(AbstractMILUnit):
    """
    The local network that takes a crop and computes its hidden representation
    Use ResNet
    """
    def add_layers(self):
        """
        Function that add layers to the parent module that implements nn.Module
        :return:
        """
        self.parent_module.dn_resnet = DownsampleNetworkResNet18V1()

    def forward(self, x_crop):
        """
        Function that takes in a single crop and return the hidden representation
        :param x_crop: (N,C,h,w)
        :return:
        """
        # forward propagte using ResNet
        res = self.parent_module.dn_resnet(x_crop.expand(-1, 3, -1 , -1))
        # global average pooling
        res = res.mean(dim=2).mean(dim=2)
        return res


class AttentionModule(AbstractMILUnit):
    """
    The attention module takes multiple hidden representations and compute the attention-weighted average
    Use Gated Attention Mechanism in https://arxiv.org/pdf/1802.04712.pdf
    """
    def add_layers(self):
        """
        Function that add layers to the parent module that implements nn.Module
        :return:
        """
        # The gated attention mechanism
        self.parent_module.mil_attn_V = nn.Linear(512, 128, bias=False)
        self.parent_module.mil_attn_U = nn.Linear(512, 128, bias=False)
        self.parent_module.mil_attn_w = nn.Linear(128, 1, bias=False)
        # classifier
        self.parent_module.classifier_linear = nn.Linear(512, self.parameters["num_classes"], bias=False)

    def forward(self, h_crops):
        """
        Function that takes in the hidden representations of crops and use attention to generate a single hidden vector
        :param h_small:
        :param h_crops:
        :return:
        """
        batch_size, num_crops, h_dim = h_crops.size()
        h_crops_reshape = h_crops.view(batch_size * num_crops, h_dim)
        # calculate the attn score
        attn_projection = torch.sigmoid(self.parent_module.mil_attn_U(h_crops_reshape)) * \
                          torch.tanh(self.parent_module.mil_attn_V(h_crops_reshape))
        attn_score = self.parent_module.mil_attn_w(attn_projection)
        # use softmax to map score to attention
        attn_score_reshape = attn_score.view(batch_size, num_crops)
        attn = F.softmax(attn_score_reshape, dim=1)
        
        # final hidden vector
        z_weighted_avg = torch.sum(attn.unsqueeze(-1) * h_crops, 1)
        
        # map to the final layer
        y_crops = self.parent_module.classifier_linear(z_weighted_avg)
        return z_weighted_avg, attn, y_crops

### Embedded source: `pretraining/tools.py`

In [ ]:
%%writefile pretraining/tools.py
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn.functional as F



def partition_batch(ls, size):
    """
    Partitions a list into buckets of given maximum length.
    """
    i = 0
    partitioned_lists = []
    while i < len(ls):
        partitioned_lists.append(ls[i: i+size])
        i += size
    return partitioned_lists


def make_sure_in_range(val, min_val, max_val):
    """
    Function that make sure that min < val < max; otherwise return the limit value
    """
    if val < min_val:
        return min_val
    if val > max_val:
        return max_val
    return val


def crop(original_img, crop_shape, crop_position, method="center",
         in_place=False, background_val="min"):
    """
    Function that take a crop on the original image.
    This function must staty in numpy since original_img should not be loaded into Pytorch during the network time.
    original_img is large and would consume lots of GPU memory.
    :param original_img:
    :param crop_shape:
    :param crop_position:
    :param method: supported in ["center", "upper_left"]
    :param in_place: if in_place, the effective pixels in the crop will be flagged (1.0) in the original_img
    """
    # retrieve inputs
    I, J = original_img.shape
    crop_x, crop_y = crop_position
    x_delta, y_delta = crop_shape

    # locate the four corners
    if method == "center":
        min_x = int(np.round(crop_x - x_delta / 2))
        max_x = int(np.round(crop_x + x_delta / 2))
        min_y = int(np.round(crop_y - y_delta / 2))
        max_y = int(np.round(crop_y + y_delta/2))
    elif method == "upper_left":
        min_x = int(np.round(crop_x))
        max_x = int(np.round(crop_x + x_delta))
        min_y = int(np.round(crop_y))
        max_y = int(np.round(crop_y + y_delta))

    # make sure that the crops are in range
    min_x = make_sure_in_range(min_x, 0, I)
    max_x = make_sure_in_range(max_x, 0, I)
    min_y = make_sure_in_range(min_y, 0, J)
    max_y = make_sure_in_range(max_y, 0, J)

    # if in_place, flag the original inputs
    if in_place:
        original_img[min_x:max_x, min_y:max_y] = 1.0
    # else make the new matrix
    else:
        # somehow background is normalized to this number
        if background_val == "min":
            output = np.ones(crop_shape) * np.min(original_img)
        else:
            output = np.ones(crop_shape) * background_val
        real_x_delta = max_x - min_x
        real_y_delta = max_y - min_y
        origin_x = crop_shape[0] - real_x_delta
        origin_y = crop_shape[1] - real_y_delta
        output[origin_x:, origin_y:] = original_img[min_x:max_x, min_y:max_y]
        return output


def get_crop_mask(loc, crop_shape, image_shape, method, indicator=True):
    """
    Function that generates the mask
    :param loc:
    :param crop_shape:
    :param image_shape:
    :param method:
    :return:
    """
    crop_map = np.zeros(image_shape)
    for crop_loc in loc:
        # this is the indicator for point of crop
        if indicator:
            crop_map[int(crop_loc[0]), int(crop_loc[1])] = 999.0
        # fill in 1.0 in the cropped regions
        crop(crop_map, crop_shape, crop_loc, method=method, in_place=True)
    return crop_map


def crop_pytorch(original_img_pytorch, crop_shape, crop_position, out,
                 method="center", background_val="min"):
    """
    Function that take a crop on the original image.
    Use PyTorch to do this.
    :param original_img_pytorch: (N,C,H,W) PyTorch Tensor
    :param crop_shape: (h, w) integer tuple
    :param method: supported in ["center", "upper_left"]
    :return: (N, K, h, w) PyTorch Tensor
    """
    # retrieve inputs
    H, W = original_img_pytorch.shape
    crop_x, crop_y = crop_position
    x_delta, y_delta = crop_shape

    # locate the four corners
    if method == "center":
        min_x = int(np.round(crop_x - x_delta / 2))
        max_x = int(np.round(crop_x + x_delta / 2))
        min_y = int(np.round(crop_y - y_delta / 2))
        max_y = int(np.round(crop_y + y_delta / 2))
    elif method == "upper_left":
        min_x = int(np.round(crop_x))
        max_x = int(np.round(crop_x + x_delta))
        min_y = int(np.round(crop_y))
        max_y = int(np.round(crop_y + y_delta))

    # make sure that the crops are in range
    min_x = make_sure_in_range(min_x, 0, H)
    max_x = make_sure_in_range(max_x, 0, H)
    min_y = make_sure_in_range(min_y, 0, W)
    max_y = make_sure_in_range(max_y, 0, W)

    # somehow background is normalized to this number
    if background_val == "min":
        out[:, :] = original_img_pytorch.min()
    else:
        out[:, :] = background_val
    real_x_delta = max_x - min_x
    real_y_delta = max_y - min_y
    origin_x = crop_shape[0] - real_x_delta
    origin_y = crop_shape[1] - real_y_delta
    out[origin_x:, origin_y:] = original_img_pytorch[min_x:max_x, min_y:max_y]


def get_max_window(input_image, window_shape, pooling_logic="avg"):
    """
    Function that makes a sliding window of size window_shape over the
    input_image and return the UPPER_LEFT corner index with max sum
    :param input_image: N*C*H*W
    :param window_shape: h*w
    :return: N*C*2 tensor
    """
    N, C, H, W = input_image.size()
    if pooling_logic == "avg":
        # use average pooling to locate the window sums
        pool_map = torch.nn.functional.avg_pool2d(input_image, window_shape, stride=1)
    elif pooling_logic in ["std", "avg_entropy"]:
        # create sliding windows
        output_size = (H - window_shape[0] + 1, W - window_shape[1] + 1)
        sliding_windows = F.unfold(input_image, kernel_size=window_shape).view(N,C, window_shape[0]*window_shape[1], -1)
        # apply aggregation function on each sliding windows
        if pooling_logic == "std":
            agg_res = sliding_windows.std(dim=2, keepdim=False)
        elif pooling_logic == "avg_entropy":
            agg_res = -sliding_windows*torch.log(sliding_windows)-(1-sliding_windows)*torch.log(1-sliding_windows)
            agg_res = agg_res.mean(dim=2, keepdim=False)
        # merge back
        pool_map = F.fold(agg_res, kernel_size=(1, 1), output_size=output_size)
    _, _, _, W_map = pool_map.size()
    # transform to linear and get the index of the max val locations
    _, max_linear_idx = torch.max(pool_map.view(N, C, -1), -1)
    # convert back to 2d index
    #max_idx_x = max_linear_idx // W_map
    max_idx_x = torch.div(max_linear_idx, W_map, rounding_mode='trunc')
    max_idx_y = max_linear_idx - max_idx_x * W_map
    # put together the 2d index
    upper_left_points = torch.cat([max_idx_x.unsqueeze(-1), max_idx_y.unsqueeze(-1)], dim=-1)
    return upper_left_points


def generate_mask_uplft(input_image, window_shape, upper_left_points, gpu_number):
    """
    Function that generates mask that sets crops given upper_left
    corners to 0
    :param input_image:
    :param window_shape:
    :param upper_left_points:
    """
    N, C, H, W = input_image.size()
    window_h, window_w = window_shape
    # get the positions of masks
    mask_x_min = upper_left_points[:,:,0]
    mask_x_max = upper_left_points[:,:,0] + window_h
    mask_y_min = upper_left_points[:,:,1]
    mask_y_max = upper_left_points[:,:,1] + window_w
    # generate masks
    mask_x = Variable(torch.arange(0, H).view(-1, 1).repeat(N, C, 1, W))
    mask_y = Variable(torch.arange(0, W).view(1, -1).repeat(N, C, H, 1))
    if gpu_number is not None:
        device = gpu_number
        mask_x = mask_x.cuda().to(device)
        mask_y = mask_y.cuda().to(device)
    x_gt_min = mask_x.float() >= mask_x_min.unsqueeze(-1).unsqueeze(-1).float()
    x_ls_max = mask_x.float() < mask_x_max.unsqueeze(-1).unsqueeze(-1).float()
    y_gt_min = mask_y.float() >= mask_y_min.unsqueeze(-1).unsqueeze(-1).float()
    y_ls_max = mask_y.float() < mask_y_max.unsqueeze(-1).unsqueeze(-1).float()

    # since logic operation is not supported for variable
    # I used * for logic ANd
    selected_x = x_gt_min * x_ls_max
    selected_y = y_gt_min * y_ls_max
    selected = selected_x * selected_y
    mask = 1 - selected.float()
    return mask

### Embedded source: `tests/test_aptos_pipeline.py`

In [ ]:
%%writefile tests/test_aptos_pipeline.py
import csv, tempfile, unittest
from pathlib import Path
from PIL import Image
from aptos_research.data import DatasetError, resolve_aptos, make_splits, save_split_manifest, load_split_manifest
from aptos_research.metrics import classification_metrics

class AptosTests(unittest.TestCase):
 def setUp(self):
  self.tmp=tempfile.TemporaryDirectory();self.root=Path(self.tmp.name);(self.root/'train_images').mkdir()
  with (self.root/'train.csv').open('w',newline='') as f:
   w=csv.writer(f);w.writerow(['id_code','diagnosis'])
   for i in range(20):
    w.writerow(['img%02d'%i,i%5]);Image.new('RGB',(20+i,30),(i,2,3)).save(self.root/'train_images'/('IMG%02d.JPG'%i))
 def tearDown(self):self.tmp.cleanup()
 def test_csv_ids_case_insensitive_and_distribution(self):
  records,stats=resolve_aptos(str(self.root));self.assertEqual(len(records),20);self.assertEqual(stats['class_counts'],{'0':4,'1':4,'2':4,'3':4,'4':4})
 def test_splits_are_disjoint_and_deterministic(self):
  records,_=resolve_aptos(str(self.root));a=make_splits(records,seed=7);b=make_splits(records,seed=7);self.assertEqual(a,b)
  ids=[{x['id'] for x in a[k]} for k in ('train','validation','test')];self.assertFalse(ids[0]&ids[1]);self.assertFalse(ids[0]&ids[2]);self.assertFalse(ids[1]&ids[2])
  manifest=self.root/'splits.json';save_split_manifest(a,manifest,7);self.assertEqual(load_split_manifest(manifest)['seed'],7)
 def test_invalid_label_and_missing_image_fail(self):
  p=self.root/'train.csv';p.write_text('id_code,diagnosis\nimg00,8\n',encoding='utf-8')
  with self.assertRaises(DatasetError):resolve_aptos(str(self.root))
  p.write_text('id_code,diagnosis\nabsent,1\n',encoding='utf-8')
  with self.assertRaises(DatasetError):resolve_aptos(str(self.root))
 def test_multiclass_metrics_include_ordinal_and_calibration(self):
  result=classification_metrics([0,1,2,3,4],[[1,0,0,0,0],[0,1,0,0,0],[0,0,1,0,0],[0,0,0,1,0],[0,0,0,0,1]])
  self.assertEqual(result['accuracy'],1);self.assertEqual(result['quadratic_weighted_kappa'],1);self.assertEqual(result['ordinal_mae'],0);self.assertEqual(result['macro_auroc_ovr'],1);self.assertEqual(result['ece_10_bins'],0)

if __name__=='__main__':unittest.main()


### Embedded source: `tests/test_research_components.py`

In [ ]:
%%writefile tests/test_research_components.py
import tempfile, unittest
from pathlib import Path
from types import SimpleNamespace
from aptos_research.gate import evaluate_gate, qrc_is_eligible
from aptos_research.features import FeatureStore
from aptos_research.checkpoint import inspect_state_dict

class ResearchTests(unittest.TestCase):
 def test_gate_decisions_and_artifact(self):
  with tempfile.TemporaryDirectory() as d:
   out=Path(d)/'gate.json'
   inc=evaluate_gate({},out,bootstrap_samples=200);self.assertEqual(inc['decision'],'INCONCLUSIVE')
   labels=[0]*100
   passed=evaluate_gate({'validation':{'ids':['x%d'%i for i in range(100)],'labels':labels,'ordered_predictions':labels,'control_predictions':[[1]*40+[0]*60]}},out,minimum_delta=.05,bootstrap_samples=1000,seed=1)
   self.assertEqual(passed['decision'],'PASS');self.assertTrue(qrc_is_eligible(out))
   failed=evaluate_gate({'validation':{'ids':['x%d'%i for i in range(100)],'labels':labels,'ordered_predictions':labels,'control_predictions':[labels]}},out,bootstrap_samples=100,seed=1)
   self.assertEqual(failed['decision'],'FAIL');self.assertFalse(qrc_is_eligible(out))
 def test_feature_store_schema_and_empty_integrity(self):
  with tempfile.TemporaryDirectory() as d:
   store=FeatureStore(str(Path(d)/'features.sqlite'));self.assertEqual(store.integrity()['splits'],{});store.close()
 def test_checkpoint_compatibility_reports_shapes_and_keys(self):
  target={'a':SimpleNamespace(shape=(2,3)),'b':SimpleNamespace(shape=(5,))}
  incoming={'a':SimpleNamespace(shape=(2,3)),'b':SimpleNamespace(shape=(4,)),'unused':SimpleNamespace(shape=(1,))}
  result=inspect_state_dict(target,incoming)['report']
  self.assertEqual(result['loaded_keys'],['a']);self.assertIn('b',result['mismatched_shapes']);self.assertEqual(result['unexpected_keys'],['unused'])

if __name__=='__main__':unittest.main()


### Embedded source: `utils.py`

In [ ]:
%%writefile utils.py
import random
import math
import numpy as np
import argparse
import torch
import torch.optim as optim
import torchvision
from torch import nn
from torchvision import transforms
from dataloader.loading import *
import torch.nn.functional as F

def set_random_seed(seed):
    print(f"\n* Set seed {seed}")
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)  # torch.cuda.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)


def dict2namespace(config):
    namespace = argparse.Namespace()
    for key, value in config.items():
        if isinstance(value, dict):
            new_value = dict2namespace(value)
        else:
            new_value = value
        setattr(namespace, key, new_value)
    return namespace


def sizeof_fmt(num, suffix='B'):
    """
    https://stackoverflow.com/questions/24455615/python-how-to-display-size-of-all-variables
    """
    for unit in ['', 'Ki', 'Mi', 'Gi', 'Ti', 'Pi', 'Ei', 'Zi']:
        if abs(num) < 1024.0:
            return "%3.1f %s%s" % (num, unit, suffix)
        num /= 1024.0
    return "%.1f %s%s" % (num, 'Yi', suffix)


# print("Check memory usage of different variables:")
# for name, size in sorted(((name, sys.getsizeof(value)) for name, value in locals().items()),
#                          key=lambda x: -x[1])[:10]:
#     print("{:>30}: {:>8}".format(name, sizeof_fmt(size)))


def get_optimizer(config_optim, parameters):
    if config_optim.optimizer == 'Adam':
        return optim.Adam(parameters, lr=config_optim.lr, weight_decay=config_optim.weight_decay,
                          betas=(config_optim.beta1, 0.999), amsgrad=config_optim.amsgrad,
                          eps=config_optim.eps)
    elif config_optim.optimizer == 'AdamW':
        return optim.AdamW(parameters, lr=1e-3, weight_decay=0.05,
                          betas=(config_optim.beta1, 0.999),
                          eps=1e-8)
    elif config_optim.optimizer == 'RMSProp':
        return optim.RMSprop(parameters, lr=config_optim.lr, weight_decay=config_optim.weight_decay)
    elif config_optim.optimizer == 'SGD':
        return optim.SGD(parameters, lr=config_optim.lr, weight_decay=1e-4, momentum=0.9)
    else:
        raise NotImplementedError(
            'Optimizer {} not understood.'.format(config_optim.optimizer))


def get_optimizer_and_scheduler(config, parameters, epochs, init_epoch):
    scheduler = None
    optimizer = get_optimizer(config, parameters)
    if hasattr(config, "T_0"):
        T_0 = config.T_0
    else:
        T_0 = epochs // (config.n_restarts + 1)
    if config.use_scheduler:
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,
                                                                   T_0=T_0,
                                                                   T_mult=config.T_mult,
                                                                   eta_min=config.eta_min,
                                                                   last_epoch=-1)
        scheduler.last_epoch = init_epoch - 1
    return optimizer, scheduler


def adjust_learning_rate(optimizer, epoch, config):
    """Decay the learning rate with half-cycle cosine after warmup"""
    if epoch < config.training.warmup_epochs:
        lr = config.optim.lr * epoch / config.training.warmup_epochs
    else:
        lr = config.optim.min_lr + (config.optim.lr - config.optim.min_lr) * 0.5 * \
             (1. + math.cos(math.pi * (epoch - config.training.warmup_epochs) / (
                     config.training.n_epochs - config.training.warmup_epochs)))
    for param_group in optimizer.param_groups:
        if "lr_scale" in param_group:
            param_group["lr"] = lr * param_group["lr_scale"]
        else:
            param_group["lr"] = lr
    return lr


def get_dataset(config):
    data_object = None
    if config.data.dataset == "PLACENTAL":
        train_dataset = BUDataset(data_list=config.data.traindata, train=True)
        test_dataset = BUDataset(data_list=config.data.testdata, train=False)
    elif config.data.dataset == "APTOS":
        if hasattr(config.data, "splits_manifest"):
            from aptos_research.data import load_split_manifest, AptosImageDataset
            manifest = load_split_manifest(config.data.splits_manifest)
            train_dataset = AptosImageDataset(manifest['splits']['train'], 'train', size=224, training=True)
            test_dataset = AptosImageDataset(manifest['splits']['validation'], 'validation', size=224, training=False)
        else:
            # Retain the original pickle-list route for existing APTOS experiments.
            train_dataset = APTOSDataset(data_list=config.data.traindata, train=True)
            test_dataset = APTOSDataset(data_list=config.data.testdata, train=False)
    elif config.data.dataset == "ISIC":
        train_dataset = ISICDataset(data_list=config.data.traindata, train=True)
        test_dataset = ISICDataset(data_list=config.data.testdata, train=False)
    elif config.data.dataset == "CHEST":

        train_dataset = ChestXrayDataSet(image_list_file=config.data.traindata, train=True)
        test_dataset = ChestXrayDataSet(image_list_file=config.data.testdata, train=False)
    else:
        raise NotImplementedError(
            "Options: toy (classification of two Gaussian), MNIST, FashionMNIST, CIFAR10.")
    return data_object, train_dataset, test_dataset

from sklearn.metrics import cohen_kappa_score
# ------------------------------------------------------------------------------------
# Revised from timm == 0.3.2
# https://github.com/rwightman/pytorch-image-models/blob/master/timm/utils/metrics.py
# output: the prediction from diffusion model (B x n_classes)
# target: label indices (B)
# ------------------------------------------------------------------------------------
def accuracy(output, target, topk=(1,)):
    """
    Computes the accuracy over the k top predictions for the specified values of k.
    """
    maxk = min(max(topk), output.size()[1])
    # output = torch.softmax(-(output - 1)**2,  dim=-1)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))
    return [correct[:min(k, maxk)].reshape(-1).float().sum(0) * 100. / batch_size for k in topk]



def cohen_kappa(output, target, topk=(1,)):
    maxk = min(max(topk), output.size()[1])
    _, pred = output.topk(maxk, 1, True, True)
    kappa = cohen_kappa_score(pred, target, weights='quadratic')
    return kappa


def cast_label_to_one_hot_and_prototype(y_labels_batch, config, return_prototype=True):
    """
    y_labels_batch: a vector of length batch_size.
    """
    y_one_hot_batch = nn.functional.one_hot(y_labels_batch, num_classes=config.data.num_classes).float()
    if return_prototype:
        label_min, label_max = config.data.label_min_max
        y_logits_batch = torch.logit(nn.functional.normalize(
            torch.clip(y_one_hot_batch, min=label_min, max=label_max), p=1.0, dim=1))
        return y_one_hot_batch, y_logits_batch
    else:
        return y_one_hot_batch




import numpy as np
import sklearn.metrics as metrics
#from imblearn.metrics import sensitivity_score, specificity_score
import pdb
# from sklearn.metrics.ranking import roc_auc_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def compute_isic_metrics(gt, pred):
    gt_np = gt.cpu().detach().numpy()
    pred_np = pred.cpu().detach().numpy()

    gt_class = np.argmax(gt_np, axis=1)
    pred_class = np.argmax(pred_np, axis=1)

    ACC = accuracy_score(gt_class, pred_class)
    BACC = balanced_accuracy_score(gt_class, pred_class) # balanced accuracy
    Prec = precision_score(gt_class, pred_class, average='macro')
    Rec = recall_score(gt_class, pred_class, average='macro')
    F1 = f1_score(gt_class, pred_class, average='macro')
    try:
        AUC_ovo = metrics.roc_auc_score(gt_np, pred_np)
    except:
        AUC_ovo=0.
    #AUC_macro = metrics.roc_auc_score(gt_class, pred_np, average='macro', multi_class='ovo')

    #SPEC = specificity_score(gt_class, pred_class, average='macro')

    kappa = cohen_kappa_score(gt_class, pred_class, weights='quadratic')

    # cm = confusion_matrix(gt_class, pred_class)

    # sns.heatmap(cm, 
    #         annot=True,
    #         fmt='g')
    # plt.ylabel('Prediction',fontsize=13)
    # plt.xlabel('Actual',fontsize=13)
    # plt.title('Confusion Matrix',fontsize=17)
    # plt.savefig('confusion_matrix_placental.png')

    # print(confusion_matrix(gt_class, pred_class))
    return ACC, BACC, Prec, Rec, F1, AUC_ovo, kappa
    #return ACC, BACC, Prec, Rec, F1, AUC_ovo, AUC_macro, SPEC, kappa

def compute_f1_score(gt, pred):
    gt_class = gt.cpu().detach().numpy()
    pred_np = pred.cpu().detach().numpy()

    #gt_class = np.argmax(gt_np, axis=1)
    pred_class = np.argmax(pred_np, axis=1)

    F1 = f1_score(gt_class, pred_class, average='macro')
    #AUC_ovo = metrics.roc_auc_score(gt_np, pred_np, average='macro', multi_class='ovo')
    #AUC_macro = metrics.roc_auc_score(gt_class, pred_np, average='macro', multi_class='ovo')

    #SPEC = specificity_score(gt_class, pred_class, average='macro')

    # print(confusion_matrix(gt_class, pred_class))
    return F1


def compute_kernel(x, y):
    x_size = x.size(0)
    y_size = y.size(0)
    dim = x.size(1)
    x = x.unsqueeze(1) # (x_size, 1, dim)
    y = y.unsqueeze(0) # (1, y_size, dim)
    tiled_x = x.expand(x_size, y_size, dim)
    tiled_y = y.expand(x_size, y_size, dim)
    kernel_input = (tiled_x - tiled_y).pow(2).mean(2)/float(dim)
    return torch.exp(-kernel_input) # (x_size, y_size)

def compute_mmd(x, y):
    x_kernel = compute_kernel(x, x)
    y_kernel = compute_kernel(y, y)
    xy_kernel = compute_kernel(x, y)
    mmd = x_kernel.mean() + y_kernel.mean() - 2*xy_kernel.mean()
    return mmd

from sklearn.metrics import roc_auc_score
def compute_AUCs(gt, pred):
    """Computes Area Under the Curve (AUC) from prediction scores.
    Args:
        gt: Pytorch tensor on GPU, shape = [n_samples, n_classes]
          true binary labels.
        pred: Pytorch tensor on GPU, shape = [n_samples, n_classes]
          can either be probability estimates of the positive class,
          confidence values, or binary decisions.
    Returns:
        List of AUROCs of all classes.
    """
    AUROCs = []
    gt_np = gt.cpu().numpy()
    pred_np = pred.detach().cpu().numpy()
    #AUROCs_mean = roc_auc_score(gt_np[:, 0], pred_np[:, 0])
    pred_np[np.where(np.isnan(pred_np))] = 0.
    for i in range(15):
        AUROCs.append(roc_auc_score(gt_np[:, i], pred_np[:, i],average=None))
    AUROCs_mean = np.array(AUROCs[:-1]).mean()
    print(AUROCs)
    return AUROCs_mean



import scipy.special

# for left-multiplication for RGB -> Y'PbPr
RGB_TO_YUV = np.array([[0.29900, -0.16874, 0.50000],
                       [0.58700, -0.33126, -0.41869],
                       [0.11400, 0.50000, -0.08131]])


def normalize_data(x, mode=None):
    if mode is None or mode == 'rgb':
        return x / 127.5 - 1.
    elif mode == 'rgb_unit_var':
        return 2. * normalize_data(x, mode='rgb')
    elif mode == 'yuv':
        return (x / 127.5 - 1.).dot(RGB_TO_YUV)
    else:
        raise NotImplementedError(mode)


def log_min_exp(a, b, epsilon=1.e-6):
    """Computes the log(exp(a) - exp(b)) (b<a) in a numerically stable fashion."""
    y = a + torch.log1p(-torch.exp(b - a) + epsilon)
    return y


def categorical_kl_logits(logits1, logits2, eps=1.e-6):
    """KL divergence between categorical distributions.

    Distributions parameterized by logits.

    Args:
      logits1: logits of the first distribution. Last dim is class dim.
      logits2: logits of the second distribution. Last dim is class dim.
      eps: float small number to avoid numerical issues.

    Returns:
      KL(C(logits1) || C(logits2)): shape: logits1.shape[:-1]
    """
    out = (F.softmax(logits1 + eps, dim=-1) * (
                F.log_softmax(logits1 + eps, dim=-1) - F.log_softmax(logits2 + eps, dim=-1)))
    return torch.sum(out, dim=-1)


def categorical_kl_probs(probs1, probs2, eps=1.e-6):
    """KL divergence between categorical distributions.

    Distributions parameterized by logits.

    Args:
      probs1: probs of the first distribution. Last dim is class dim.
      probs2: probs of the second distribution. Last dim is class dim.
      eps: float small number to avoid numerical issues.

    Returns:
      KL(C(probs) || C(logits2)): shape: logits1.shape[:-1]
    """
    out = probs1 * (torch.log(probs1 + eps) - torch.log(probs2 + eps))
    return torch.sum(out, dim=-1)


def categorical_log_likelihood(x, logits):
    """Log likelihood of a discretized Gaussian specialized for image data.

    Assumes data `x` consists of integers [0, num_classes-1].

    Args:
      x: where to evaluate the distribution. shape = (bs, ...), dtype=int32/int64
      logits: logits, shape = (bs, ..., num_classes)

    Returns:
      log likelihoods
    """
    log_probs = F.log_softmax(logits, dim=-1)
    x_onehot = F.one_hot(x.to(torch.int64), logits.shape[-1])
    return torch.sum(log_probs * x_onehot, dim=-1)


def meanflat(x):
    """Take the mean over all axes except the first batch dimension."""
    return x.mean(dim=tuple(range(1, len(x.shape))))


### Embedded runtime file: `configs/aptos.yml`

In [ ]:
%%writefile configs/aptos.yml
data:
  dataset: APTOS
  seed: 2026
  label_min_max: [0.001, 0.999]
  num_classes: 5
  num_workers: 4
  dataroot: ${APTOS_ROOT}
  splits_manifest: outputs/aptos_splits.json
  aux_checkpoint: ${DIFFMIC_AUX_CHECKPOINT}
model:
  type: simple
  data_dim: 150528
  n_input_channels: 3
  n_input_padding: 0
  feature_dim: 6144
  hidden_dim: 6144
  cat_x: true
  cat_y_pred: true
  arch: resnet18
  var_type: fixedlarge
  ema_rate: 0.9999
  ema: true
  num_k: 6
diffusion:
  beta_schedule: linear
  beta_start: 0.0001
  beta_end: 0.02
  timesteps: 1000
  test_timesteps: 100
  vis_step: 100
  num_figs: 10
  include_guidance: true
  apply_aux_cls: true
  aux_cls:
    arch: resnet18_ckpt
    pre_train: true
    joint_train: false
    n_pretrain_epochs: 10
    logging_interval: 1
training:
  batch_size: 32
  n_epochs: 1000
  warmup_epochs: 40
  validation_freq: 5
  logging_freq: 100
  snapshot_freq: 1
  image_folder: outputs/training_images
testing:
  batch_size: 32
optim:
  optimizer: Adam
  lr: 0.001
  weight_decay: 0.0
  beta1: 0.9
  amsgrad: false
  eps: 1.0e-8
  min_lr: 1.0e-6
  grad_clip: 1.0


### Embedded runtime file: `configs/placental.yml`

In [ ]:
%%writefile configs/placental.yml
data:
    dataset: "PLACENTAL"
    seed: 2000
    label_min_max: [0.001, 0.999]
    num_classes: 4
    num_workers: 8
    dataroot: './dataset/'
    traindata: './dataset/ultrasound_train_list.pkl'
    testdata: './dataset/ultrasound_test_list.pkl'

model:
    type: "simple"
    data_dim: 150528
    #data_dim: 442368
#    data_dim: [3, 32, 32]
    n_input_channels: 3
    n_input_padding: 0
    feature_dim: 6144
    hidden_dim: 6144
    cat_x: True
    cat_y_pred: True
    arch: resnet18
#    arch: resnet50
#    arch: resnet18
    var_type: fixedlarge
    ema_rate: 0.9999
    ema: True
    num_k: 6

diffusion:
    beta_schedule: linear  # cosine_anneal, cosine
    beta_start: 0.0001
    beta_end: 0.02
    timesteps: 1000
    test_timesteps: 100
    vis_step: 100
    num_figs: 10
    include_guidance: True  # concat y_t with aux pred as eps_theta input
    apply_aux_cls: True
    #trained_aux_cls_ckpt_path: "/data/classification/pretrained/cifar10_ckpt"
#    trained_aux_cls_ckpt_name: "snapshot_supervised.ckpt"
    #trained_aux_cls_ckpt_name: "snapshot_self_supervised.ckpt"
    aux_cls:
        arch: resnet18_ckpt  # lenet5
        pre_train: True
        joint_train: True
        n_pretrain_epochs: 10
        logging_interval: 1
    
training:
    batch_size: 64
    n_epochs: 2000
    warmup_epochs: 40
    add_t0_loss: False
    n_steps_req_grad: 100
    n_minibatches_add_ce: 20
    n_ce_epochs_warmup: 10
    n_ce_epochs_interval: 50
    n_sanity_check_epochs_freq: 500
    snapshot_freq: 1000000000
    logging_freq: 1200
    validation_freq: 10
    #lambda_ce: 0.01
    image_folder: 'training_image_samples'

sampling:
    batch_size: 64
    sampling_size: 1000
    last_only: True
    image_folder: 'sampling_image_samples'

testing:
    batch_size: 64
    sampling_size: 1000
    last_only: True
    plot_freq: 200
    image_folder: 'testing_image_samples'
    n_samples: 100
    n_bins: 10
    compute_metric_all_steps: False
    metrics_t: 0
    ttest_alpha: 0.05
    trimmed_mean_range: [0.0, 100.0]
    PICP_range: [2.5, 97.5]
    make_plot: True
    squared_plot: True
    plot_true: True
    plot_gen: True
    fig_size: [8, 5]

optim:
    weight_decay: 0.000
    optimizer: "Adam"
    lr: 0.001
    beta1: 0.9
    amsgrad: False
    eps: 0.00000001
    grad_clip: 1.0
    lr_schedule: True
    min_lr: 0.0

aux_optim:
    weight_decay: 0.000
    optimizer: "Adam"
    lr: 0.001
    beta1: 0.9
    amsgrad: True
    eps: 0.00000001
    grad_clip: 1.0



### Embedded runtime file: `option/diff_DDIM.yaml`

In [ ]:
%%writefile option/diff_DDIM.yaml
scheduler:
  type: DDIM
  num_train_timesteps: 1000
  num_test_timesteps: 10
  beta_start: 0.0001
  beta_end: 0.02
  beta_schedule: linear
  eta: 0

model:
  image_size: 224
  in_channels: 6
  out_channels: 3
  layers_per_block: 3
  block_out_channels:
    - 32
    - 64
    - 128
    - 256
  down_block_types:
    - DownBlock2D
    - DownBlock2D
    - DownBlock2D
    - DownBlock2D
  up_block_types:
    - UpBlock2D
    - UpBlock2D
    - UpBlock2D
    - UpBlock2D


### Embedded runtime file: `requirements-research.txt`

In [ ]:
%%writefile requirements-research.txt
# Add to the repository's documented PyTorch/CUDA environment. Select torch/torchvision
# builds appropriate for the target hardware before installation.
numpy>=1.23
Pillow>=9.3
scikit-learn>=1.3
scipy>=1.9
matplotlib>=3.7
joblib>=1.3
torch>=1.13
torchvision>=0.14
pytorch-lightning>=2.0
diffusers>=0.20
timm>=0.9
einops>=0.6
easydict>=1.10
PyYAML>=6.0


## 5. Runtime helpers and preflight checks

In [ ]:
def run_logged(command, log_path, allowed_returncodes=(0,)):
    command = [str(x) for x in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('\n$ ' + ' '.join(command))
    print('Log:', log_path)
    started = time.time()
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n\n===== START %s =====\n$ %s\n' % (time.ctime(), ' '.join(command)))
        process = subprocess.Popen(
            command,
            cwd=str(REPO_ROOT),
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        output_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            output_lines.append(line)
        returncode = process.wait()
        log.write('===== END returncode=%s duration=%.1fs =====\n' % (returncode, time.time() - started))
    if returncode not in allowed_returncodes:
        print(f'Command returned {returncode}. If this was a disconnect, rerun the same cell; otherwise inspect the log.')
    return returncode, ''.join(output_lines)


def mark_stage(name, status, details=None):
    payload = {'stage': name, 'status': status, 'time': time.strftime('%Y-%m-%dT%H:%M:%S%z'), 'details': details or {}}
    (STATUS_ROOT / f'{name}.json').write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(payload, indent=2))


def find_checkpoints():
    return sorted(
        [p for p in LOG_ROOT.rglob('*.ckpt') if p.is_file()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )


def newest_resume_checkpoint():
    checkpoints = [p for p in find_checkpoints() if p.name == 'last.ckpt']
    return checkpoints[0] if checkpoints else None


def best_or_latest_checkpoint():
    checkpoints = find_checkpoints()
    if not checkpoints:
        return None
    best = [p for p in checkpoints if p.name != 'last.ckpt']
    return (best or checkpoints)[0]


print('Python:', sys.version)
try:
    import torch
    import torchvision
    print('torch:', torch.__version__)
    print('torchvision:', torchvision.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch preflight failed:', repr(exc))

required_paths = {
    'repository': REPO_ROOT,
    'EfficientSAM': REPO_ROOT / 'EfficientSAM' / 'efficient_sam' / 'build_efficient_sam.py',
}
for label, path in required_paths.items():
    print(f'{label}: {path} -> {path.exists()}')

if APTOS_ROOT.exists():
    print('APTOS_ROOT exists; data validation can run.')
else:
    print('APTOS_ROOT does not exist yet. Edit the configuration cell before enabling data stages.')
if AUX_CHECKPOINT.exists():
    print('Auxiliary checkpoint exists; training can run.')
else:
    print('AUX_CHECKPOINT does not exist yet. Edit the configuration cell before enabling training.')


## 6. Create a Colab-specific YAML config

In [ ]:
import yaml

base_config_path = REPO_ROOT / 'configs' / 'aptos.yml'
colab_config_path = REPO_ROOT / 'configs' / 'aptos_colab.yml'
config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))
config['data']['dataroot'] = str(APTOS_ROOT)
config['data']['splits_manifest'] = str(OUTPUT_ROOT / 'aptos_splits.json')
config['data']['aux_checkpoint'] = str(AUX_CHECKPOINT)
config['data']['num_workers'] = int(NUM_WORKERS)
config['training']['n_epochs'] = int(EPOCHS)
config['training']['batch_size'] = int(BATCH_SIZE)
config['testing']['batch_size'] = int(BATCH_SIZE)
colab_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

os.environ['APTOS_ROOT'] = str(APTOS_ROOT)
os.environ['APTOS_SPLITS'] = str(OUTPUT_ROOT / 'aptos_splits.json')
os.environ['DIFFMIC_AUX_CHECKPOINT'] = str(AUX_CHECKPOINT)
print(colab_config_path.read_text(encoding='utf-8'))


## 7. Run lightweight tests and compile checks

In [ ]:
if RUN_UNIT_TESTS:
    mark_stage('unit_tests', 'RUNNING')
    rc_tests, _ = run_logged(
        [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
        OUTPUT_ROOT / 'unit_tests.log',
    )
    rc_compile, _ = run_logged(
        [sys.executable, '-m', 'compileall', '-q', 'aptos_research', 'dataloader', 'pretraining', 'tests', 'diffuser_trainer.py', 'model.py', 'pipeline.py', 'utils.py'],
        OUTPUT_ROOT / 'compileall.log',
    )
    mark_stage('unit_tests', 'COMPLETED' if rc_tests == 0 and rc_compile == 0 else 'FAILED', {'unittest_returncode': rc_tests, 'compileall_returncode': rc_compile})
else:
    print('Unit tests disabled.')


## 8. Validate APTOS once and preserve the split manifest

In [ ]:
split_manifest = OUTPUT_ROOT / 'aptos_splits.json'
if RUN_DATA_VALIDATION:
    if split_manifest.exists():
        print('Split manifest already exists; reusing it to preserve the experiment protocol:', split_manifest)
    else:
        if not APTOS_ROOT.is_dir():
            raise FileNotFoundError(f'APTOS_ROOT is not a directory: {APTOS_ROOT}')
        mark_stage('data_validation', 'RUNNING')
        rc, _ = run_logged(
            [sys.executable, '-m', 'aptos_research.cli', 'validate-data',
             '--root', str(APTOS_ROOT), '--output', str(split_manifest), '--seed', str(SEED)],
            OUTPUT_ROOT / 'data_validation.log',
        )
        if rc == 0:
            mark_stage('data_validation', 'COMPLETED', {'manifest': str(split_manifest)})
        else:
            mark_stage('data_validation', 'FAILED', {'returncode': rc})
            raise RuntimeError('APTOS validation failed; inspect outputs/data_validation.log')
else:
    print('Data validation disabled. Set RUN_DATA_VALIDATION=True after editing APTOS_ROOT.')


## 9. Train or resume DiffMIC-v2 with persistent checkpoints

In [ ]:
if RUN_TRAINING:
    if not AUX_CHECKPOINT.is_file():
        raise FileNotFoundError(f'Compatible DCG checkpoint not found: {AUX_CHECKPOINT}')
    if not split_manifest.is_file():
        raise FileNotFoundError(f'Split manifest not found: {split_manifest}. Run data validation first.')
    mark_stage('training', 'RUNNING')
    resume_checkpoint = newest_resume_checkpoint()
    command = [
        sys.executable, 'diffuser_trainer.py',
        '--config', 'configs/aptos_colab.yml',
        '--output-dir', str(LOG_ROOT),
        '--run-name', RUN_NAME,
    ]
    if resume_checkpoint is not None:
        command.extend(['--resume', str(resume_checkpoint)])
        print('Resuming from:', resume_checkpoint)
    else:
        print('Starting a new training run.')
    rc, _ = run_logged(command, OUTPUT_ROOT / 'training_console.log')
    mark_stage('training', 'COMPLETED' if rc == 0 else 'INCOMPLETE_OR_FAILED', {'returncode': rc, 'checkpoints': [str(p) for p in find_checkpoints()]})
else:
    print('Training disabled. Set RUN_TRAINING=True to start/resume; rerun this cell after a Colab reconnect.')


## 10. Select the trained checkpoint

In [ ]:
if EXPLICIT_DIFFMIC_CHECKPOINT:
    DIFFMIC_CHECKPOINT = Path(EXPLICIT_DIFFMIC_CHECKPOINT).expanduser()
else:
    DIFFMIC_CHECKPOINT = best_or_latest_checkpoint()

if DIFFMIC_CHECKPOINT is not None:
    DIFFMIC_CHECKPOINT = Path(DIFFMIC_CHECKPOINT)
    os.environ['DIFFMIC_CHECKPOINT'] = str(DIFFMIC_CHECKPOINT)
    print('Selected DiffMIC checkpoint:', DIFFMIC_CHECKPOINT)
    print('Exists:', DIFFMIC_CHECKPOINT.is_file())
else:
    print('No DiffMIC checkpoint discovered yet. Complete training or set EXPLICIT_DIFFMIC_CHECKPOINT.')


## 11. Smoke-test trajectory extraction

In [ ]:
smoke_cache = OUTPUT_ROOT / 'trajectories_smoke.sqlite'
if RUN_EXTRACTION_SMOKE:
    if DIFFMIC_CHECKPOINT is None or not Path(DIFFMIC_CHECKPOINT).is_file():
        raise FileNotFoundError('A trained DiffMIC checkpoint is required for extraction.')
    if not AUX_CHECKPOINT.is_file():
        raise FileNotFoundError(f'Compatible DCG checkpoint not found: {AUX_CHECKPOINT}')
    if not split_manifest.is_file():
        raise FileNotFoundError(f'Split manifest not found: {split_manifest}')
    mark_stage('extraction_smoke', 'RUNNING')
    rc, _ = run_logged(
        [sys.executable, '-m', 'aptos_research.extract',
         '--factory', 'aptos_research.diffmic_adapter:factory',
         '--checkpoint', str(DIFFMIC_CHECKPOINT), '--cache', str(smoke_cache),
         '--seed', str(SEED), '--inference-steps', '10', '--max-samples', '8', '--pooling', POOLING],
        OUTPUT_ROOT / 'extraction_smoke.log',
    )
    mark_stage('extraction_smoke', 'COMPLETED' if rc == 0 else 'FAILED', {'returncode': rc, 'cache': str(smoke_cache)})
else:
    print('Extraction smoke test disabled.')


## 12. Full trajectory extraction in resumable chunks

In [ ]:
trajectory_cache = OUTPUT_ROOT / 'trajectories.sqlite'
if RUN_FULL_EXTRACTION:
    if DIFFMIC_CHECKPOINT is None or not Path(DIFFMIC_CHECKPOINT).is_file():
        raise FileNotFoundError('A trained DiffMIC checkpoint is required for extraction.')
    if not split_manifest.is_file():
        raise FileNotFoundError(f'Split manifest not found: {split_manifest}')
    mark_stage('full_extraction', 'RUNNING')
    for chunk_index in range(int(EXTRACTION_CHUNKS_PER_CELL)):
        rc, text = run_logged(
            [sys.executable, '-m', 'aptos_research.extract',
             '--factory', 'aptos_research.diffmic_adapter:factory',
             '--checkpoint', str(DIFFMIC_CHECKPOINT), '--cache', str(trajectory_cache),
             '--seed', str(SEED), '--inference-steps', str(INFERENCE_STEPS),
             '--max-samples', str(EXTRACTION_CHUNK), '--pooling', POOLING],
            OUTPUT_ROOT / 'trajectory_extraction.log',
        )
        if rc != 0:
            mark_stage('full_extraction', 'INCOMPLETE_OR_FAILED', {'chunk': chunk_index + 1, 'returncode': rc})
            break
        if '"new_samples": 0' in text or "'new_samples': 0" in text:
            print('Extractor reported no new samples; the resumable cache is complete or no eligible samples remain.')
            break
    inspect_rc, inspect_text = run_logged(
        [sys.executable, '-m', 'aptos_research.cli', 'inspect-cache', str(trajectory_cache)],
        OUTPUT_ROOT / 'trajectory_cache_inspection.log',
    )
    mark_stage('full_extraction', 'COMPLETED' if inspect_rc == 0 else 'INCOMPLETE_OR_FAILED', {'cache': str(trajectory_cache), 'inspect_returncode': inspect_rc})
else:
    print('Full extraction disabled. Rerun this cell repeatedly after a reconnect; the SQLite cache skips completed samples.')


## 13. Run classical trajectory experiments

In [ ]:
classical_results = OUTPUT_ROOT / 'experiments' / 'classical.json'
if RUN_CLASSICAL_EXPERIMENTS:
    if not trajectory_cache.is_file():
        raise FileNotFoundError(f'Trajectory cache not found: {trajectory_cache}')
    mark_stage('classical_experiments', 'RUNNING')
    rc, _ = run_logged(
        [sys.executable, '-m', 'aptos_research.experiments',
         '--cache', str(trajectory_cache), '--output', str(classical_results), '--seed', str(SEED)],
        OUTPUT_ROOT / 'classical_experiments.log',
    )
    mark_stage('classical_experiments', 'COMPLETED' if rc == 0 else 'FAILED', {'returncode': rc, 'results': str(classical_results)})
else:
    print('Classical experiments disabled.')


## 14. Run the validation-only evidence gate and conditional QRC

In [ ]:
gate_result = OUTPUT_ROOT / 'experiments' / 'gate.json'
if RUN_GATE:
    if not classical_results.is_file():
        raise FileNotFoundError(f'Classical result file not found: {classical_results}')
    mark_stage('gate', 'RUNNING')
    rc, _ = run_logged(
        [sys.executable, '-m', 'aptos_research.cli', 'gate',
         '--results', str(classical_results), '--output', str(gate_result),
         '--minimum-delta', '0.01', '--alpha', '0.05', '--bootstrap-samples', '5000', '--seed', str(SEED)],
        OUTPUT_ROOT / 'gate.log',
        allowed_returncodes=(0, 2, 3),  # FAIL/INCONCLUSIVE are recorded decisions, not shell failures.
    )
    mark_stage('gate', 'COMPLETED' if gate_result.is_file() else 'FAILED', {'returncode': rc, 'gate': str(gate_result)})
else:
    print('Evidence gate disabled.')

qrc_result = OUTPUT_ROOT / 'experiments' / 'qrc.json'
if RUN_QRC:
    if not gate_result.is_file():
        raise FileNotFoundError(f'Gate artifact not found: {gate_result}')
    rc, _ = run_logged(
        [sys.executable, '-m', 'aptos_research.run_qrc',
         '--gate', str(gate_result), '--cache', str(trajectory_cache), '--output', str(qrc_result),
         '--qubits', '4', '--shots', '0', '--seed', str(SEED)],
        OUTPUT_ROOT / 'qrc.log',
        allowed_returncodes=(0, 2),  # 2 is the intentional SKIPPED_GATE_NOT_PASS result.
    )
    mark_stage('qrc', 'COMPLETED_OR_SKIPPED', {'returncode': rc, 'result': str(qrc_result)})
else:
    print('QRC disabled.')


## 15. Trust evaluation and one XAI trajectory plot

In [ ]:
if RUN_TRUST:
    trust_result = OUTPUT_ROOT / 'experiments' / 'trust.json'
    rc, _ = run_logged(
        [sys.executable, '-m', 'aptos_research.trust_eval',
         '--cache', str(trajectory_cache), '--results', str(classical_results),
         '--model', 'ordered_esn_k16', '--output', str(trust_result)],
        OUTPUT_ROOT / 'trust.log',
    )
    mark_stage('trust', 'COMPLETED' if rc == 0 else 'FAILED', {'returncode': rc, 'result': str(trust_result)})
else:
    print('Trust evaluation disabled.')

if RUN_XAI:
    import sqlite3
    xai_split = 'test'
    with sqlite3.connect(str(trajectory_cache)) as connection:
        row = connection.execute('SELECT sample_id FROM states WHERE split=? ORDER BY sample_id LIMIT 1', (xai_split,)).fetchone()
    if row is None:
        raise RuntimeError('No test sample is present in the trajectory cache for XAI.')
    sample_id = row[0]
    rc, _ = run_logged(
        [sys.executable, '-m', 'aptos_research.xai_eval',
         '--cache', str(trajectory_cache), '--sample-id', sample_id,
         '--split', xai_split, '--output', str(OUTPUT_ROOT / 'experiments' / 'xai')],
        OUTPUT_ROOT / 'xai.log',
    )
    mark_stage('xai', 'COMPLETED' if rc == 0 else 'FAILED', {'returncode': rc, 'sample_id': sample_id})
else:
    print('XAI disabled.')


## 16. Inspect persisted artifacts

In [ ]:
print('Persistent workspace:', REPO_ROOT)
print('Logs and outputs:', OUTPUT_ROOT)
print('Checkpoint candidates:')
for checkpoint in find_checkpoints():
    print('  ', checkpoint)
print('Stage status files:')
for status_file in sorted(STATUS_ROOT.glob('*.json')):
    print('  ', status_file.name, '->', json.loads(status_file.read_text(encoding='utf-8')).get('status'))
